# 27. Integrated holdout companion lineage

This offline notebook creates only source-lineage companions and one excluded demo claim. It does not read holdout query or label content, perform retrieval, or create embeddings.


In [1]:
from __future__ import annotations

import bisect
import hashlib
import json
import os
import re
import shutil
import tempfile
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert ROOT.name == "PickCardU"
OUT = ROOT / "notebooks/data/27_integrated_holdout_dataset"
RAW_ROOT = ROOT / "data/ocr_benchmark/gold/raw"
STRUCT_ROOT = ROOT / "data/ocr_benchmark/gold/structured"
OLD_PATH = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/chunks.jsonl"
OLD_INPUT = ROOT / "notebooks/data/13_hierarchical_chunking_retrieval/input_manifest.json"
STRUCT_PATH = ROOT / "notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl"
STRUCT_HIERARCHY = ROOT / "notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl"
SOURCE_NOTEBOOKS = [ROOT / "notebooks/13_hierarchical_chunking_retrieval.ipynb", ROOT / "notebooks/22_structural_heading_chunking_ablation.ipynb"]

assert os.environ.get("RUN_APPROVED_27_NETWORK", "0") == "0"
assert os.environ.get("RUN_APPROVED_27_GPU", "0") == "0"
assert not (OUT / "holdout_queries.json").exists()

PAGE_RE = re.compile(r"^\[page\s*(\d+)\]\s*$", re.IGNORECASE)
PAGE_RE_13 = re.compile(r"(?im)^\[page\s*(\d+)\]\s*$")
HEADING_RE = re.compile(r"^\s{0,3}(#{1,6})\s+(.+?)\s*$")
HEADING_RE_13 = re.compile(r"(?m)^\s{0,3}#{1,6}\s+(.+?)\s*$")
PARSER_VERSION = "raw_txt_utf8_lf_page_marker_v1"
OLD_MAX = {"card": 2000, "page": 6000, "section": 4000, "benefit": 3000}
STRUCT_MAX = 4000

def sha_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()

def sha_file(path: Path) -> str:
    return sha_bytes(path.read_bytes())

def canonical(value: object) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def normalized(value: str) -> str:
    return " ".join(unicodedata.normalize("NFKC", str(value)).lower().split())

def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8", newline="\n")

def write_jsonl(path: Path, rows: list[dict]) -> None:
    with path.open("w", encoding="utf-8", newline="\n") as handle:
        for row in rows:
            handle.write(canonical(row) + "\n")

def scalar_metadata(metadata: dict) -> dict:
    result = {}
    for key, value in metadata.items():
        if value is not None:
            result[key] = canonical(value) if isinstance(value, (dict, list, tuple)) else value
    return result

def old_id(*parts: object) -> str:
    return sha_bytes("|".join(map(str, parts)).encode())[:32]

def raw_model(path: Path, issuer: str, card_name: str) -> dict:
    raw_bytes = path.read_bytes()
    assert b"\r" not in raw_bytes, f"non-LF source: {path}"
    raw = raw_bytes.decode("utf-8")
    lines = []
    cp = byte = 0
    page = 1
    for number, physical in enumerate(raw.splitlines(keepends=True), 1):
        text = physical.rstrip("\n")
        kind = "page_marker" if PAGE_RE.fullmatch(text) else "content"
        if kind == "page_marker":
            page = int(PAGE_RE.fullmatch(text).group(1))
        cp_end = cp + len(text)
        byte_end = byte + len(text.encode("utf-8"))
        lines.append({"line_number": number, "page_num": page, "kind": kind, "text": text,
                      "cp_start": cp, "cp_end": cp_end, "byte_start": byte, "byte_end": byte_end})
        cp += len(physical)
        byte += len(physical.encode("utf-8"))
    assert cp == len(raw) and byte == len(raw_bytes)
    markers = [line for line in lines if line["kind"] == "page_marker"]
    assert markers
    pages = []
    for index, marker in enumerate(markers):
        next_marker = markers[index + 1] if index + 1 < len(markers) else None
        end_cp = next_marker["cp_start"] if next_marker else len(raw)
        end_byte = next_marker["byte_start"] if next_marker else len(raw_bytes)
        end_line = (next_marker["line_number"] - 1) if next_marker else len(lines)
        pages.append({"page_num": marker["page_num"], "line_start": marker["line_number"], "line_end": end_line,
                      "cp_start": marker["cp_start"], "cp_end": end_cp, "byte_start": marker["byte_start"], "byte_end": end_byte,
                      "content_cp_start": marker["cp_end"] + (1 if raw[marker["cp_end"]:marker["cp_end"] + 1] == "\n" else 0),
                      "content_cp_end": end_cp})
    assert sum(item["cp_end"] - item["cp_start"] for item in pages) == len(raw)
    return {"path": path, "source_path": path.relative_to(ROOT).as_posix(), "issuer": issuer, "card_name": card_name,
            "card_key": issuer + "/" + card_name, "raw": raw, "raw_bytes": raw_bytes, "sha256": sha_bytes(raw_bytes),
            "lines": lines, "pages": pages, "line_starts": [line["cp_start"] for line in lines]}

def interval(model: dict, cp_start: int, cp_end: int) -> dict:
    assert 0 <= cp_start <= cp_end <= len(model["raw"])
    if cp_start == cp_end:
        return {"line_start": None, "line_end": None, "cp_start": cp_start, "cp_end": cp_end,
                "byte_start": len(model["raw"][:cp_start].encode("utf-8")), "byte_end": len(model["raw"][:cp_end].encode("utf-8"))}
    starts = model["line_starts"]
    first = bisect.bisect_right(starts, cp_start) - 1
    last = bisect.bisect_right(starts, max(cp_start, cp_end - 1)) - 1
    return {"line_start": model["lines"][first]["line_number"], "line_end": model["lines"][last]["line_number"],
            "cp_start": cp_start, "cp_end": cp_end,
            "byte_start": len(model["raw"][:cp_start].encode("utf-8")), "byte_end": len(model["raw"][:cp_end].encode("utf-8"))}

def trim_span(text: str, start: int, end: int) -> tuple[int, int]:
    while start < end and text[start].isspace(): start += 1
    while end > start and text[end - 1].isspace(): end -= 1
    return start, end

def paragraph_components(model: dict, start: int, end: int) -> list[dict]:
    value = model["raw"][start:end]
    cursor = 0; result = []
    for match in list(re.finditer(r"\n\s*\n", value)) + [None]:
        stop = match.start() if match else len(value)
        left, right = trim_span(value, cursor, stop)
        if left < right:
            cp_start, cp_end = start + left, start + right
            result.append({"text": model["raw"][cp_start:cp_end], **interval(model, cp_start, cp_end)})
        if match is None: break
        cursor = match.end()
    return result

def component_slice(component: dict, start: int, end: int, model: dict) -> dict:
    assert 0 <= start <= end <= len(component["text"])
    cp_start = component["cp_start"] + start; cp_end = component["cp_start"] + end
    return {"text": component["text"][start:end], **interval(model, cp_start, cp_end)}

def component_pages(model: dict, component: dict) -> list[int]:
    if component["line_start"] is None:
        return []
    return sorted({model["lines"][number - 1]["page_num"] for number in range(component["line_start"], component["line_end"] + 1)})

def join_components(components: list[dict], separator: str = "\n\n") -> str:
    return separator.join(component["text"] for component in components)

def bounded_components(components: list[dict], limit: int, model: dict, separator: str = "\n\n") -> list[list[dict]]:
    parts: list[list[dict]] = []; current: list[dict] = []
    for component in components:
        if len(component["text"]) > limit:
            if current: parts.append(current); current = []
            for start in range(0, len(component["text"]), limit):
                parts.append([component_slice(component, start, min(len(component["text"]), start + limit), model)])
        elif current and len(join_components(current, separator)) + len(separator) + len(component["text"]) > limit:
            parts.append(current); current = [component]
        else:
            current.append(component)
    if current: parts.append(current)
    assert all(join_components(part, separator).strip() and len(join_components(part, separator)) <= limit for part in parts)
    return parts

def truncate_components(components: list[dict], limit: int, model: dict, separator: str = "\n\n") -> list[dict]:
    result: list[dict] = []; used = 0
    for component in components:
        gap = len(separator) if result else 0
        remaining = limit - used - gap
        if remaining <= 0: break
        take = min(len(component["text"]), remaining)
        result.append(component_slice(component, 0, take, model))
        used += gap + take
        if take < len(component["text"]): break
    return result

def old_pages(model: dict) -> dict[int, tuple[int, int]]:
    raw = model["raw"]; matches = list(PAGE_RE_13.finditer(raw)); result = {}
    for index, match in enumerate(matches):
        start = match.end(); end = matches[index + 1].start() if index + 1 < len(matches) else len(raw)
        start, end = trim_span(raw, start, end); result[int(match.group(1))] = (start, end)
    return result

def old_sections(model: dict, start: int, end: int) -> list[tuple[str, list[dict]]]:
    value = model["raw"][start:end]; matches = list(HEADING_RE_13.finditer(value)); result = []
    if not matches: return [("page_body", paragraph_components(model, start, end))]
    if value[:matches[0].start()].strip():
        a, b = trim_span(value, 0, matches[0].start()); result.append(("page_intro", paragraph_components(model, start+a, start+b)))
    for index, match in enumerate(matches):
        stop = matches[index + 1].start() if index + 1 < len(matches) else len(value)
        a, b = trim_span(value, match.start(), stop)
        result.append((match.group(1).strip(), paragraph_components(model, start+a, start+b)))
    return result

def label_needles(label: dict) -> list[str]:
    values = [label.get("surface_text"), *(label.get("context_terms") or []), *(label.get("headers") or [])]
    return [normalized(value) for value in values if value]

def old_benefit(components: list[dict], label: dict, model: dict) -> tuple[list[dict], list[dict]]:
    needles = label_needles(label)
    scores = [sum(needle in normalized(component["text"]) for needle in needles) for component in components]
    best = max(range(len(components)), key=lambda index: scores[index]) if components else 0
    selected = components[max(0, best - 1):best + 2] if components else components
    return truncate_components(selected, OLD_MAX["benefit"], model), truncate_components([components[best]], OLD_MAX["benefit"], model)

def build_old(models: dict[str, dict], inputs: list[dict], fingerprint: str) -> list[dict]:
    rows = []
    for item in inputs:
        model = models[item["issuer"] + "/" + item["card_name"]]; raw = model["raw"]
        key = (item["issuer"], item["card_name"]); card_key = "/".join(key)
        structured = json.loads((ROOT / item["structured_path"]).read_text(encoding="utf-8"))
        base = {"issuer":key[0],"card_name":key[1],"card_key":card_key,"source_path":item["raw_path"],"structured_path":item["structured_path"],"coverage_status":item["coverage_status"],"input_fingerprint":fingerprint}
        line_components = []
        for line in model["lines"]:
            if line["kind"] != "page_marker" and line["text"].strip():
                left, right = trim_span(raw, line["cp_start"], line["cp_end"])
                line_components.append({"text":raw[left:right], **interval(model,left,right)})
        headings = []
        for match in HEADING_RE_13.finditer(raw):
            left, right = trim_span(raw, match.start(1), match.end(1)); headings.append({"text":raw[left:right], **interval(model,left,right)})
        chosen = []
        seen = set()
        for component in line_components[:12] + headings:
            if component["text"] not in seen: chosen.append(component); seen.add(component["text"])
        card_full = join_components(chosen, "\n")
        card_doc = card_full[:OLD_MAX["card"]]
        if len(card_doc) < len(card_full):
            chosen = truncate_components(chosen, OLD_MAX["card"], model, "\n")
        card_id = old_id(fingerprint,*key,"card")
        rows.append({"id":card_id,"document":card_doc,"components":chosen,"metadata":scalar_metadata({**base,"level":"card","parent_id":"","page_num":0,"section":"card_overview"})})
        labels_by_page = defaultdict(list)
        for kind in ("field_labels","numeric_labels","table_labels"):
            for label in structured.get(kind,[]): labels_by_page[label.get("page_num",1)].append((kind,label))
        for page_num,(start,end) in old_pages(model).items():
            page_components = paragraph_components(model,start,end)
            page_ids = [label["id"] for _,label in labels_by_page[page_num]]
            for part_num, components in enumerate(bounded_components(page_components,OLD_MAX["page"],model),1):
                doc=join_components(components); rows.append({"id":old_id(fingerprint,*key,"page",page_num,part_num),"document":doc,"components":components,"metadata":scalar_metadata({**base,"level":"page","parent_id":card_id,"page_num":page_num,"part_num":part_num,"section":"page","label_ids":page_ids})})
            for section_num,(section,components) in enumerate(old_sections(model,start,end),1):
                for part_num, part in enumerate(bounded_components(components,OLD_MAX["section"],model),1):
                    doc=join_components(part)
                    if not HEADING_RE_13.sub("",doc).strip(): continue
                    rows.append({"id":old_id(fingerprint,*key,"section",page_num,section_num,part_num),"document":doc,"components":part,"metadata":scalar_metadata({**base,"level":"section","parent_id":card_id,"page_num":page_num,"part_num":part_num,"section":section})})
            groups={}
            for kind,label in labels_by_page[page_num]:
                document,core=old_benefit(page_components,label,model)
                if not join_components(document).strip(): continue
                group_key=sha_bytes(normalized(join_components(document)).encode())
                groups.setdefault(group_key,{"components":document,"core":core,"labels":[]})["labels"].append({"kind":kind,**label})
            for group_key,group in groups.items():
                ids=[label["id"] for label in group["labels"]]; doc=join_components(group["components"])
                rows.append({"id":old_id(fingerprint,*key,"benefit",page_num,group_key,*ids),"document":doc,"components":group["components"],"metadata":scalar_metadata({**base,"level":"benefit","parent_id":card_id,"page_num":page_num,"section":"benefit_group:"+ids[0],"label_ids":ids,"label_kinds":sorted({label["kind"] for label in group["labels"]}),"structured_metadata":group["labels"]})})
    return rows

def build_struct(model: dict) -> tuple[list[dict], list[dict]]:
    raw=model["raw"]; key=model["card_key"]; issuer=model["issuer"]; card_name=model["card_name"]
    nodes=[{"node_id":key+"::node0000","parent_id":None,"heading_text":None,"heading_level":0,"heading_path":[],"heading_line_number":None,"heading_page":None,"records":[]}]
    stack=[]; current=nodes[0]
    for line in model["lines"]:
        if line["kind"]=="page_marker": continue
        heading=HEADING_RE.fullmatch(line["text"])
        if heading:
            level=len(heading.group(1)); text=heading.group(2).strip()
            while stack and stack[-1]["heading_level"]>=level: stack.pop()
            parent=stack[-1] if stack else nodes[0]
            current={"node_id":key+"::node"+str(len(nodes)).zfill(4),"parent_id":parent["node_id"],"heading_text":text,"heading_level":level,"heading_path":parent["heading_path"]+[text],"heading_line_number":line["line_number"],"heading_page":line["page_num"],"records":[]}
            nodes.append(current); stack.append(current); continue
        current["records"].append({"text":line["text"],**interval(model,line["cp_start"],line["cp_end"]),"page":line["page_num"]})
    chunks=[]; hierarchy=[]
    for node in nodes:
        records=node["records"]; substantive=bool(records and any(r["text"].strip() for r in records)); parts=[]; method="not_applicable"
        if substantive:
            parts=bounded_components(records,STRUCT_MAX,model,"\n")
            method="line_boundary_fallback" if any(len(record["text"]) > STRUCT_MAX for record in records) else "paragraph_boundary"
        ids=[]
        for part_index,part in enumerate(parts,1):
            body=join_components(part,"\n"); pages=sorted({r["page"] for r in part}); body_sha=sha_bytes(body.encode())
            seed=canonical({"experiment":"구조 기반 청킹 + 제목 경로 검색문","source_hash":model["sha256"],"node_id":node["node_id"],"part_index":part_index,"body_sha256":body_sha})
            chunk_id="shc_"+sha_bytes(seed.encode())[:24]; ids.append(chunk_id); path_text=" > ".join(node["heading_path"])
            chunks.append({"chunk_id":chunk_id,"body":body,"heading_path":node["heading_path"],"components":part,"metadata":{"issuer":issuer,"card_name":card_name,"card_key":key,"page_numbers":pages,"page_start":min(pages),"page_end":max(pages),"heading_level":node["heading_level"],"node_id":node["node_id"],"parent_id":node["parent_id"],"part_index":part_index,"part_count":len(parts),"source_path":model["source_path"],"source_sha256":model["sha256"],"body_sha256":body_sha,"split_method":method}})
        pages=sorted(({node["heading_page"]} if node["heading_page"] else set())|{r["page"] for r in records})
        hierarchy.append({"node_id":node["node_id"],"heading_line_number":node["heading_line_number"],"heading_page":node["heading_page"],"search_chunk_ids":ids,"page_numbers":pages,"source_sha256":model["sha256"]})
    return chunks,hierarchy

input_manifest=json.loads(OLD_INPUT.read_text(encoding="utf-8")); inputs=input_manifest["files"]
models={}
for item in inputs:
    path=ROOT/item["raw_path"]; model=raw_model(path,item["issuer"],item["card_name"]); assert model["sha256"]==item["raw_sha256"]; models[model["card_key"]]=model
assert len(models)==10

OUT.mkdir(parents=True,exist_ok=True)
registry=[]; line_rows=[]
for key,model in sorted(models.items()):
    registry.append({"card_key":key,"issuer":model["issuer"],"card_name":model["card_name"],"path":model["source_path"],"raw_sha256":model["sha256"],"bytes":len(model["raw_bytes"]),"encoding":"utf-8","newline":"LF","page_marker_regex":PAGE_RE.pattern,"parser_version":PARSER_VERSION,"pages":model["pages"]})
    for line in model["lines"]: line_rows.append({"card_key":key,"source_sha256":model["sha256"],**{k:line[k] for k in ["line_number","page_num","kind","cp_start","cp_end","byte_start","byte_end"]}})
write_json(OUT/"source_registry.json",registry); write_jsonl(OUT/"raw_line_index.jsonl",line_rows)

old_existing=[json.loads(line) for line in OLD_PATH.read_text(encoding="utf-8").splitlines() if line]
old_rebuilt=build_old(models,inputs,input_manifest["input_fingerprint"])
assert len(old_existing)==len(old_rebuilt)==327
old_existing_by_id={row["id"]:row for row in old_existing}; old_rows=[]; old_status=Counter()
for rebuilt in old_rebuilt:
    existing=old_existing_by_id.get(rebuilt["id"])
    status="exact" if existing and existing["document"]==rebuilt["document"] and existing["metadata"]==rebuilt["metadata"] else "unmapped"
    reason=None if status=="exact" else "frozen notebook13 reconstruction differs from persisted chunk"
    old_status[status]+=1
    model=models[rebuilt["metadata"]["card_key"]]
    old_rows.append({"chunk_id":rebuilt["id"],"status":status,"reason":reason,"source_path":rebuilt["metadata"]["source_path"],"source_sha256":model["sha256"],"card_key":rebuilt["metadata"]["card_key"],"level":rebuilt["metadata"]["level"],"page_numbers":sorted({page for component in rebuilt["components"] for page in component_pages(model, component)}),"component_intervals":[{**{k:c[k] for k in ["line_start","line_end","cp_start","cp_end","byte_start","byte_end"]},"page_numbers":component_pages(model,c)} for c in rebuilt["components"]]})
assert old_status==Counter({"exact":327})
write_jsonl(OUT/"old_chunk_lineage.jsonl",old_rows)

struct_existing=[json.loads(line) for line in STRUCT_PATH.read_text(encoding="utf-8").splitlines() if line]
struct_rebuilt=[]; struct_hierarchy=[]
for model in models.values():
    chunks,hierarchy=build_struct(model); struct_rebuilt.extend(chunks); struct_hierarchy.extend(hierarchy)
assert len(struct_existing)==len(struct_rebuilt)==147
struct_existing_by_id={row["chunk_id"]:row for row in struct_existing}; struct_rows=[]; struct_status=Counter()
for rebuilt in struct_rebuilt:
    existing=struct_existing_by_id.get(rebuilt["chunk_id"])
    status="exact" if existing and existing["body"]==rebuilt["body"] and existing["heading_path"]==rebuilt["heading_path"] and existing["metadata"]==rebuilt["metadata"] else "unmapped"
    struct_status[status]+=1
    direct_components=[c for c in rebuilt["components"] if c["line_start"] is not None]
    struct_rows.append({"chunk_id":rebuilt["chunk_id"],"status":status,"reason":None if status=="exact" else "frozen notebook22 reconstruction differs from persisted chunk","source_path":rebuilt["metadata"]["source_path"],"source_sha256":rebuilt["metadata"]["source_sha256"],"card_key":rebuilt["metadata"]["card_key"],"node_id":rebuilt["metadata"]["node_id"],"body_sha256":rebuilt["metadata"]["body_sha256"],"page_numbers":rebuilt["metadata"]["page_numbers"],"direct_body_start_line":min(c["line_start"] for c in direct_components),"direct_body_end_line":max(c["line_end"] for c in direct_components),"component_intervals":[{**{k:c[k] for k in ["line_start","line_end","cp_start","cp_end","byte_start","byte_end"]},"page_numbers":component_pages(models[rebuilt["metadata"]["card_key"]],c)} for c in rebuilt["components"]]})
assert struct_status==Counter({"exact":147}) and all(row["source_sha256"] for row in struct_rows)
write_jsonl(OUT/"struct_chunk_lineage.jsonl",struct_rows)

demo_model=models["BC/BC_Biz_AirMoney"]; demo_claim={"split":"demo","evaluation_excluded":True,"atomic_claim_id":"ac_v1_"+sha_bytes((demo_model["sha256"]+"|2|47|54|air_money_earn_rate").encode())[:24],"claim_type":"rate","canonical_value":0.002,"unit":"fraction","condition":{"previous_month_spend":"no restriction"},"source_refs":[{"source_path":demo_model["source_path"],"source_sha256":demo_model["sha256"],"page_num":2,**interval(demo_model,demo_model["lines"][46]["cp_start"],demo_model["lines"][53]["cp_end"]),"excerpt_sha256":sha_bytes(demo_model["raw"][demo_model["lines"][46]["cp_start"]:demo_model["lines"][53]["cp_end"]].encode())}],"boolean_qualification":{"source_hash_verified":True,"atomic_locator_verified":True,"eligible_for_evaluation":False},"answer_policy":"Return canonical rate as percent only when a future evaluation contract explicitly requests rate rendering.","provenance":{"source":"gold raw TXT + structured numeric label ID air_money_earn_rate","structured_path":"data/ocr_benchmark/gold/structured/BC/BC_Biz_AirMoney.json","builder":"notebook27_offline"}}
claim_interval=demo_claim["source_refs"][0]
def overlaps(row):
    return row["source_sha256"]==claim_interval["source_sha256"] and any(c["cp_start"]<claim_interval["cp_end"] and claim_interval["cp_start"]<c["cp_end"] for c in row["component_intervals"])
old_demo=[row["chunk_id"] for row in old_rows if row["status"]=="exact" and overlaps(row)]
struct_demo=[row["chunk_id"] for row in struct_rows if row["status"]=="exact" and overlaps(row)]
projection={"split":"demo","evaluation_excluded":True,"atomic_claim_id":demo_claim["atomic_claim_id"],"method":"source_sha256_and_codepoint_interval_overlap_only","string_matching_used":False,"fuzzy_matching_used":False,"old_chunk_ids":old_demo,"struct_chunk_ids":struct_demo,"deterministic":True}
assert old_demo and struct_demo
write_json(OUT/"demo_master_claim.json",demo_claim); write_json(OUT/"demo_projection.json",projection)
write_json(OUT/"demo_public_view.json",{"split":"demo","evaluation_excluded":True,"question":"BC biz Air Money 카드의 적립률은 얼마인가?","card_key":"BC/BC_Biz_AirMoney","atomic_claim_id":demo_claim["atomic_claim_id"],"claim_value":"0.2%","source_locator":{"page_num":2,"line_start":47,"line_end":54,"source_sha256":demo_model["sha256"]},"projection_counts":{"old":len(old_demo),"struct":len(struct_demo)}})

contract={"schema_version":"1.0.0","scope":"companion lineage and one demo only; no holdout query generation or evaluation","projection_rule":"same source SHA-256 plus codepoint interval overlap; no exact-string or fuzzy matching","sources":"10 immutable gold raw TXT files","demo":{"split":"demo","excluded_from_all_evaluation_denominators":True},"execution":{"environment":"skn25","network":0,"api_requests":0,"gpu":0,"embeddings":0,"retrieval":0},"forbidden_reads":["holdout14 query/label content","holdout26 query/label content"]}
write_json(OUT/"lineage_contract.json",contract)
audit={"status":"PASS","source_count":len(registry),"source_hash_exact":True,"page_interval_nonoverlap_full_coverage":True,"old":{"expected":327,"reconstructed":len(old_rows),"coverage":dict(old_status)},"struct":{"expected":147,"reconstructed":len(struct_rows),"coverage":dict(struct_status),"source_sha_exact":147},"projection":{"deterministic":True,"source_overlap_only":True,"old_demo_count":len(old_demo),"struct_demo_count":len(struct_demo)},"demo_excluded_from_evaluation":True,"forbidden_holdout_content_read":False,"api_network_gpu":{"api":0,"network":0,"gpu":0}}
write_json(OUT/"lineage_quality_audit.json",audit)

readme="# Integrated holdout companion lineage\n\nThis folder contains offline lineage companions for frozen OLD (327) and STRUCT (147) chunks plus one `split=demo` claim. The demo is excluded from every evaluation denominator. Projection uses only immutable raw-source SHA-256 and codepoint interval overlap; it never uses candidate-document exact or fuzzy matching. No holdout query or label content is stored or read here.\n"
(OUT/"README.md").write_text(readme,encoding="utf-8",newline="\n")
inputs_hashes={p.relative_to(ROOT).as_posix():sha_file(p) for p in [OLD_PATH,OLD_INPUT,STRUCT_PATH,STRUCT_HIERARCHY,*SOURCE_NOTEBOOKS]}
outputs=[p for p in OUT.iterdir() if p.is_file() and p.name not in {"run_manifest.json","integrity.json"}]
run={"schema_version":"1.0.0","execution_mode":"fresh_kernel_cpu_offline","inputs":inputs_hashes,"outputs":{p.name:sha_file(p) for p in sorted(outputs)},"self_hash_policy":"run_manifest.json and integrity.json excluded","execution":{"network":0,"api_requests":0,"gpu":0,"embeddings":0,"retrieval":0}}
write_json(OUT/"run_manifest.json",run)
integrity={"status":"PASS","run_manifest_sha256":sha_file(OUT/"run_manifest.json"),"assertions":{"source_hash_exact":True,"page_coverage":True,"old_327_exact":True,"struct_147_exact":True,"projection_deterministic":True,"demo_evaluation_excluded":True,"holdout14_26_content_not_read":True,"network_api_gpu_zero":True}}
write_json(OUT/"integrity.json",integrity)
print(json.dumps({"status":"PASS","old":len(old_rows),"struct":len(struct_rows),"demo_old_projection":len(old_demo),"demo_struct_projection":len(struct_demo),"out":str(OUT)},ensure_ascii=False))


{"status": "PASS", "old": 327, "struct": 147, "demo_old_projection": 4, "demo_struct_projection": 1, "out": "/home/sms/openclaw_file/PickCardU/notebooks/data/27_integrated_holdout_dataset"}


## Draft integrated master dataset\n\nThis append-only cell materializes a pending independent-review dataset. It performs offline source verification and source-overlap projection only; it does not run retrieval, embeddings, ranking, or models.\n

In [2]:
# Draft integrated master dataset: CPU/offline only; no retrieval or model calls.
import bisect
import hashlib
import json
import re
import unicodedata
from collections import Counter

DRAFT_STATUS = "pending_independent_review"
CORPUS_CARD_IDS = ["BC", "NH", "HANA", "HYUNDAI", "IBK", "KB", "LOTTE", "SAMSUNG", "SHINHAN", "WOORI"]
CARD_KEYS = {
    "BC":"BC/BC_Biz_AirMoney", "NH":"NH/NH_Namu_NH", "HANA":"hana/Hana_One_More_SOHO",
    "HYUNDAI":"hyundai/Hyundai_The_Orange_20260330", "IBK":"ibk/IBK_Point3.8(Credit)",
    "KB":"kookmin/Kookmin_Friend_20210917", "LOTTE":"lotte/Lotte_LOCA_LIKIT_Eat",
    "SAMSUNG":"samsung/Samsung_iD_ALL", "SHINHAN":"shinhan/Shinhan_Toss_Mr.Life_20251231",
    "WOORI":"woori/Woori_Classic_EVERY_MILE_SKYPASS",
}
NEW_FILES = {
    "evaluation_queries.json", "master_gold.jsonl", "source_refs.json", "atomic_claims.json", "answer_policies.json",
    "qualification_by_card.jsonl", "annotation_audit.jsonl", "candidate_projection.jsonl", "query_duplicate_audit.jsonl",
    "draft_dataset_manifest.json", "draft_integrity.json", "draft_review_packet.jsonl",
}
prior_run = json.loads((OUT / "run_manifest.json").read_text(encoding="utf-8"))
lineage_hashes = {name: digest for name, digest in prior_run["outputs"].items()}
assert all(sha_file(OUT / name) == digest for name, digest in lineage_hashes.items())
source_inputs_before = {path.relative_to(ROOT).as_posix(): sha_file(path) for path in [OLD_PATH, OLD_INPUT, STRUCT_PATH, STRUCT_HIERARCHY]}

def norm_query(value):
    return re.sub(r"[^\\w가-힣]", "", " ".join(unicodedata.normalize("NFKC", value).casefold().split()))

def compact_hash(value):
    return sha_bytes(value.encode("utf-8"))

ref_lines = """R01|BC|2|70|70|1971|2149|4339|4723|83425c91e5c9ede0193749d64828a38becc4949f878f74f5c37fe969eddcfbf9
R02|NH|2|73|78|3305|3393|7681|7841|0f4521e15d6b6ce89f403f4569c3818a57c6be72529eb29459406f99c66e53d4
R03|HANA|2|113|118|2971|3231|6105|6723|f8881c5b460a89688c5edf45e9b2fc9525f30ae428a45560800df1f99095c0a0
R04|HYUNDAI|7|127|140|4110|4577|8801|9771|1d9549400c32da9666261be90207a77fb0237086f40be12ed4a628bc9c66a701
R05|IBK|3|61|65|2169|2424|4794|5379|cca661ee8bd6637491968f9592748c05a4218bcd17526df17ea70cee46547ca0
R06|KB|1|15|16|772|1094|1648|2326|8384ab5c46c8c9b09fd3e594a07fbbf4e152033eb454f92947b650d0ead0333f
R07|LOTTE|3|39|44|482|702|979|1468|1ad6afacba1d0452d938637fd6ce1d369ba6a63b0b8ee1abdc79d3c181253090
R08|SAMSUNG|2|11|33|114|780|194|1578|956f26d64bcbd2ff89963a59872cf85094ed16d937f2557ba378d54e32462493
R09|SHINHAN|2|157|180|5083|5866|10723|12292|8860157df9e6709dc086ad16a14646ed0976f957613354091e44916614bb20f5
R10|WOORI|1|23|27|793|1305|1711|2867|343467d8a4f6b4bc1602dff9ab5c2ea66a4c0a68d40ceb1071b2e64eff3ad4c0
R11|HANA|1|19|26|235|519|401|1036|696fae2ef42473f79198ef6dde2ff68ec469f955b2af4e992f363d61f6c845c1
R12|NH|2|91|122|3912|5343|8911|11992|fe88c11b27c1e629652b5ea81e38a6ae15dad0df5364e87fe33a983bdd38d1ed
R13|HYUNDAI|4|55|75|1179|2310|2440|4839|fc9d4007d42e230f537ffee381201485cf222e2404babdcc39c0b4552abe7abf
R14|SHINHAN|2|199|207|6585|6838|13917|14444|cbf1123b1c07df76084f2cae8007849c933115c0bdc74387783ec80bf2a36693
R15|HYUNDAI|6|90|123|2751|4089|5856|8770|c6f1ba9d0ceed33d181a167b5f7c2d680057120a8ced670c76de0dea1900504a
R16|WOORI|2|163|194|6985|7905|15597|17471|fe4b8b49c5ac85874f039c83da9f50bd73e8e48919318e88d26d45819e87dc35
R17|HYUNDAI|8|169|178|5322|5596|11167|11749|ab2d97bc8d3544370cc4c81123c29d1f6cf0f27ea5aa53a73c87af1837130736
R18|HANA|2|136|143|3946|4184|8424|8818|cd60125d3bbe46e87ed9bacf4a7843df2f7dba085b0f32f005e4a8d71e3180fe
R19|LOTTE|7|82|90|2387|2595|5165|5519|febb8ee9bf8f1c5bfbe34743c6cb3f3a5b260679b7381bc2343e748d60d32682
R20|KB|2|21|28|1143|1536|2425|3266|75f7b5fab08f0dc8934fa208addee97df1d591883111ce18018981563c585de7
R21|HANA|1|36|50|704|1261|1349|2522|fcc4e25f5bf5d7c4658099db204ee610c0a8e31ace4ee7f64a33c44812904c9d
R22|HANA|1|52|76|1263|2076|2524|4211|de71e18cf7ba5e6dcd0a297538e249028ad5536ccd834e40ecd1b068dd4aba76
R23|SAMSUNG|3|37|61|792|1675|1590|3485|b18c9aa30adc26ec44768bb94acc409ed5d69b513e4382e8b2f0c088da322d37
R24|LOTTE|3|17|21|117|239|153|440|85053cd062ee5bc58040442541fb2a1b51062c31930316f5a23badde4f6a3d3b
R25|LOTTE|3|35|44|440|702|891|1468|9e833f76d030a9fdf80a114afb759769e6c2952e0347e7c33fa56f58c92881b1
R26|NH|2|129|133|5504|5716|12331|12831|3b59c552feaf6df3c1dd5847482db79f14bb2dde524dad5e2ccab111451f7dfa"""
registry = {row["card_key"]: row for row in json.loads((OUT / "source_registry.json").read_text(encoding="utf-8"))}
line_index = {}
for line in (OUT / "raw_line_index.jsonl").read_text(encoding="utf-8").splitlines():
    row = json.loads(line); line_index[(row["card_key"], row["line_number"])] = row
source_refs = []
for text in ref_lines.splitlines():
    ref_id, card_id, page, ls, le, cs, ce, bs, be, excerpt_hash = text.split("|")
    card_key = CARD_KEYS[card_id]; source = registry[card_key]; raw_bytes = (ROOT / source["path"]).read_bytes(); raw = raw_bytes.decode("utf-8")
    page, ls, le, cs, ce, bs, be = map(int, [page, ls, le, cs, ce, bs, be])
    assert sha_bytes(raw_bytes) == source["raw_sha256"] and raw[cs:ce].encode("utf-8") == raw_bytes[bs:be]
    assert sha_bytes(raw[cs:ce].encode("utf-8")) == excerpt_hash and 0 <= cs < ce <= len(raw) and 0 <= bs < be <= len(raw_bytes)
    first = line_index[(card_key, ls)]; last = line_index[(card_key, le)]
    assert first["page_num"] == page and last["page_num"] == page and first["cp_start"] <= cs <= first["cp_end"] and last["cp_start"] <= ce <= last["cp_end"]
    assert first["byte_start"] <= bs <= first["byte_end"] and last["byte_start"] <= be <= last["byte_end"]
    source_refs.append({"source_ref_id":ref_id,"card_id":card_id,"card_key":card_key,"source_path":source["path"],"source_sha256":source["raw_sha256"],"page_num":page,"line_start":ls,"line_end":le,"cp_start":cs,"cp_end":ce,"byte_start":bs,"byte_end":be,"excerpt_sha256":excerpt_hash,"support_logic":"manual atomic-claim locator; independent reviewer must confirm predicate and conditions"})
refs_by_id = {row["source_ref_id"]: row for row in source_refs}
assert len(source_refs) == 26

def claim(claim_id, card_id, predicate, value, unit, conditions, criticality, requirement, refs):
    return {"atomic_claim_id":claim_id,"card_id":card_id,"card_key":CARD_KEYS[card_id],"predicate":predicate,"canonical_value":value,"unit":unit,"conditions":conditions,"criticality":criticality,"requirement":requirement,"source_ref_ids":refs,"provenance":{"annotator":"data_analyst_custom_agent","reviewer":None,"source_audit_status":"single_analyst_full_corpus_audit_pending_independent_review","ranking_blind":True}}

claims = [
claim("C01","BC","air_money_fractional_rounding","round_to_nearest_whole_point","AirMoney point",["per purchase","day after payment"],"critical_numeric","required",["R01"]),
claim("C02","NH","family_card_issuance",False,"boolean",[],"critical_eligibility","required",["R02"]),
claim("C03","HANA","discounted_expense_sales_count_toward_prior_spend",False,"boolean",["operating/required-expense discount sales"],"critical_eligibility","required",["R03"]),
claim("C04","HYUNDAI","unrepaid_emergency_mpoints_after_24_months","cash_charge_after_existing_mpoints","KRW at 1Mpoint=1KRW",["remaining balance after at most 24 months"],"critical_numeric","required",["R04"]),
claim("C05","IBK","primary_family_spend_aggregation_for_service",False,"boolean",["family-card service calculation"],"critical_eligibility","required",["R05"]),
claim("C06","KB","caribbean_bay_on_site_discount_rate",0.30,"ratio",["on-site"],"critical_numeric","required",["R06"]),
claim("C07","LOTTE","combined_monthly_discount_cap",13000,"KRW/month",["restaurant, delivery, coffee, membership combined","prior month 400000 KRW"],"critical_numeric","required",["R07"]),
claim("C08","SAMSUNG","auto_best_5pct_available_to_family_card",False,"boolean",["highest-spend 5% area"],"critical_eligibility","required",["R08"]),
claim("C09","SHINHAN","animal_hospital_in_hospital_pharmacy_discount",False,"boolean",["hospital/pharmacy 10%"],"critical_eligibility","required",["R09"]),
claim("C10","SHINHAN","dental_clinic_in_hospital_pharmacy_discount",True,"boolean",["hospital/pharmacy 10%"],"critical_eligibility","required",["R09"]),
claim("C11","WOORI","domestic_pg_overseas_site_fee_waiver",False,"boolean",["overseas site approved through domestic PG"],"critical_eligibility","required",["R10"]),
claim("C12","WOORI","domestic_pg_overseas_site_extra_mileage",False,"boolean",["overseas site approved through domestic PG"],"critical_eligibility","required",["R10"]),
claim("C13","HANA","vat_refund_record_auto_classification",True,"boolean",["Tax Diet enrollment","no spend requirement"],"required_fact","qualification",["R11"]),
claim("C14","HANA","commercial_district_analysis",3,"uses/month",["Tax Diet","no spend requirement"],"critical_numeric","qualification",["R11"]),
claim("C15","NH","smart_cashback_destination","designated_namu_securities_demand_deposit_account","account",["account opened/designated","prior month 400000 KRW"],"critical_eligibility","qualification",["R12"]),
claim("C16","NH","smart_cashback_cashability",True,"freely_cashable",["designated deposit account"],"required_fact","qualification",["R12"]),
claim("C17","KB","theme_park_discount",{"everland":0.50,"caribbean_bay":0.30},"ratio",["on-site","separate cap"],"critical_numeric","qualification",["R06"]),
claim("C18","HYUNDAI","wellness_mpoint_rate",0.10,"ratio",["prior month 1000000 KRW","monthly 10000 Mpoints by area","fitness/pilates/yoga/tennis/swimming"],"critical_numeric","qualification",["R13"]),
claim("C19","HYUNDAI","ai_subscription_mpoint_rate",0.10,"ratio",["ChatGPT/Perplexity/Google One","official app/site subscription","prior month 1000000 KRW","monthly 10000 Mpoints"],"critical_numeric","qualification",["R13"]),
claim("C20","SHINHAN","intake_mall_discount_rate",0.20,"ratio",["direct access","prior-month credit spend","four uses/month"],"critical_numeric","qualification",["R14"]),
claim("C21","WOORI","overseas_service_fee_waiver",{"mastercard_fee":0.01,"issuer_fee":0.003},"ratio_waived",["overseas sale","domestic PG/ATM/cash advance excluded"],"critical_numeric","qualification",["R10"]),
claim("C22","HYUNDAI","annual_shopping_hotel_travel_voucher",150000,"KRW/year",["shopping/hotel/travel or 200000 Mpoints","first year cumulative 1000000 KRW","later year previous 12000000 KRW"],"critical_numeric","qualification",["R15"]),
claim("C23","WOORI","airport_lounge_companion_included",1,"companion",["cardholder plus one","two uses/year","prior domestic spend 500000 KRW","cardholder card"],"critical_numeric","qualification",["R16"]),
claim("C24","HYUNDAI","airport_hotel_valet",5,"uses/month_combined",["prior month 500000 KRW","cardholder/own vehicle","parking fee separate"],"critical_numeric","qualification",["R17"]),
claim("C25","HANA","family_card_annual_fee",0,"KRW/year",["sole proprietor"],"critical_numeric","qualification",["R18"]),
claim("C26","LOTTE","family_card_annual_fee",0,"KRW/year",[],"critical_numeric","qualification",["R19"]),
claim("C27","KB","next_year_base_annual_fee_waiver",True,"boolean",["not first year","annual 1000000 KRW","cash advance included","base fee only"],"critical_eligibility","qualification",["R20"]),
claim("C28","SHINHAN","hospital_pharmacy_discount_rate",0.10,"ratio",["once/day five/month by area","approval 10000 KRW","animal hospital excluded","dental/oriental included"],"critical_numeric","qualification",["R09"]),
claim("C29","SHINHAN","laundry_discount_rate",0.10,"ratio",["laundry","once/day five/month by area","approval 10000 KRW"],"critical_numeric","qualification",["R09"]),
claim("C30","HANA","ev_charging_discount_rate",0.05,"ratio",["prior month 500000 KRW","management-fee charging excluded","registered merchant"],"critical_numeric","qualification",["R21"]),
claim("C31","HANA","four_social_insurance_autopay_discount_rate",0.03,"ratio",["national/employment/industrial/health autopay","prior month 500000 KRW"],"critical_numeric","qualification",["R22"]),
claim("C32","KB","major_category_interest_free_installment",[2,3],"months",["department store/large discount etc.","KB industry code"],"critical_numeric","qualification",["R06"]),
claim("C33","SAMSUNG","largest_retail_category_auto_discount_rate",0.05,"ratio",["department store/discount store/supermarket greatest one","offline","prior month 400000 KRW","family excluded"],"critical_numeric","qualification",["R08"]),
claim("C34","SAMSUNG","ev_charging_discount_rate",0.025,"ratio",["prior month 400000 KRW","registered operator","management-fee/portable excluded"],"critical_numeric","qualification",["R23"]),
claim("C35","LOTTE","restaurant_discount_rate",0.60,"ratio",["restaurant industry","prior month 400000 KRW","combined cap 13000 KRW","bar/coffee/tenant excluded"],"critical_numeric","qualification",["R24"]),
claim("C36","LOTTE","paid_membership_discount_rate",0.60,"ratio",["Rocket Wow/Naver Plus","prior month 400000 KRW","combined cap 13000 KRW"],"critical_numeric","qualification",["R25"]),
claim("C37","NH","smart_cashback_top_two_categories",{"rank1":0.08,"rank2":0.04},"ratio",["account opened/designated","prior month 400000 KRW","monthly cap"],"critical_numeric","qualification",["R12"]),
claim("C38","NH","namu_members_monthly_fee_cashback",2900,"KRW/month_max",["actual paid amount","prior month 400000 KRW","registration month/free period excluded"],"critical_numeric","qualification",["R26"]),
]
claims_by_id = {row["atomic_claim_id"]: row for row in claims}
assert len(claims_by_id) == 38 and all(ref in refs_by_id for row in claims for ref in row["source_ref_ids"]) and all(row["card_id"] == refs_by_id[ref]["card_id"] for row in claims for ref in row["source_ref_ids"])

policies = {
"AP_DIRECT":{"required":["designated-card claim direct answer","required condition","raw citation"],"optional":["necessary limitation"],"forbidden":["other-card recommendation","unsupported value","chunk-ID gold"],"abstain_when":"required claim locator unsupported","output":"direct"},
"AP_REC_POS":{"required":["all expected cards up to three","per-card graph reason and raw citation"],"optional":[],"forbidden":["unexpected card","unsupported ranking","cross-card evidence attribution"],"abstain_when":"zero supported cards","output":"recommendation"},
"AP_ABSTAIN":{"required":["no qualifying card established within the ten-source closed corpus"],"optional":["near-card exclusion reason"],"forbidden":["recommendation","external inference","invented expected"],"abstain_when":"always","output":"abstention"},
}

query_lines = """D01|biz Air Money 법인카드는 매입 건별 Air Money가 1원 미만일 때 어떻게 처리하나?|direct|entity,numeric_condition|BC|BC:all:C01|C01|AP_DIRECT|
D02|나무 NH농협카드는 가족카드를 따로 발급할 수 있나?|direct|entity,numeric_condition|NH|NH:all:C02|C02|AP_DIRECT|
D03|하나 더 소호에서 운영경비·필수경비 할인을 받은 매출은 다음 달 실적에 포함되나?|direct|entity,numeric_condition|HANA|HANA:all:C03|C03|AP_DIRECT|
D04|the Orange의 M 긴급적립을 24개월 안에 모두 갚지 못하면 남은 포인트는 어떻게 처리되나?|direct|entity,numeric_condition,semantic|HYUNDAI|HYUNDAI:all:C04|C04|AP_DIRECT|
D05|IBK포인트 3.8 가족카드는 본인카드와 이용금액을 합산해 서비스를 계산하나?|direct|entity,numeric_condition|IBK|IBK:all:C05|C05|AP_DIRECT|
D06|프랜드카드로 캐리비안베이를 이용하면 현장에서 몇 퍼센트 할인되나?|direct|entity,numeric_condition|KB|KB:all:C06|C06|AP_DIRECT|
D07|LOCA LIKIT Eat의 음식점·배달앱·커피·멤버십 할인은 한 달에 합쳐 최대 얼마까지 받을 수 있나?|direct|entity,numeric_condition|LOTTE|LOTTE:all:C07|C07|AP_DIRECT|
D08|삼성 iD ALL 가족카드에도 가장 많이 쓴 쇼핑 영역 5% 자동 할인이 제공되나?|direct|entity,numeric_condition,semantic|SAMSUNG|SAMSUNG:all:C08|C08|AP_DIRECT|
D09|Mr.Life의 병원·약국 할인에서 동물병원과 치과는 각각 대상인가?|direct|entity,numeric_condition,semantic|SHINHAN|SHINHAN:all:C09,C10|C09,C10|AP_DIRECT|
D10|EVERY MILE SKYPASS로 해외 사이트에서 결제했지만 국내 결제대행사로 승인되면 해외 수수료 면제와 추가 마일 적립을 받을 수 있나?|direct|entity,numeric_condition,semantic,mixed|WOORI|WOORI:all:C11,C12|C11,C12|AP_DIRECT|
O01|부가세 신고 자료를 자동으로 나눠 주거나 상권 정보를 분석해 주는 카드를 추천해줘.|OR|semantic|HANA|HANA:any:C13,C14|C13,C14|AP_REC_POS|
O02|카드 사용 보상을 증권계좌로 현금 입금해 주거나 현금처럼 바로 찾을 수 있는 카드를 추천해줘.|OR|semantic|NH|NH:any:C15,C16|C15,C16|AP_REC_POS|
O03|놀이공원 현장 할인이나 헬스장·필라테스·수영장 같은 운동 적립이 있는 카드를 추천해줘.|OR|semantic|KB,HYUNDAI|KB:any:C17;HYUNDAI:any:C18|C17,C18|AP_REC_POS|
O04|AI 구독료 적립이나 인테이크 건강식품 온라인몰 할인을 제공하는 카드를 추천해줘.|OR|entity,semantic|HYUNDAI,SHINHAN|HYUNDAI:any:C19;SHINHAN:any:C20|C19,C20|AP_REC_POS|
O05|마스터카드 해외 결제 수수료를 면제해 주거나 쇼핑·호텔·여행 바우처를 주는 카드를 추천해줘.|OR|entity,numeric_condition,semantic|WOORI,HYUNDAI|WOORI:any:C21;HYUNDAI:any:C22|C21,C22|AP_REC_POS|
O06|더라운지 앱으로 동반 1인까지 무료이거나 공항·호텔 발레파킹이 무료인 카드를 추천해줘.|OR|entity,numeric_condition|WOORI,HYUNDAI|WOORI:any:C23;HYUNDAI:any:C24|C23,C24|AP_REC_POS|
O07|가족카드 연회비가 없거나 다음 연도 기본 연회비를 실적으로 면제받을 수 있는 카드를 추천해줘.|OR|numeric_condition|HANA,LOTTE,KB|HANA:any:C25;LOTTE:any:C26;KB:any:C27|C25,C26,C27|AP_REC_POS|
O08|병원·약국 또는 세탁소에서 결제일 할인을 받을 수 있는 카드를 추천해줘.|OR|semantic|SHINHAN|SHINHAN:any:C28,C29|C28,C29|AP_REC_POS|
O09|AWS·Azure 같은 클라우드 서버 요금이나 공유오피스 이용료를 직접 할인해 주는 카드를 추천해줘.|OR|entity,semantic||| |AP_ABSTAIN|cloud_server_fee_direct_discount,coworking_fee_direct_discount
O10|동물병원 진료비나 반려동물 보험료를 직접 할인해 주는 카드를 추천해줘.|OR|numeric_condition,semantic,mixed|||C09|AP_ABSTAIN|animal_hospital_direct_discount,pet_insurance_direct_discount
A01|병원·약국과 세탁소를 둘 다 할인해 주는 카드를 추천해줘.|AND|numeric_condition,semantic|SHINHAN|SHINHAN:all:C28,C29|C28,C29|AP_REC_POS|
A02|전기차 충전과 4대 사회보험 자동납부를 둘 다 할인해 주는 카드를 추천해줘.|AND|numeric_condition,semantic|HANA|HANA:all:C30,C31|C30,C31|AP_REC_POS|
A03|에버랜드·캐리비안베이 할인과 주요 업종 2~3개월 무이자 할부가 모두 있는 카드를 추천해줘.|AND|entity,numeric_condition,semantic|KB|KB:all:C17,C32|C17,C32|AP_REC_POS|
A04|가장 많이 쓴 백화점·마트 영역 자동 할인과 전기차 충전 할인을 둘 다 제공하는 카드를 추천해줘.|AND|numeric_condition,semantic|SAMSUNG|SAMSUNG:all:C33,C34|C33,C34|AP_REC_POS|
A05|음식점과 쿠팡 로켓와우·네이버플러스 멤버십을 둘 다 할인하는 카드를 추천해줘.|AND|entity,numeric_condition,semantic|LOTTE|LOTTE:all:C35,C36|C35,C36|AP_REC_POS|
A06|나무증권계좌로 들어오는 스마트 캐시백과 나무멤버스 이용료 캐시백이 모두 있는 카드를 추천해줘.|AND|entity,numeric_condition,semantic|NH|NH:all:C37,C38|C37,C38|AP_REC_POS|
A07|해외 결제 수수료 면제와 국내 공항라운지 무료 이용을 모두 제공하는 카드를 추천해줘.|AND|entity,numeric_condition|WOORI|WOORI:all:C21,C23|C21,C23|AP_REC_POS|
A08|ChatGPT·Perplexity AI·Google One 구독 적립과 헬스장·필라테스·요가 적립을 모두 제공하는 카드를 추천해줘.|AND|entity,numeric_condition,semantic|HYUNDAI|HYUNDAI:all:C19,C18|C19,C18|AP_REC_POS|
A09|놀이공원 할인과 병원·약국 할인을 모두 제공하는 카드를 추천해줘.|AND|numeric_condition,semantic,mixed|||C17,C28|AP_ABSTAIN|theme_park_discount,hospital_pharmacy_discount
A10|전기차 충전 할인과 공항·호텔 발레파킹을 모두 제공하는 카드를 추천해줘.|AND|numeric_condition,semantic|||C30,C34,C24|AP_ABSTAIN|ev_charging_discount,airport_or_hotel_valet"""
records = []
for text in query_lines.splitlines():
    rid, question, family, axes, expected, override, claim_ids, policy, negative_type = text.split("|")
    expected_ids = expected.split(",") if expected else []
    graph = {}
    if override.strip():
        for item in override.split(";"):
            card_id, operator, leaves = item.split(":"); graph[card_id] = {"all_of": leaves.split(",")} if operator == "all" else {"any_of": leaves.split(",")}
    records.append({"record_id":rid,"split":"evaluation","question":question,"query_family":family,"axes":axes.split(","),"expected_card_ids":expected_ids,"qualification_overrides":graph,"claim_ids":claim_ids.split(",") if claim_ids.strip() else [],"answer_policy_id":policy,"negative_type":negative_type or None,"provenance":{"annotator":"data_analyst_custom_agent","reviewer":None,"status":DRAFT_STATUS,"ranking_blind":True}})
assert len(records) == 30 and len({r["record_id"] for r in records}) == 30

qualification_rows = []; master_rows = []; review_rows = []; duplicate_rows = []
demo_question = json.loads((OUT / "demo_public_view.json").read_text(encoding="utf-8"))["question"]
demo_norm = norm_query(demo_question)
for record in records:
    expected_set = set(record["expected_card_ids"])
    assert set(record["qualification_overrides"]) == expected_set
    for card_id in CORPUS_CARD_IDS:
        graph = record["qualification_overrides"].get(card_id, {"all_of": []})
        qualified = card_id in expected_set
        leaves = graph.get("all_of", graph.get("any_of", []))
        assert (not qualified and not leaves) or (qualified and leaves and all(claims_by_id[c]["card_id"] == card_id for c in leaves))
        qualification_rows.append({"record_id":record["record_id"],"card_id":card_id,"card_key":CARD_KEYS[card_id],"qualified":qualified,"qualification_graph":graph,"base":"ALL_FALSE_10","override_applied":qualified,"status":DRAFT_STATUS})
    source_ref_ids = sorted({ref for cid in record["claim_ids"] for ref in claims_by_id[cid]["source_ref_ids"]})
    master_rows.append({"record_id":record["record_id"],"split":"evaluation","expected_card_ids":record["expected_card_ids"],"qualification_by_card_ref":"qualification_by_card.jsonl","qualification_graph":record["qualification_overrides"],"atomic_claim_ids":record["claim_ids"],"source_ref_ids":source_ref_ids,"answer_policy_id":record["answer_policy_id"],"scoring_unit":"atomic_claim_id + boolean graph + source_ref_id","forbidden_scoring_units":["chunk_id","contiguous source sentence"],"status":DRAFT_STATUS})
    review_rows.append({"record_id":record["record_id"],"question":record["question"],"expected_card_ids":record["expected_card_ids"],"all_card_qualifications":[row for row in qualification_rows if row["record_id"] == record["record_id"]],"atomic_claims":[claims_by_id[cid] for cid in record["claim_ids"]],"source_refs":[refs_by_id[rid] for rid in source_ref_ids],"reviewer":None,"review_status":DRAFT_STATUS})
    duplicate_rows.append({"record_id":record["record_id"],"query_normalized_sha256":compact_hash(norm_query(record["question"])),"demo_exact_normalized_collision":norm_query(record["question"]) == demo_norm,"demo_claim_reuse":False,"existing_query_inventory":{"status":"proposal_asserted_pending_independent_review","exact_normalized_collision":False,"predicate_set_equality":False,"predicate_jaccard":"not_materialized_without permitted inventory payload","manual_semantic_intent":"pending_independent_review"}})
assert len(qualification_rows) == 300 and all(sum(row["qualified"] for row in qualification_rows if row["record_id"] == record["record_id"]) == len(record["expected_card_ids"]) for record in records)
assert not any(row["demo_exact_normalized_collision"] for row in duplicate_rows)

families = Counter(record["query_family"] for record in records); negatives = Counter(record["query_family"] for record in records if not record["expected_card_ids"]); sizes = Counter(len(record["expected_card_ids"]) for record in records)
assert families == Counter({"direct":10,"OR":10,"AND":10}) and negatives == Counter({"OR":2,"AND":2}) and sizes == Counter({1:21,2:4,3:1,0:4}) and all(len(r["expected_card_ids"]) <= 3 for r in records)
axis_counts = {family: Counter(axis for record in records if record["query_family"] == family for axis in record["axes"]) for family in families}
axis_minimum = {"direct":{"entity":10,"numeric_condition":10,"semantic":4,"mixed":1},"OR":{"entity":3,"numeric_condition":4,"semantic":8,"mixed":1},"AND":{"entity":5,"numeric_condition":10,"semantic":9,"mixed":1}}
assert all(axis_counts[family][axis] >= count for family, mins in axis_minimum.items() for axis, count in mins.items())

old_lineage = [json.loads(line) for line in (OUT / "old_chunk_lineage.jsonl").read_text(encoding="utf-8").splitlines()]
struct_lineage = [json.loads(line) for line in (OUT / "struct_chunk_lineage.jsonl").read_text(encoding="utf-8").splitlines()]
def project(ref, lineage):
    candidate = [row["chunk_id"] for row in lineage if row["source_sha256"] == ref["source_sha256"] and row["status"] == "exact" and any(component["cp_start"] < ref["cp_end"] and ref["cp_start"] < component["cp_end"] for component in row["component_intervals"])]
    return sorted(candidate)
projection_rows = []
for record in records:
    source_ref_ids = next(row["source_ref_ids"] for row in master_rows if row["record_id"] == record["record_id"])
    for corpus_name, lineage in [("OLD", old_lineage), ("STRUCT", struct_lineage)]:
        by_ref = {rid: project(refs_by_id[rid], lineage) for rid in source_ref_ids}
        all_ids = sorted({cid for ids in by_ref.values() for cid in ids})
        projection_rows.append({"record_id":record["record_id"],"corpus":corpus_name,"method":"source_sha256_and_half_open_codepoint_interval_overlap_only","string_matching_used":False,"fuzzy_matching_used":False,"lineage_statuses":sorted(set(row["status"] for row in lineage)),"source_ref_ids":source_ref_ids,"chunk_ids_by_source_ref":by_ref,"projected_chunk_ids":all_ids,"coverage_status":"projected" if all_ids else "projection_ceiling_miss","dataset_invalid":False})
assert all(row["method"].endswith("overlap_only") and not row["string_matching_used"] and not row["fuzzy_matching_used"] for row in projection_rows)

write_json(OUT / "evaluation_queries.json", {"schema_version":"integrated_holdout_master_v1","status":DRAFT_STATUS,"records":records})
write_jsonl(OUT / "master_gold.jsonl", master_rows)
write_json(OUT / "source_refs.json", {"schema_version":"integrated_holdout_master_v1","status":DRAFT_STATUS,"refs":source_refs})
write_json(OUT / "atomic_claims.json", {"schema_version":"integrated_holdout_master_v1","status":DRAFT_STATUS,"claims":claims})
write_json(OUT / "answer_policies.json", {"schema_version":"integrated_holdout_master_v1","policies":policies})
write_jsonl(OUT / "qualification_by_card.jsonl", qualification_rows)
write_jsonl(OUT / "annotation_audit.jsonl", [{"record_id":r["record_id"],"annotator":"data_analyst_custom_agent","reviewer":None,"status":DRAFT_STATUS,"limitation":"single analyst; independent full-corpus review required","ranking_blind":True} for r in records])
write_jsonl(OUT / "candidate_projection.jsonl", projection_rows)
write_jsonl(OUT / "query_duplicate_audit.jsonl", duplicate_rows)
write_jsonl(OUT / "draft_review_packet.jsonl", review_rows)

scoring_contract = {"status":DRAFT_STATUS,"evaluator_contract_only":True,"retrieval_metrics":{"positive":"CardP@3, CardR@3, capped Recall@3, supported-card P/R, required-claim coverage, fully-supported rate","negative":"no-card-output accuracy, false recommendation rate, retrieval negative-card count@3","candidate_ceiling":"same metrics before rerank; candidate_zero and ranking_catastrophic distinguished"},"llm_metrics":{"all30_primary":"answer card P/R, required fact recall, complete condition, unsupported claim rate, critical numeric error, citation ownership/coverage, abstention accuracy","positive_exact":"expected set exact + graph complete + no condition omission + no unsupported/critical numeric error + citation ownership/coverage","negative_exact":"explicit abstention + zero recommendation + zero unsupported claim"},"invalid_vs_ceiling":{"dataset_invalid":["source SHA mismatch","invalid/range-reversed locator","UTF-8 decode failure","excerpt hash mismatch","card/source mismatch"],"projection_ceiling_miss":"valid master locator with no OLD/STRUCT projection; do not delete or invalidate record"},"evaluation":{"unconditional":"all 30 is primary","conditional":"only graph-complete positive evidence is diagnostic; report intersection and n"},"deferred_execution_seal":["candidate configuration","prompt","model","K/rank depth","absolute selection thresholds"]}
write_json(OUT / "draft_scoring_contract.json", scoring_contract)

manifest_files = sorted(NEW_FILES | {"draft_scoring_contract.json"})
manifest = {"schema_version":"integrated_holdout_master_v1","status":DRAFT_STATUS,"corpus_card_ids":CORPUS_CARD_IDS,"max_output_cards":3,"lineage_inputs":lineage_hashes,"source_inputs_before":source_inputs_before,"artifacts":{name:sha_file(OUT / name) for name in manifest_files if name not in {"draft_dataset_manifest.json","draft_integrity.json"}},"query_plaintext_policy":"evaluation query plaintext only in restricted artifacts; excluded from README, manifest and logs","demo_exclusion":{"demo_records_in_evaluation_queries":0,"demo_records_in_master_gold":0,"demo_records_in_projections":0,"demo_records_in_scoring_payload":0},"execution":{"environment":"skn25","network":0,"api_requests":0,"gpu":0,"embedding":0,"retrieval":0,"model":0}}
write_json(OUT / "draft_dataset_manifest.json", manifest)
assert all(sha_file(OUT / name) == digest for name, digest in lineage_hashes.items()) and {path.relative_to(ROOT).as_posix():sha_file(path) for path in [OLD_PATH, OLD_INPUT, STRUCT_PATH, STRUCT_HIERARCHY]} == source_inputs_before
integrity = {"status":"PASS","dataset_status":DRAFT_STATUS,"manifest_sha256":sha_file(OUT / "draft_dataset_manifest.json"),"assertions":{"records":30,"qualification_rows":300,"source_refs":26,"atomic_claims":38,"old_struct_source_inputs_preserved":True,"existing_lineage_outputs_preserved":True,"projection_deterministic_overlap_only":True,"demo_excluded":True,"api_network_gpu_embedding_retrieval_model_zero":True},"projection_coverage":{corpus:Counter(row["coverage_status"] for row in projection_rows if row["corpus"] == corpus) for corpus in ["OLD","STRUCT"]}}
write_json(OUT / "draft_integrity.json", integrity)
print(json.dumps({"status":"PASS","records":30,"qualification_rows":300,"projection":integrity["projection_coverage"],"draft_status":DRAFT_STATUS}, ensure_ascii=False, default=dict))


{"status": "PASS", "records": 30, "qualification_rows": 300, "projection": {"OLD": {"projected": 29, "projection_ceiling_miss": 1}, "STRUCT": {"projected": 29, "projection_ceiling_miss": 1}}, "draft_status": "pending_independent_review"}


## Revision v2 — proposal canonical materialization

Codex coder agent가 proposal.part01~08만 source of truth로 병합합니다. 이전 draft 셀은 provenance로 남기되 이번 실행에서는 사용하지 않습니다. 이 self-contained 셀은 restricted core 전체를 staging 후 교체하고 기존 lineage/demo를 byte-exact 보존합니다. 상태는 독립 검토 전까지 `pending_independent_review`입니다.


In [1]:
# Integrated holdout master v2: proposal-only CPU/offline materialization.
from __future__ import annotations
import copy, csv, hashlib, json, os, re, shutil, tempfile, unicodedata
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "notebooks").is_dir(): ROOT = cwd
elif cwd.name == "notebooks" and (cwd.parent / "notebooks").is_dir(): ROOT = cwd.parent
else: raise RuntimeError("Run from repository root or notebooks/")
NOTEBOOK = ROOT / "notebooks/27_integrated_holdout_dataset.ipynb"
OUT = ROOT / "notebooks/data/27_integrated_holdout_dataset"
STATUS, VERSION, CELL_ID = "pending_independent_review", "integrated_holdout_master_v2", "27-v2-proposal-materialization"

def need(ok, message):
    if not ok: raise RuntimeError(message)
def canon(value): return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
def sha_bytes(value): return hashlib.sha256(value).hexdigest()
def sha_file(path): return sha_bytes(path.read_bytes())
def put_json(path, value): path.write_text(canon(value), encoding="utf-8")
def put_jsonl(path, rows): path.write_text("\n".join(map(canon, rows)) + "\n", encoding="utf-8")

part_paths = [OUT / f"proposal.part{i:02d}.json" for i in range(1, 9)]
need(all(p.is_file() for p in part_paths), "Eight proposal parts required")
parts = [json.loads(p.read_bytes()) for p in part_paths]
part_hashes = {p.name: sha_file(p) for p in part_paths}
need(parts[7]["counts"]["overrides"] == 49, "PART8 corrected override count must be 49")

preserved = ["demo_master_claim.json","demo_projection.json","demo_public_view.json","integrity.json","lineage_contract.json","lineage_quality_audit.json","old_chunk_lineage.jsonl","raw_line_index.jsonl","run_manifest.json","source_registry.json","struct_chunk_lineage.jsonl"]
preserved_before = {n: sha_file(OUT/n) for n in preserved}
restricted = ["README.md","annotation_audit.jsonl","answer_policies.json","atomic_claims.json","candidate_projection.jsonl","draft_dataset_manifest.json","draft_integrity.json","draft_review_packet.jsonl","draft_scoring_contract.json","evaluation_queries.json","master_gold.jsonl","proposal_canonical.json","qualification_by_card.jsonl","query_duplicate_audit.jsonl","source_components.jsonl","source_refs.json"]
restricted_before_current = {n: sha_file(OUT/n) for n in restricted if (OUT/n).is_file()}
previous_manifest = json.loads((OUT/"draft_dataset_manifest.json").read_text()) if (OUT/"draft_dataset_manifest.json").is_file() else {}
replaced_v1_hashes = previous_manifest.get("replaced_restricted_v1_raw_sha256", restricted_before_current) if previous_manifest.get("schema_version")==VERSION else restricted_before_current

base = parts[0]
sf, cf, clf, rf, qf = (base["schemas"][k] for k in ("source_component","condition","claim","record","qualification_row"))
source_values = {}
for part in parts[:4]:
    raw = part["source_components"]
    items = raw.items() if isinstance(raw, dict) else ((row[0], row[1:]) for row in raw)
    for sid, values in items:
        need(sid not in source_values, f"duplicate {sid}")
        source_values[sid] = values
claims_raw = {cid: values for part in parts[4:6] for cid, values in part["claims"].items()}
records_raw = parts[6]["records"]
record_ids = [*(f"D{i:02d}" for i in range(1,11)),*(f"O{i:02d}" for i in range(1,11)),*(f"A{i:02d}" for i in range(1,11))]
need(set(source_values)=={f"S{i:03d}" for i in range(1,73)}, "S001-S072 required")
need(set(claims_raw)=={f"C{i:02d}" for i in range(1,48)}, "C01-C47 required")
need(set(records_raw)==set(record_ids), "30 record IDs required")

proposal = {k:base[k] for k in ("schema_version","corpus_card_ids","max_output_cards","conversions","schemas")}
proposal.update({"source_components":source_values,"claims":claims_raw,"records":records_raw,"qualification_compact":parts[7]["qualification_compact"]})
for key in ("projection_contract","boolean_scoring","retrieval_metrics","llm_metrics","unit_normalization","denominators","selection_contract","distribution","duplicate_audit","source_locator_audit"):
    proposal[key] = parts[7][key]
proposal["integrity"] = copy.deepcopy(parts[7]["integrity"])
proposal["provenance"] = {"proposal_part_raw_sha256":part_hashes,"part8_correction_history":["removed one extra closing brace after denominators","corrected overrides count 48 to 49 without deleting any override row"]}
without_self = copy.deepcopy(proposal); without_self["integrity"].pop("canonical_sha256",None)
proposal_sha = sha_bytes(canon(without_self).encode())
proposal["integrity"]["canonical_sha256"] = proposal_sha
generation = f"integrated_holdout_master_v2_{proposal_sha[:16]}"

registry = {r["card_key"]:r for r in json.loads((OUT/"source_registry.json").read_text())}
raw_cache, sources = {}, []
for sid in sorted(source_values):
    values = source_values[sid]; need(len(values)==len(sf),f"bad source width {sid}")
    row = {"source_component_id":sid,**dict(zip(sf,values))}
    reg = registry.get(row["card_key"]); need(reg and reg["raw_sha256"]==row["source_sha256"],f"registry mismatch {sid}")
    raw = raw_cache.setdefault(reg["path"],(ROOT/reg["path"]).read_bytes()); text = raw.decode("utf-8")
    need(sha_bytes(raw)==row["source_sha256"],f"source hash {sid}")
    need(0<=row["cp_start"]<row["cp_end"]<=len(text) and 0<=row["byte_start"]<row["byte_end"]<=len(raw),f"interval {sid}")
    excerpt = raw[row["byte_start"]:row["byte_end"]]
    need(text[row["cp_start"]:row["cp_end"]].encode()==excerpt and sha_bytes(excerpt)==row["excerpt_sha256"],f"excerpt {sid}")
    sources.append({**row,"source_path":reg["path"],"status":STATUS,"reviewer_proposal":None,"generation_id":generation})
by_source = {r["source_component_id"]:r for r in sources}
card_keys = {}
for r in sources:
    card_keys.setdefault(r["card_id"],r["card_key"]); need(card_keys[r["card_id"]]==r["card_key"],"card key mismatch")
need(base["corpus_card_ids"]==parts[7]["qualification_compact"]["card_order"],"card order mismatch")

claims = []
for cid in sorted(claims_raw):
    values=claims_raw[cid]; need(len(values)==len(clf),f"bad claim width {cid}")
    row=dict(zip(clf,values)); row["conditions"]=[dict(zip(cf,c)) for c in row["conditions"]]
    need(row["required_component_ids"] and all(s in by_source for s in row["required_component_ids"]),f"claim refs {cid}")
    need(all(by_source[s]["card_id"]==row["card_id"] for s in row["required_component_ids"]),f"cross card source {cid}")
    claims.append({"atomic_claim_id":cid,"card_key":card_keys[row["card_id"]],**row,"status":STATUS,"reviewer_proposal":None,"generation_id":generation})
by_claim={r["atomic_claim_id"]:r for r in claims}

records=[]
for rid in record_ids:
    values=records_raw[rid]; need(len(values)==len(rf),f"bad record width {rid}")
    row=dict(zip(rf,values)); need(all(c in by_claim for c in row["atomic_claim_ids"]),f"record claims {rid}")
    for card,graph in row["qualification_graphs"].items():
        need(card in row["expected_card_ids"],f"graph card {rid}")
        for branch in ("all_of","any_of"):
            for leaf in graph.get(branch,[]):
                need(len(leaf)==3 and leaf[0] in by_claim and by_claim[leaf[0]]["card_id"]==card,f"graph leaf {rid}")
    records.append({"record_id":rid,**row,"status":STATUS,"reviewer_proposal":None,"generation_id":generation})
by_record={r["record_id"]:r for r in records}

qc=parts[7]["qualification_compact"]; override_count=sum(map(len,qc["overrides"].values()))
need(len(qc["defaults"])==30 and override_count==49,"qualification compact count")
qual=[]
for rid in record_ids:
    for card in qc["card_order"]:
        overridden=card in qc["overrides"].get(rid,{})
        values=qc["overrides"].get(rid,{}).get(card,qc["defaults"][rid]); need(len(values)==len(qf),f"qualification width {rid}/{card}")
        row=dict(zip(qf,values)); need(row["reviewer_proposal"] is None and row["full_source_reviewed"] is True,f"review state {rid}/{card}")
        need(all(c in by_claim for c in row["present_claim_ids"]+row["missing_claim_ids"]),f"qualification claim {rid}/{card}")
        need(all(s in by_source for s in row["near_miss_source_ref_ids"]),f"qualification source {rid}/{card}")
        qual.append({"record_id":rid,"card_id":card,"card_key":card_keys[card],**row,"override_applied":overridden,"status":STATUS,"generation_id":generation})
need(len(qual)==300 and len({(r["record_id"],r["card_id"]) for r in qual})==300,"qualification 300 unique")
for rid in record_ids:
    need({r["card_id"] for r in qual if r["record_id"]==rid and r["qualified"]}==set(by_record[rid]["expected_card_ids"]),f"qualified set {rid}")

master=[]
for r in records:
    sids=sorted({s for c in r["atomic_claim_ids"] for s in by_claim[c]["required_component_ids"]})
    master.append({"schema_version":VERSION,"generation_id":generation,"record_id":r["record_id"],"task_family":r["task_family"],"expected_card_ids":r["expected_card_ids"],"atomic_claim_ids":r["atomic_claim_ids"],"required_source_component_ids":sids,"requested_predicate_ids":r["requested_predicate_ids"],"qualification_graphs":r["qualification_graphs"],"answer_policy_ref":r["answer_policy_ref"],"negative_type":r["negative_type"],"negative_record_status":"not_applicable_negative" if not r["expected_card_ids"] else None,"qualification_by_card_ref":"qualification_by_card.jsonl","scoring_unit":"atomic_claim_id","forbidden_scoring_units":["chunk_id","contiguous_source_sentence"],"status":STATUS,"reviewer_proposal":None})

policies={
"AP_DIRECT":{"task_family":"direct_single_card_fact","max_output_cards":1,"required":["named-card graph","all required conditions","owned source-component citation"],"forbidden":["other-card recommendation","unsupported value"],"abstain_when":"required graph unsupported"},
"AP_REC_POS":{"task_family":"multi_card_recommendation","max_output_cards":base["max_output_cards"],"required":["same-card graph","owned source-component citation"],"forbidden":["cross-card satisfaction","unsupported ranking"],"abstain_when":"zero supported cards"},
"AP_ABSTAIN":{"task_family":"negative","max_output_cards":0,"required":["closed-corpus insufficiency"],"forbidden":["recommendation","external inference"],"abstain_when":"always"}}
need({r["answer_policy_ref"] for r in records}==set(policies),"policy refs")

old=[json.loads(x) for x in (OUT/"old_chunk_lineage.jsonl").read_text().splitlines() if x]
struct=[json.loads(x) for x in (OUT/"struct_chunk_lineage.jsonl").read_text().splitlines() if x]
need(len(old)==327 and len(struct)==147,"lineage counts")
def project(component,lineage,levels=None):
    return sorted({r["chunk_id"] for r in lineage if r["status"]=="exact" and r["source_sha256"]==component["source_sha256"] and r["card_key"]==component["card_key"] and (levels is None or r.get("level") in levels) and any(i["cp_start"]<component["cp_end"] and component["cp_start"]<i["cp_end"] for i in r["component_intervals"])})
projections=[]
for claim in claims:
    for corpus,lineage in (("OLD",old),("STRUCT",struct)):
        full={s:project(by_source[s],lineage) for s in claim["required_component_ids"]}
        eligible={s:(project(by_source[s],lineage,{"section","benefit"}) if corpus=="OLD" else full[s]) for s in claim["required_component_ids"]}
        full_ids=sorted({x for v in full.values() for x in v}); eligible_ids=sorted({x for v in eligible.values() for x in v})
        projections.append({"schema_version":VERSION,"generation_id":generation,"projection_scope":"atomic_claim","atomic_claim_id":claim["atomic_claim_id"],"card_id":claim["card_id"],"corpus":corpus,"method":"source_sha256_card_key_half_open_codepoint_overlap_only","string_matching_used":False,"fuzzy_matching_used":False,"full_corpus_presence":{"source_component_reproduced":True,"component_chunk_ids":full,"chunk_ids":full_ids,"chunk_projection_complete":all(full.values()),"complete":True},"searchable_eligible":{"component_chunk_ids":eligible,"chunk_ids":eligible_ids,"complete":all(eligible.values()),"rule":"OLD_section_or_benefit_only" if corpus=="OLD" else "all_frozen_structural_retrieval_chunks"},"payload_evidence":{"actual_payload_chunk_ids":None,"actual_payload_frozen":False,"eligible_component_chunk_ids":eligible,"eligible_chunk_ids":eligible_ids,"eligible_complete":all(eligible.values()),"rule":"OLD_section_or_benefit_only" if corpus=="OLD" else "requires_future_frozen_bundle_membership","status":"execution_deferred_not_scored_as_present"},"candidate_projection_miss":not all(eligible.values()),"dataset_invalid":False,"status":STATUS})
negative_ids=[r["record_id"] for r in records if not r["expected_card_ids"]]
need(negative_ids==["O09","O10","A09","A10"],"negative IDs")
for rid in negative_ids:
    for corpus in ("OLD","STRUCT"):
        projections.append({"schema_version":VERSION,"generation_id":generation,"projection_scope":"negative_record","record_id":rid,"corpus":corpus,"full_corpus_presence":"not_applicable_negative","searchable_eligible":"not_applicable_negative","payload_evidence":"not_applicable_negative","dataset_invalid":False,"status":STATUS})
need(len(projections)==102,"projection count")

specs=[r for r in parts[7]["duplicate_audit"]["inventories"] if r["path"].endswith(".csv")]
need(len(specs)==4,"four query inventories")
def norm(value):
    return " ".join(re.sub(r"[^\w가-힣]+"," ",unicodedata.normalize("NFKC",str(value)).lower()).split())
inventory=[]; inventory_hashes={}
for spec in specs:
    path=ROOT/spec["path"]; need(sha_file(path)==spec["sha256"],f"inventory hash {spec['path']}")
    inventory_hashes[spec["path"]]=spec["sha256"]
    with path.open(encoding="utf-8",newline="") as f:
        inventory += [(spec["path"],r["query_id"],norm(r["query_text"])) for r in csv.DictReader(f)]
dups=[]; exact=near=0
for r in records:
    normalized=norm(r["sealed_query_text"]); tokens=set(normalized.split()); ex=[]; ne=[]; maximum=0.0
    for path,qid,text in inventory:
        if normalized==text: ex.append({"inventory_path":path,"query_id":qid})
        other=set(text.split()); score=len(tokens&other)/len(tokens|other) if tokens|other else 1.0; maximum=max(maximum,score)
        if score>=0.8: ne.append({"inventory_path":path,"query_id":qid,"token_jaccard":score})
    exact+=len(ex); near+=len(ne)
    dups.append({"schema_version":VERSION,"generation_id":generation,"record_id":r["record_id"],"query_normalized_sha256":sha_bytes(normalized.encode()),"inventory_count":4,"exact_normalized_matches":ex,"token_jaccard_threshold":0.8,"token_jaccard_matches":ne,"maximum_token_jaccard":maximum,"manual_same_semantic_matches":0,"manual_semantic_status":STATUS,"status":STATUS})
need(exact==0 and near==0,"duplicate collision")

def leaf(value,op,expected):
    if op=="eq": return value==expected
    if op=="gte": return isinstance(value,(int,float)) and value>=expected
    if op=="object_has_any_key": return isinstance(value,dict) and any(k in value and value[k] is not None for k in expected)
    raise RuntimeError(f"unsupported operator {op}")
def graph_ok(graph):
    key="all_of" if "all_of" in graph else "any_of"
    vals=[leaf(by_claim[c]["canonical_value"],op,v) for c,op,v in graph[key]]
    return all(vals) if key=="all_of" else any(vals)
for r in records:
    for card,g in r["qualification_graphs"].items(): need(graph_ok(g),f"graph execution {r['record_id']}/{card}")

scoring={"schema_version":VERSION,"generation_id":generation,"status":STATUS,"executable_reference_validated":True,"scoring_unit":"atomic_claim_id","operators":{"eq":"claim_value == expected_value","gte":"numeric claim_value >= expected_value","object_has_any_key":"requested key exists with non-null supported value"},"boolean_graph":parts[7]["boolean_scoring"],"retrieval_metrics":parts[7]["retrieval_metrics"],"llm_metrics":parts[7]["llm_metrics"],"unit_normalization":parts[7]["unit_normalization"],"denominators":parts[7]["denominators"],"selection_contract":parts[7]["selection_contract"],"projection_contract":parts[7]["projection_contract"],"execution_deferred":True}
by_record_qual={rid:[q for q in qual if q["record_id"]==rid] for rid in record_ids}
audits=[]; reviews=[]
for r in records:
    cids=r["atomic_claim_ids"]; sids=sorted({s for c in cids for s in by_claim[c]["required_component_ids"]})
    audits.append({"schema_version":VERSION,"generation_id":generation,"record_id":r["record_id"],"source_components_reproduced":len(sids),"qualification_rows_reviewed_from_proposal":10,"full_source_reviewed":all(q["full_source_reviewed"] for q in by_record_qual[r["record_id"]]),"annotator":"data_analyst_custom_agent","reviewer_proposal":None,"status":STATUS,"limitation":"single-analyst proposal pending independent review"})
    reviews.append({"schema_version":VERSION,"generation_id":generation,"record_id":r["record_id"],"sealed_query_text":r["sealed_query_text"],"task_family":r["task_family"],"expected_card_ids":r["expected_card_ids"],"atomic_claims":[by_claim[c] for c in cids],"source_components":[by_source[s] for s in sids],"qualification_rows":by_record_qual[r["record_id"]],"answer_policy_ref":r["answer_policy_ref"],"reviewer_proposal":None,"review_status":STATUS})

queries={"schema_version":VERSION,"generation_id":generation,"status":STATUS,"records":[{"record_id":r["record_id"],"sealed_query_text":r["sealed_query_text"],"task_family":r["task_family"],"axes":r["axes"],"status":STATUS,"generation_id":generation} for r in records]}
source_refs={"schema_version":VERSION,"generation_id":generation,"status":STATUS,"unit":"source_component_id","refs":sources}
claim_doc={"schema_version":VERSION,"generation_id":generation,"status":STATUS,"claims":claims}
policy_doc={"schema_version":VERSION,"generation_id":generation,"status":STATUS,"policies":policies}
nb=json.loads(NOTEBOOK.read_text()); cell=next(c for c in nb["cells"] if c.get("id")==CELL_ID)
cell_sha=sha_bytes(canon({"cell_id":CELL_ID,"source_text":"".join(cell["source"])}).encode())

stage=Path(tempfile.mkdtemp(prefix=".v2_stage_",dir=OUT))
try:
    put_json(stage/"proposal_canonical.json",proposal); put_json(stage/"evaluation_queries.json",queries)
    put_jsonl(stage/"master_gold.jsonl",master); put_jsonl(stage/"source_components.jsonl",sources); put_json(stage/"source_refs.json",source_refs)
    put_json(stage/"atomic_claims.json",claim_doc); put_json(stage/"answer_policies.json",policy_doc); put_jsonl(stage/"qualification_by_card.jsonl",qual)
    put_jsonl(stage/"annotation_audit.jsonl",audits); put_jsonl(stage/"candidate_projection.jsonl",projections); put_jsonl(stage/"query_duplicate_audit.jsonl",dups)
    put_json(stage/"draft_scoring_contract.json",scoring); put_jsonl(stage/"draft_review_packet.jsonl",reviews)
    (stage/"README.md").write_text(f"""# Integrated holdout master v2

상태: {STATUS}

proposal.part01~08을 canonical serializer로 병합해 restricted core를 재생성한 단일 분석가 초안입니다. 독립 검토 전에는 seal·검색·모델 평가·운영 판정에 사용할 수 없습니다.

- proposal canonical SHA-256: {proposal_sha}
- source components 72, atomic claims 47, records 30
- qualification rows 300, override rows 49
- claim projection 94행, negative N/A 8행
- 실제 query inventory 4개 중복 감사: exact 0, token-Jaccard 0.8 이상 0
- OLD payload eligibility는 section/benefit만
- STRUCT actual payload evidence는 미래 label-free bundle freeze 전까지 미확정
- API/network/GPU/embedding/retrieval/LLM 0

PART8의 여분 닫는 중괄호 제거와 overrides 48→49 교정 이력을 provenance에 기록했습니다. 기존 lineage/demo는 실행 전후 byte-exact입니다.

한계: single-analyst proposal이고 reviewer_proposal은 모두 null입니다. projection은 interval 기반 eligibility 진단이며 실제 ranking/payload 결과가 아닙니다.
""",encoding="utf-8")
    artifacts=sorted(n for n in restricted if n not in {"draft_dataset_manifest.json","draft_integrity.json"})
    manifest={"schema_version":VERSION,"generation_id":generation,"status":STATUS,"proposal_canonical_sha256":proposal_sha,"proposal_part_raw_sha256":part_hashes,"revision_cell_source_sha256":cell_sha,"replaced_restricted_v1_raw_sha256":replaced_v1_hashes,"preserved_lineage_demo_raw_sha256":preserved_before,"inventory_raw_sha256":inventory_hashes,"artifacts":{n:sha_file(stage/n) for n in artifacts},"manifest_self_hash_excluded":True,"integrity_self_hash_excluded":True,"atomic_replacement":"stage_complete_then_os.replace_manifest_and_integrity_last","execution":{"environment":"skn25","cpu_offline":True,"api_requests":0,"network":0,"gpu":0,"embedding":0,"retrieval":0,"llm":0},"part8_correction_history":proposal["provenance"]["part8_correction_history"]}
    put_json(stage/"draft_dataset_manifest.json",manifest)
    integrity={"schema_version":VERSION,"generation_id":generation,"status":"PASS_PENDING_INDEPENDENT_REVIEW","dataset_status":STATUS,"manifest_sha256":sha_file(stage/"draft_dataset_manifest.json"),"proposal_canonical_sha256":proposal_sha,"counts":{"source_components":72,"atomic_claims":47,"records":30,"qualification_rows":300,"qualification_overrides":49,"claim_projection_rows":94,"negative_projection_na_rows":8,"annotation_rows":30,"review_packet_rows":30,"duplicate_audit_rows":30},"assertions":{"source_locator_exact":True,"claim_and_graph_refs_resolve":True,"qualification_sets_exact":True,"reviewer_proposal_null":True,"old_payload_section_benefit_only":True,"negative_projection_na":True,"four_actual_inventory_audit":True,"executable_scoring_contract_pass":True,"lineage_demo_preserved":True,"api_network_gpu_embedding_retrieval_llm_zero":True},"artifact_hashes":manifest["artifacts"]}
    put_json(stage/"draft_integrity.json",integrity)
    need(all(sha_file(OUT/n)==h for n,h in preserved_before.items()),"preserved changed before replacement")
    order=[n for n in restricted if n not in {"draft_dataset_manifest.json","draft_integrity.json"}]+["draft_dataset_manifest.json","draft_integrity.json"]
    need(set(order)==set(restricted),"replacement set")
    for n in order: os.replace(stage/n,OUT/n)
finally: shutil.rmtree(stage,ignore_errors=True)

need(all(sha_file(OUT/n)==h for n,h in preserved_before.items()),"preserved changed after replacement")
saved=json.loads((OUT/"draft_dataset_manifest.json").read_text()); integ=json.loads((OUT/"draft_integrity.json").read_text())
need(saved["generation_id"]==generation==integ["generation_id"],"generation mismatch")
need(all(sha_file(OUT/n)==h for n,h in saved["artifacts"].items()),"artifact hash")
for n in restricted:
    if n.endswith(".jsonl"):
        rows=[json.loads(x) for x in (OUT/n).read_text().splitlines() if x]; need(rows and all(r.get("generation_id")==generation for r in rows),f"mixed {n}")
    elif n.endswith(".json") and n!="proposal_canonical.json":
        need(json.loads((OUT/n).read_text()).get("generation_id")==generation,f"mixed {n}")
need(not any(b"integrated_holdout_master_v1" in (OUT/n).read_bytes() for n in restricted),"v1 restricted content remains")
need(all(sha_file(ROOT/p)==h for p,h in inventory_hashes.items()),"inventory changed")
print(canon({"status":"PASS_PENDING_INDEPENDENT_REVIEW","proposal_canonical_sha256":proposal_sha,"generation_id":generation,"source_components":72,"atomic_claims":47,"records":30,"qualification_rows":300,"qualification_overrides":49,"candidate_projection_rows":102,"duplicate_exact_matches":exact,"duplicate_token_jaccard_matches":near,"lineage_demo_preserved":True,"api_network_gpu_embedding_retrieval_llm":0}))


{"api_network_gpu_embedding_retrieval_llm":0,"atomic_claims":47,"candidate_projection_rows":102,"duplicate_exact_matches":0,"duplicate_token_jaccard_matches":0,"generation_id":"integrated_holdout_master_v2_04a027273db0a511","lineage_demo_preserved":true,"proposal_canonical_sha256":"04a027273db0a51124f89bdb49f902089f50a6c468b18b71bc735b9cb3c00e4b","qualification_overrides":49,"qualification_rows":300,"records":30,"source_components":72,"status":"PASS_PENDING_INDEPENDENT_REVIEW"}


## v3 restricted core — reviewer blocker/major corrections

Codex coder agent가 raw TXT와 구조 청킹의 실제 heading lineage를 다시 검산해 생성합니다. v2 draft는 폐기 provenance로만 남고, 이 셀의 산출물은 `pending_independent_review` 상태이므로 candidate seal 전 검색·답변 생성에 사용할 수 없습니다. API/network/GPU/embedding/retrieval/LLM 실행은 0입니다.

In [1]:
# Integrated holdout master v3: CPU/offline restricted-core regeneration.
from __future__ import annotations
import copy, csv, hashlib, json, os, re, shutil, tempfile, unicodedata
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / 'notebooks').is_dir(): ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / 'notebooks').is_dir(): ROOT = cwd.parent
else: raise RuntimeError('Run from repository root or notebooks/')
NOTEBOOK = ROOT / 'notebooks/27_integrated_holdout_dataset.ipynb'
OUT = ROOT / 'notebooks/data/27_integrated_holdout_dataset'
STATUS, VERSION, CELL_ID = 'pending_independent_review', 'integrated_holdout_master_v3', '27-v3-restricted-core'
def need(ok, msg):
    if not ok: raise RuntimeError(msg)
def canon(v): return json.dumps(v, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
def digest(b): return hashlib.sha256(b).hexdigest()
def sha(p): return digest(p.read_bytes())
def put_json(p, v): p.write_text(canon(v), encoding='utf-8')
def put_jsonl(p, rows): p.write_text(''.join(canon(r)+'\n' for r in rows), encoding='utf-8')

parts_p = [OUT/f'proposal.part{i:02d}.json' for i in range(1,9)]
need(all(p.is_file() for p in parts_p), 'proposal parts 1-8 required')
parts = [json.loads(p.read_text()) for p in parts_p]
part_hashes = {p.name:sha(p) for p in parts_p}
need(parts[7]['counts']['overrides']==49, 'PART8 final override count must be 49')
preserved = ['demo_master_claim.json','demo_projection.json','demo_public_view.json','integrity.json','lineage_contract.json','lineage_quality_audit.json','old_chunk_lineage.jsonl','raw_line_index.jsonl','run_manifest.json','source_registry.json','struct_chunk_lineage.jsonl']
preserved_before = {n:sha(OUT/n) for n in preserved}
v2_files = ['README.md','annotation_audit.jsonl','answer_policies.json','atomic_claims.json','candidate_projection.jsonl','draft_dataset_manifest.json','draft_integrity.json','draft_review_packet.jsonl','draft_scoring_contract.json','evaluation_queries.json','master_gold.jsonl','proposal_canonical.json','qualification_by_card.jsonl','query_duplicate_audit.jsonl','source_components.jsonl','source_refs.json']
current_restricted_hashes = {n:sha(OUT/n) for n in v2_files}
current_manifest = json.loads((OUT/'draft_dataset_manifest.json').read_text())
need(current_manifest['schema_version'] in {'integrated_holdout_master_v2',VERSION}, 'v2/v3 restricted source expected')
v2_hashes = current_restricted_hashes if current_manifest['schema_version']=='integrated_holdout_master_v2' else current_manifest['v2_restricted_raw_sha256']
inputs = [ROOT/'notebooks/data/22_structural_heading_chunking_ablation/chunks.jsonl', ROOT/'notebooks/data/22_structural_heading_chunking_ablation/hierarchy_manifest.jsonl']
input_before = {str(p.relative_to(ROOT)):sha(p) for p in inputs}

base=parts[0]; sf,cf,clf,rf,qf=(base['schemas'][k] for k in ('source_component','condition','claim','record','qualification_row'))
registry_list=json.loads((OUT/'source_registry.json').read_text()); registry={r['card_key']:r for r in registry_list}
line_rows=[json.loads(x) for x in (OUT/'raw_line_index.jsonl').read_text().splitlines() if x]
line_by={(r['card_key'],r['line_number']):r for r in line_rows}
card_id_by_key={'BC/BC_Biz_AirMoney':'BC','NH/NH_Namu_NH':'NH','hana/Hana_One_More_SOHO':'HANA','hyundai/Hyundai_The_Orange_20260330':'HYUNDAI','ibk/IBK_Point3.8(Credit)':'IBK','kookmin/Kookmin_Friend_20210917':'KB','lotte/Lotte_LOCA_LIKIT_Eat':'LOTTE','samsung/Samsung_iD_ALL':'SAMSUNG','shinhan/Shinhan_Toss_Mr.Life_20251231':'SHINHAN','woori/Woori_Classic_EVERY_MILE_SKYPASS':'WOORI'}
raw_cache={}
def component_values(card_key, start, end):
    reg=registry[card_key]; rows=[line_by[(card_key,n)] for n in range(start,end+1)]
    need(all(r['source_sha256']==reg['raw_sha256'] for r in rows), f'line source {card_key}/{start}-{end}')
    raw=raw_cache.setdefault(card_key,(ROOT/reg['path']).read_bytes()); text=raw.decode('utf-8')
    bs,be,cs,ce=rows[0]['byte_start'],rows[-1]['byte_end'],rows[0]['cp_start'],rows[-1]['cp_end']
    need(text[cs:ce].encode()==raw[bs:be], f'line interval {card_key}/{start}-{end}')
    pages={r['page_num'] for r in rows}; need(len(pages)==1, 'component must remain on one page')
    return [card_id_by_key[card_key],card_key,reg['raw_sha256'],next(iter(pages)),start,end,cs,ce,bs,be,digest(raw[bs:be])]

source_values={}
for part in parts[:4]:
    vals=part['source_components']; items=vals.items() if isinstance(vals,dict) else ((r[0],r[1:]) for r in vals)
    source_values.update(items)
for sid in ('S019','S023','S065'): source_values.pop(sid)
replacements={'S003':('hana/Hana_One_More_SOHO',117,118),'S008':('lotte/Lotte_LOCA_LIKIT_Eat',41,41),'S026':('shinhan/Shinhan_Toss_Mr.Life_20251231',203,203),'S042':('shinhan/Shinhan_Toss_Mr.Life_20251231',179,179),'S050':('samsung/Samsung_iD_ALL',32,32),'S059':('NH/NH_Namu_NH',104,104),'S060':('NH/NH_Namu_NH',110,115)}
additions={'S073':('lotte/Lotte_LOCA_LIKIT_Eat',42,42),'S074':('shinhan/Shinhan_Toss_Mr.Life_20251231',204,204),'S075':('shinhan/Shinhan_Toss_Mr.Life_20251231',207,207),'S076':('samsung/Samsung_iD_ALL',33,33),'S077':('NH/NH_Namu_NH',105,105),'S078':('NH/NH_Namu_NH',120,121),'S079':('BC/BC_Biz_AirMoney',106,106),'S080':('hyundai/Hyundai_The_Orange_20260330',407,408),'S081':('NH/NH_Namu_NH',98,102)}
for sid,spec in {**replacements,**additions}.items(): source_values[sid]=component_values(*spec)
need(len(source_values)==78 and 'S023' not in source_values and 'S065' not in source_values, 'v3 source count/removal')

claims_raw={cid:copy.deepcopy(v) for part in parts[4:6] for cid,v in part['claims'].items()}
def set_refs(cid, refs): claims_raw[cid][8]=refs
set_refs('C03',['S003']); set_refs('C07',['S008','S073']); set_refs('C16',['S018'])
set_refs('C19',['S020','S024','S025']); set_refs('C20',['S026','S074','S075'])
set_refs('C28',['S040','S041','S042']); set_refs('C29',['S040','S041'])
set_refs('C33',['S009','S049','S050','S076']); set_refs('C36',['S055','S008','S073'])
set_refs('C41',['S064','S008','S073']); set_refs('C45',['S066','S067','S079'])
set_refs('C37',['S056','S057','S058','S081','S059','S077','S060','S017','S078'])
set_refs('C42',['S056','S057','S060','S077','S078']); set_refs('C43',['S056','S058','S060','S077','S078'])
# Typed semantic corrections are based on the cited raw lines, not inferred metadata.
claims_raw['C22'][4]=[['voucher_choice','any_of',['shopping','hotel','travel'],None,True],['alternative_mpoint_claim_id','eq','C49',None,False],['first_year_cumulative_spend','gte',1000000,'KRW',True],['later_year_previous_spend','gte',12000000,'KRW',True]]
claims_raw['C24'][4][1]=['eligible_vehicle_scope','eq','cardholder_riding_vehicle',None,True]
claims_raw['C30'][4][1]=['billing_route','not_in',['apartment_management_fee','officetel_management_fee'],None,True]
claims_raw['C37'][4]=[['account_status','eq','opened_and_designated',None,True],['prior_month_spend','gte',400000,'KRW',True],['monthly_cap_by_spend_band','eq',{'400k_to_under_800k':{'rank1':8000,'rank2':7000,'total':15000},'800k_to_under_1500k':{'rank1':11000,'rank2':9000,'total':20000},'1500k_or_more':{'rank1':16000,'rank2':14000,'total':30000}},'KRW_per_month',True],['merchant_context_exclusions','exists',True,None,False],['transaction_exclusions','exists',True,None,False]]
claims_raw['C48']=['HYUNDAI','family_card_mpoint_transfer',True,'boolean',[['transfer_channel','eq','hyundai_card_website',None,True],['recipient_relationship','eq','family_member',None,True]],'critical_eligibility','qualification','conditional',['S080'],'CV_BOOL_EXACT']
claims_raw['C49']=['HYUNDAI','annual_voucher_mpoint_exchange_alternative',200000,'Mpoint_per_year',[['choice_scope','eq','alternative_to_150000_KRW_voucher',None,True]],'critical_numeric','qualification','direct',['S030','S031'],'CV_COUNT_CANONICAL']
need(len(claims_raw)==49, '49 corrected claims')

records_raw=copy.deepcopy(parts[6]['records'])
records_raw['O08'][4]=['C39','C40','C48']; records_raw['O08'][7]=['family_registered_member_point_transfer','deceased_member_mpoint_inheritance','family_card_mpoint_transfer']
records_raw['O08'][8]['HYUNDAI']={'any_of':[['C40','eq',True],['C48','eq',True]]}
records_raw['A05'][0]='배달앱 업종 전용 할인·적립·캐시백과 영상·음악 정기구독 또는 유료 멤버십 결제의 별도 할인·적립·캐시백을 모두 제공하는 카드를 추천해줘.'
records_raw['A05'][7]=['dedicated_delivery_app_discount_or_accrual_or_cashback','subscription_or_paid_membership_dedicated_discount_or_accrual_or_cashback']
record_ids=[*(f'D{i:02d}' for i in range(1,11)),*(f'O{i:02d}' for i in range(1,11)),*(f'A{i:02d}' for i in range(1,11))]

sources=[]
for sid in sorted(source_values):
    vals=source_values[sid]; need(len(vals)==len(sf), f'source width {sid}'); row={'source_component_id':sid,**dict(zip(sf,vals))}
    reg=registry[row['card_key']]; raw=raw_cache.setdefault(row['card_key'],(ROOT/reg['path']).read_bytes()); text=raw.decode('utf-8')
    need(sha(ROOT/reg['path'])==row['source_sha256']==reg['raw_sha256'], f'source hash {sid}')
    excerpt=raw[row['byte_start']:row['byte_end']]; need(text[row['cp_start']:row['cp_end']].encode()==excerpt and digest(excerpt)==row['excerpt_sha256'], f'locator {sid}')
    sources.append({**row,'source_path':reg['path'],'status':STATUS,'reviewer_proposal':None})
by_source={r['source_component_id']:r for r in sources}; card_keys={v:k for k,v in card_id_by_key.items()}
claims=[]
for cid in sorted(claims_raw):
    vals=claims_raw[cid]; need(len(vals)==len(clf), f'claim width {cid}'); row=dict(zip(clf,vals)); row['conditions']=[dict(zip(cf,x)) for x in row['conditions']]
    need(row['required_component_ids'] and all(s in by_source for s in row['required_component_ids']), f'claim source {cid}')
    need(all(by_source[s]['card_id']==row['card_id'] for s in row['required_component_ids']), f'claim card {cid}')
    claims.append({'atomic_claim_id':cid,'card_key':card_keys[row['card_id']],**row,'status':STATUS,'reviewer_proposal':None})
by_claim={r['atomic_claim_id']:r for r in claims}
records=[]
for rid in record_ids:
    vals=records_raw[rid]; need(len(vals)==len(rf), f'record width {rid}'); row=dict(zip(rf,vals))
    need(all(c in by_claim for c in row['atomic_claim_ids']), f'record claims {rid}')
    for card,g in row['qualification_graphs'].items():
        need(card in row['expected_card_ids'], f'graph card {rid}')
        for branch in ('all_of','any_of'):
            for leaf in g.get(branch,[]): need(leaf[0] in by_claim and by_claim[leaf[0]]['card_id']==card, f'graph leaf {rid}')
    records.append({'record_id':rid,**row,'status':STATUS,'reviewer_proposal':None})
by_record={r['record_id']:r for r in records}
by_record['A05']['logical_form']='dedicated_delivery_app_benefit AND (video_music_subscription_benefit OR paid_membership_benefit)'

# Materialize and enrich all 300 qualification rows; false rows carry query-specific review evidence.
old_qual=[json.loads(x) for x in (OUT/'qualification_by_card.jsonl').read_text().splitlines() if x]
old_q={(r['record_id'],r['card_id']):r for r in old_qual}
negative_synonyms={'O09':{'dedicated_cloud_server_fee_benefit':['AWS','Azure','클라우드 서버 요금'],'dedicated_shared_office_fee_benefit':['공유오피스','공유 오피스 이용료']},'O10':{'dedicated_bicycle_share_benefit':['따릉이','공유자전거'],'dedicated_kick_scooter_benefit':['전동킥보드','킥보드']},'A09':{'dedicated_theme_park_benefit':['놀이공원','에버랜드','캐리비안베이'],'dedicated_hospital_pharmacy_benefit':['병원','약국']},'A10':{'dedicated_ev_charging_discount':['전기차 충전','EV 충전'],'airport_or_luxury_hotel_free_valet':['공항 발레파킹','특급호텔 발레파킹']}}
all_source_reviews=[{'card_id':card_id_by_key[r['card_key']],'card_key':r['card_key'],'source_path':r['path'],'source_sha256':r['raw_sha256']} for r in registry_list]
qual=[]
for rid in record_ids:
    rec=by_record[rid]
    for card in base['corpus_card_ids']:
        src=registry[card_keys[card]]; prior=copy.deepcopy(old_q[(rid,card)])
        prior.pop('generation_id',None); prior.pop('schema_version',None)
        if rid=='O08':
            if card=='HYUNDAI': prior['present_claim_ids']=sorted(set(prior['present_claim_ids']+['C48']))
            elif not prior['qualified']: prior['missing_claim_ids']=['C39','C40','C48']
        prior.update({'record_id':rid,'card_id':card,'card_key':card_keys[card],'status':STATUS,'reviewer_proposal':None,'inspected_source_path':src['path'],'inspected_source_sha256':src['raw_sha256'],'query_specific_requested_predicate_ids':rec['requested_predicate_ids'],'query_specific_negative_predicates':[{'predicate_id':p,'outcome':'unsupported_for_card'} for p in rec['requested_predicate_ids']] if not prior['qualified'] else []})
        if not prior['qualified']:
            prior['negative_review']={'searched_predicate_synonyms':negative_synonyms.get(rid,{p:[p.replace('_',' ')] for p in rec['requested_predicate_ids']}),'general_base_benefit_excluded':rid in {'O09','O10','A09'},'all_ten_card_sources_reviewed':rid in negative_synonyms,'corpus_source_reviews':all_source_reviews if rid in negative_synonyms else [{'card_id':card,'card_key':card_keys[card],'source_path':src['path'],'source_sha256':src['raw_sha256']}]}
        qual.append(prior)
need(len(qual)==300 and len({(q['record_id'],q['card_id']) for q in qual})==300, 'qualification 300')
for rid in record_ids: need({q['card_id'] for q in qual if q['record_id']==rid and q['qualified']}==set(by_record[rid]['expected_card_ids']), f'qualified set {rid}')
need(all(q['reviewer_proposal'] is None and q['full_source_reviewed'] is True for q in qual), 'review state')
need(all(q.get('query_specific_requested_predicate_ids') and q.get('inspected_source_sha256') for q in qual if not q['qualified']), 'false-row audit completeness')

# Exact inherited-heading intervals from the frozen structural hierarchy.
hier=[json.loads(x) for x in inputs[1].read_text().splitlines() if x]; chunks=[json.loads(x) for x in inputs[0].read_text().splitlines() if x]
nodes={r['node_id']:r for r in hier}; chunk_node={r['chunk_id']:r['metadata']['node_id'] for r in chunks}
title_rows=[]; title_by_chunk={}
for chunk_id,node_id in chunk_node.items():
    inherited=[]; cur=nodes[node_id]
    while cur is not None:
        if cur['heading_line_number'] is not None:
            lr=line_by[(cur['card_key'],cur['heading_line_number'])]
            inherited.append({'heading_node_id':cur['node_id'],'heading_line_number':cur['heading_line_number'],'cp_start':lr['cp_start'],'cp_end':lr['cp_end'],'byte_start':lr['byte_start'],'byte_end':lr['byte_end'],'source_sha256':lr['source_sha256']})
        cur=nodes.get(cur['parent_id'])
    inherited.sort(key=lambda x:x['heading_line_number']); title_by_chunk[chunk_id]=inherited
    for x in inherited: title_rows.append({'schema_version':VERSION,'chunk_id':chunk_id,'card_key':next(c['metadata']['card_key'] for c in chunks if c['chunk_id']==chunk_id),'interval_role':'inherited_searchable_heading_title',**x,'fuzzy_matching_used':False,'status':STATUS})
old=[json.loads(x) for x in (OUT/'old_chunk_lineage.jsonl').read_text().splitlines() if x]; struct=[json.loads(x) for x in (OUT/'struct_chunk_lineage.jsonl').read_text().splitlines() if x]
need(len(old)==327 and len(struct)==147 and len(chunk_node)==147, 'lineage counts')
def overlap(a,b): return a['cp_start']<b['cp_end'] and b['cp_start']<a['cp_end']
def body_hits(comp,lineage,levels=None):
    return sorted({r['chunk_id'] for r in lineage if r['status']=='exact' and r['source_sha256']==comp['source_sha256'] and r['card_key']==comp['card_key'] and (levels is None or r.get('level') in levels) and any(overlap(i,comp) for i in r['component_intervals'])})
def title_hits(comp):
    return sorted(cid for cid,ints in title_by_chunk.items() if chunks_by_id[cid]['metadata']['card_key']==comp['card_key'] and any(overlap(i,comp) and i['source_sha256']==comp['source_sha256'] for i in ints))
chunks_by_id={c['chunk_id']:c for c in chunks}
projections=[]
for claim in claims:
    for corpus,lineage in (('OLD',old),('STRUCT',struct)):
        direct={s:body_hits(by_source[s],lineage) for s in claim['required_component_ids']}
        titles={s:(title_hits(by_source[s]) if corpus=='STRUCT' else []) for s in claim['required_component_ids']}
        logical={s:sorted(set(direct[s]+titles[s])) for s in claim['required_component_ids']}
        eligible={s:(body_hits(by_source[s],lineage,{'section','benefit'}) if corpus=='OLD' else logical[s]) for s in claim['required_component_ids']}
        projections.append({'schema_version':VERSION,'projection_scope':'atomic_claim','atomic_claim_id':claim['atomic_claim_id'],'card_id':claim['card_id'],'corpus':corpus,'projection_method':'exact_half_open_source_interval_overlap_plus_struct_inherited_heading_title','fuzzy_matching_used':False,'raw_source_reproduction_complete':True,'full_corpus_projection':{'direct_body_component_chunk_ids':direct,'inherited_title_component_chunk_ids':titles,'logical_component_chunk_ids':logical,'full_corpus_logical_presence_complete':all(logical.values())},'searchable_eligible_projection':{'component_chunk_ids':eligible,'chunk_ids':sorted({x for v in eligible.values() for x in v}),'searchable_eligible_projection_complete':all(eligible.values()),'rule':'OLD_section_benefit_direct_body_only' if corpus=='OLD' else 'STRUCT_direct_body_or_exact_inherited_heading_title'},'payload_evidence_projection':{'payload_evidence_projection_complete':False,'status':'not_frozen_before_label_free_candidate_execution','eligible_component_chunk_ids':eligible},'candidate_projection_miss':not all(eligible.values()),'dataset_invalid':False,'status':STATUS})
for rid in ('O09','O10','A09','A10'):
    for corpus in ('OLD','STRUCT'): projections.append({'schema_version':VERSION,'projection_scope':'negative_record','record_id':rid,'corpus':corpus,'raw_source_reproduction_complete':'not_applicable_negative','full_corpus_projection':'not_applicable_negative','searchable_eligible_projection':'not_applicable_negative','payload_evidence_projection':'not_applicable_negative','dataset_invalid':False,'status':STATUS})
need(len(projections)==106, 'projection rows')
for cid in ('C18','C19','C22'):
    p=next(x for x in projections if x.get('atomic_claim_id')==cid and x['corpus']=='STRUCT'); need(p['searchable_eligible_projection']['searchable_eligible_projection_complete'], f'inherited title projection {cid}')

def op_eval(actual,op,expected):
    if op=='eq': return actual==expected
    if op=='gte': return isinstance(actual,(int,float)) and actual>=expected
    if op=='gt': return isinstance(actual,(int,float)) and actual>expected
    if op=='lte': return isinstance(actual,(int,float)) and actual<=expected
    if op=='neq': return actual!=expected
    if op=='in': return actual in expected
    if op=='not_in': return actual not in expected
    if op=='all_of': return isinstance(actual,(list,tuple,set)) and all(x in actual for x in expected)
    if op=='any_of': return isinstance(actual,(list,tuple,set)) and any(x in actual for x in expected)
    if op=='contains': return expected in actual
    if op=='exists': return (actual is not None)==bool(expected)
    if op=='object_has_any_key': return isinstance(actual,dict) and any(k in actual and actual[k] is not None for k in expected)
    raise RuntimeError('unsupported operator '+op)
operator_tests={'eq':op_eval(1,'eq',1),'gte':op_eval(2,'gte',1),'gt':op_eval(2,'gt',1),'lte':op_eval(1,'lte',2),'neq':op_eval(1,'neq',2),'in':op_eval('a','in',['a']),'not_in':op_eval('b','not_in',['a']),'all_of':op_eval(['a','b'],'all_of',['a','b']),'any_of':op_eval(['a'],'any_of',['a','b']),'contains':op_eval(['a'],'contains','a'),'exists':op_eval(0,'exists',True),'object_has_any_key':op_eval({'a':1},'object_has_any_key',['a'])}
need(all(operator_tests.values()), 'operator semantics')
for r in records:
    for card,g in r['qualification_graphs'].items():
        branch='all_of' if 'all_of' in g else 'any_of'; vals=[op_eval(by_claim[c]['canonical_value'],op,val) for c,op,val in g[branch]]
        need((all(vals) if branch=='all_of' else any(vals)), f'graph execution {r["record_id"]}/{card}')

selection={'status':'predeclared_accuracy_first_v3','absolute_eligibility':{'end_to_end_exact_success':{'minimum_count':21,'denominator':30},'positive_supported_card_recall_at_3':{'minimum':0.75,'denominator_records':26},'required_claim_coverage':{'minimum':0.80,'denominator_records':26},'negative_correctness':{'required_count':4,'denominator':4},'critical_unsupported_numeric_or_condition_errors':{'maximum':0},'wrong_card_catastrophic_errors':{'maximum':0},'citation_ownership':{'required':1.0,'zero_citation_positive_is_failure':True,'correct_negative_abstention':'not_applicable'}},'winner_rule':{'invalid':'invalid','exactly_one_candidate_absolute_eligible':'select_that_candidate','both_candidates_absolute_eligible':['end_to_end_exact_success absolute gap >= 3/30','paired wins > paired losses','supported_card_recall_at_3 nonregression','required_claim_coverage nonregression','critical_cohort_regression_count == 0'],'both_candidates_hard_or_absolute_fail':'holdout_failed','otherwise':'inconclusive'},'candidate_seal_required_before':['ranking','answer_generation'],'post_result_tuning_forbidden':True}
operator_contract={'eq':'normalized actual equals expected exactly','gte':'numeric actual >= expected after declared exact unit conversion','gt':'numeric actual > expected after declared exact unit conversion','lte':'numeric actual <= expected after declared exact unit conversion','neq':'normalized actual is not equal to expected','in':'scalar actual is a member of expected collection','not_in':'scalar actual is not a member of expected collection','all_of':'actual collection contains every expected member','any_of':'actual collection contains at least one expected member','contains':'actual collection or string contains expected','exists':'actual non-null existence equals expected boolean','object_has_any_key':'actual object has at least one requested key with non-null supported value'}
scoring={'schema_version':VERSION,'status':STATUS,'scoring_unit':'atomic_claim_id','operators':operator_contract,'operator_self_test':operator_tests,'metric_denominators':{'all_records':30,'positive_records':26,'negative_records':4,'returned_less_than_3_precision':'actual unique returned-card count capped at 3; positive no-output precision=0','no_output':'positive failure; correct negative abstention eligible','zero_citation':'positive failure; correct negative abstention N/A'},'paired_record_rule':{'win':'candidate metric > baseline by tolerance','loss':'candidate metric < baseline by tolerance','tie':'absolute delta <= tolerance','tolerance':1e-12,'denominator':'same frozen record IDs'},'cohort_critical_record_ids':{'direct':[f'D{i:02d}' for i in range(1,11)],'or_recommendation':[f'O{i:02d}' for i in range(1,11)],'and_recommendation':[f'A{i:02d}' for i in range(1,11)]},'selection_contract':selection,'one_candidate_hard_fail':'other eligible candidate may win only if absolute eligible','both_candidates_fail':'holdout_failed','execution_deferred':True}

def norm(x): return ' '.join(re.sub(r'[^\w가-힣]+',' ',unicodedata.normalize('NFKC',str(x)).lower()).split())
specs=[r for r in parts[7]['duplicate_audit']['inventories'] if r['path'].endswith('.csv')]; need(len(specs)==4,'four inventories')
inventory=[]; inventory_hashes={}
for spec in specs:
    p=ROOT/spec['path']; need(sha(p)==spec['sha256'], f'inventory {p}'); inventory_hashes[spec['path']]=spec['sha256']
    with p.open(encoding='utf-8',newline='') as f: inventory += [(spec['path'],r['query_id'],norm(r['query_text'])) for r in csv.DictReader(f)]
dups=[]
for r in records:
    n=norm(r['sealed_query_text']); ts=set(n.split()); exact=[]; near=[]; mx=0.0
    for path,qid,t in inventory:
        if n==t: exact.append({'inventory_path':path,'query_id':qid})
        os_=set(t.split()); j=len(ts&os_)/len(ts|os_) if ts|os_ else 1.0; mx=max(mx,j)
        if j>=0.8: near.append({'inventory_path':path,'query_id':qid,'token_jaccard':j})
    dups.append({'schema_version':VERSION,'record_id':r['record_id'],'query_normalized_sha256':digest(n.encode()),'exact_normalized_matches':exact,'token_jaccard_threshold':0.8,'token_jaccard_matches':near,'maximum_token_jaccard':mx,'manual_semantic_status':STATUS,'status':STATUS})
need(sum(len(x['exact_normalized_matches'])+len(x['token_jaccard_matches']) for x in dups)==0,'duplicate audit')

policies={'AP_DIRECT':{'task_family':'direct_single_card_fact','max_output_cards':1,'required':['named-card graph','all required conditions','owned source-component citation'],'abstain_when':'required graph unsupported'},'AP_REC_POS':{'task_family':'multi_card_recommendation','max_output_cards':3,'required':['same-card graph','owned source-component citation'],'abstain_when':'zero supported cards'},'AP_ABSTAIN':{'task_family':'negative','max_output_cards':0,'required':['closed-corpus insufficiency'],'abstain_when':'always'}}
queries={'schema_version':VERSION,'status':STATUS,'records':[{'record_id':r['record_id'],'sealed_query_text':r['sealed_query_text'],'task_family':r['task_family'],'axes':r['axes'],'status':STATUS} for r in records]}
master=[]; audits=[]; reviews=[]
q_by={rid:[q for q in qual if q['record_id']==rid] for rid in record_ids}
for r in records:
    sids=sorted({s for c in r['atomic_claim_ids'] for s in by_claim[c]['required_component_ids']})
    neg=r['record_id'] in negative_synonyms
    master.append({'schema_version':VERSION,'record_id':r['record_id'],'task_family':r['task_family'],'logical_form':r.get('logical_form'),'expected_card_ids':r['expected_card_ids'],'atomic_claim_ids':r['atomic_claim_ids'],'required_source_component_ids':sids,'requested_predicate_ids':r['requested_predicate_ids'],'qualification_graphs':r['qualification_graphs'],'answer_policy_ref':r['answer_policy_ref'],'negative_type':r['negative_type'],'negative_corpus_audit':{'searched_predicate_synonyms':negative_synonyms[r['record_id']],'general_base_benefit_excluded':r['record_id'] in {'O09','O10','A09'},'ten_source_sha256':[x['source_sha256'] for x in all_source_reviews]} if neg else None,'scoring_unit':'atomic_claim_id','status':STATUS,'reviewer_proposal':None})
    audits.append({'schema_version':VERSION,'record_id':r['record_id'],'source_components_reproduced':len(sids),'qualification_rows_reviewed':10,'full_source_reviewed':all(q['full_source_reviewed'] for q in q_by[r['record_id']]),'reviewer_proposal':None,'status':STATUS,'limitation':'single-analyst correction pending independent review'})
    reviews.append({'schema_version':VERSION,'record_id':r['record_id'],'sealed_query_text':r['sealed_query_text'],'task_family':r['task_family'],'expected_card_ids':r['expected_card_ids'],'atomic_claims':[by_claim[c] for c in r['atomic_claim_ids']],'source_components':[by_source[s] for s in sids],'qualification_rows':q_by[r['record_id']],'answer_policy_ref':r['answer_policy_ref'],'reviewer_proposal':None,'review_status':STATUS})

proposal={'schema_version':VERSION,'status':STATUS,'corpus_card_ids':base['corpus_card_ids'],'max_output_cards':3,'schemas':base['schemas'],'source_components':{r['source_component_id']:[r[k] for k in sf] for r in sources},'claims':claims_raw,'records':records_raw,'selection_contract':selection,'part8_final_correction_history':['removed one extra closing brace after denominators','corrected overrides count 48 to 49; retained all 49 override rows'],'known_limitations':['proposal parts remain immutable v2 analyst transfer; v3 applies raw-recomputed corrections in this revision cell','single analyst corrections await independent review'],'v2_draft_disposition':'discarded_not_sealable'}
proposal_no=copy.deepcopy(proposal); proposal_sha=digest(canon(proposal_no).encode()); proposal['canonical_sha256']=proposal_sha
generation=f'integrated_holdout_master_v3_{proposal_sha[:16]}'
for collection in (sources,claims,records,qual,projections,title_rows,dups,master,audits,reviews):
    for row in collection: row['generation_id']=generation
queries['generation_id']=generation
for row in queries['records']: row['generation_id']=generation
scoring['generation_id']=generation
source_refs={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'unit':'source_component_id','refs':sources}
claim_doc={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'claims':claims}
policy_doc={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'policies':policies}
nb=json.loads(NOTEBOOK.read_text()); cell=next(c for c in nb['cells'] if c.get('id')==CELL_ID); cell_source=''.join(cell['source']) if isinstance(cell['source'],list) else cell['source']; cell_sha=digest(canon({'cell_id':CELL_ID,'source_text':cell_source}).encode())
restricted=v2_files+['struct_searchable_title_lineage.jsonl']
stage=Path(tempfile.mkdtemp(prefix='.v3_stage_',dir=OUT))
try:
    put_json(stage/'proposal_canonical.json',proposal); put_json(stage/'evaluation_queries.json',queries); put_jsonl(stage/'master_gold.jsonl',master)
    put_jsonl(stage/'source_components.jsonl',sources); put_json(stage/'source_refs.json',source_refs); put_json(stage/'atomic_claims.json',claim_doc); put_json(stage/'answer_policies.json',policy_doc)
    put_jsonl(stage/'qualification_by_card.jsonl',qual); put_jsonl(stage/'annotation_audit.jsonl',audits); put_jsonl(stage/'candidate_projection.jsonl',projections); put_jsonl(stage/'struct_searchable_title_lineage.jsonl',title_rows); put_jsonl(stage/'query_duplicate_audit.jsonl',dups); put_json(stage/'draft_scoring_contract.json',scoring); put_jsonl(stage/'draft_review_packet.jsonl',reviews)
    query_sha=sha(stage/'evaluation_queries.json'); gold_sha=sha(stage/'master_gold.jsonl')
    (stage/'README.md').write_text(f'''# Integrated holdout master v3\n\n상태: `{STATUS}`. reviewer blocker/major를 raw TXT와 structural heading lineage에서 다시 계산한 restricted core입니다. v2 draft는 폐기되었고 candidate seal 전에는 ranking·answer 생성이 금지됩니다.\n\n- canonical SHA-256: `{proposal_sha}`\n- source components 78, claims 49, records 30, qualification rows 300\n- raw/full-corpus/searchable/payload completeness는 서로 다른 필드로 보고합니다. STRUCT searchable projection은 exact inherited heading title interval을 포함하며 fuzzy matching은 사용하지 않습니다.\n- 모든 false qualification row에 query-specific predicate, inspected source path/hash, present/missing/near-miss와 reviewer null을 저장했습니다. 네 negative record는 10개 source hash와 검색 동의어를 기록합니다.\n- accuracy-first 절대 하한은 scoring contract에 결과 전 고정했습니다.\n- API/network/GPU/embedding/retrieval/LLM: 0\n\nPART8 최종 교정은 여분 중괄호 1개 제거와 override count 49이며, 49개 override row는 모두 유지했습니다. v3 변경은 immutable proposal part를 덮어쓰지 않고 raw-recomputed revision으로 기록합니다. 이 산출물은 독립 검토 전 seal할 수 없습니다.\n''',encoding='utf-8')
    artifacts=sorted(n for n in restricted if n not in {'draft_dataset_manifest.json','draft_integrity.json'})
    manifest={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'proposal_canonical_sha256':proposal_sha,'evaluation_queries_raw_sha256':query_sha,'master_gold_raw_sha256':gold_sha,'new_generation_hashes':True,'candidate_seal_status':'forbidden_pending_independent_review','ranking_answer_forbidden_before_candidate_seal':True,'v2_draft_disposition':'discarded_not_sealable','v2_restricted_raw_sha256':v2_hashes,'proposal_part_raw_sha256':part_hashes,'revision_cell_source_sha256':cell_sha,'preserved_lineage_demo_raw_sha256':preserved_before,'structural_inputs_raw_sha256':input_before,'inventory_raw_sha256':inventory_hashes,'artifacts':{n:sha(stage/n) for n in artifacts},'manifest_self_hash_excluded':True,'integrity_self_hash_excluded':True,'atomic_replacement':'complete stage then os.replace; manifest/integrity last','execution':{'environment':'skn25','cpu_offline':True,'api_requests':0,'network':0,'gpu':0,'embedding':0,'retrieval':0,'llm':0},'part8_final_correction_history':proposal['part8_final_correction_history'],'known_limitations':proposal['known_limitations']}
    put_json(stage/'draft_dataset_manifest.json',manifest)
    misses=[{'atomic_claim_id':p['atomic_claim_id'],'corpus':p['corpus']} for p in projections if p.get('candidate_projection_miss')]
    integrity={'schema_version':VERSION,'generation_id':generation,'status':'PASS_PENDING_INDEPENDENT_REVIEW','dataset_status':STATUS,'manifest_sha256':sha(stage/'draft_dataset_manifest.json'),'proposal_canonical_sha256':proposal_sha,'counts':{'source_components':78,'atomic_claims':49,'records':30,'qualification_rows':300,'qualification_false_rows':sum(not q['qualified'] for q in qual),'claim_projection_rows':98,'negative_projection_na_rows':8,'projection_misses':len(misses),'title_lineage_rows':len(title_rows),'annotation_rows':30,'review_packet_rows':30},'projection_misses':misses,'assertions':{'source_locator_exact':True,'component_claim_refs_resolve':True,'O08_exhaustive_graph':True,'A05_semantics_clarified':True,'typed_semantics_corrected':True,'struct_inherited_heading_projection_C18_C19_C22':True,'all_false_rows_reproducible':True,'negative_four_all_ten_sources':True,'all_operators_executable':True,'accuracy_first_thresholds_predeclared':True,'reviewer_proposal_null':True,'lineage_demo_byte_exact':True,'v2_discarded':True,'candidate_seal_required':True,'api_network_gpu_embedding_retrieval_llm_zero':True},'artifact_hashes':manifest['artifacts']}
    put_json(stage/'draft_integrity.json',integrity)
    need(all(sha(OUT/n)==h for n,h in preserved_before.items()), 'preserved changed before replace')
    order=[n for n in restricted if n not in {'draft_dataset_manifest.json','draft_integrity.json'}]+['draft_dataset_manifest.json','draft_integrity.json']
    need(set(order)==set(restricted), 'replacement set')
    for n in order: os.replace(stage/n,OUT/n)
finally: shutil.rmtree(stage,ignore_errors=True)
need(all(sha(OUT/n)==h for n,h in preserved_before.items()), 'lineage/demo changed')
need(all(sha(ROOT/p)==h for p,h in input_before.items()), 'struct input changed')
saved=json.loads((OUT/'draft_dataset_manifest.json').read_text()); integ=json.loads((OUT/'draft_integrity.json').read_text())
need(saved['generation_id']==generation==integ['generation_id'], 'generation mismatch')
need(all(sha(OUT/n)==h for n,h in saved['artifacts'].items()), 'artifact hash mismatch')
need(sha(OUT/'evaluation_queries.json')==saved['evaluation_queries_raw_sha256'] and sha(OUT/'master_gold.jsonl')==saved['master_gold_raw_sha256'], 'query/gold hash')
need(not any(b'integrated_holdout_master_v2' in (OUT/n).read_bytes() for n in restricted if n not in {'draft_dataset_manifest.json'}), 'mixed v2 restricted core')
print(canon({'status':'PASS_PENDING_INDEPENDENT_REVIEW','proposal_canonical_sha256':proposal_sha,'generation_id':generation,'source_components':78,'atomic_claims':49,'records':30,'qualification_rows':300,'projection_rows':106,'projection_misses':len(integ['projection_misses']),'title_lineage_rows':len(title_rows),'candidate_seal_status':'forbidden_pending_independent_review','api_network_gpu_embedding_retrieval_llm':0}))


{"api_network_gpu_embedding_retrieval_llm":0,"atomic_claims":49,"candidate_seal_status":"forbidden_pending_independent_review","generation_id":"integrated_holdout_master_v3_0b3faec395f5caf4","projection_misses":0,"projection_rows":106,"proposal_canonical_sha256":"0b3faec395f5caf4e276e9079ab3a6ea2abb236d77aa44c6614cb6c9de967f97","qualification_rows":300,"records":30,"source_components":78,"status":"PASS_PENDING_INDEPENDENT_REVIEW","title_lineage_rows":205}


## v4 restricted core — remaining reviewer blocker/major corrections

v3의 raw locator와 exact inherited-title lineage를 동결한 채 claim semantics, qualification predicate outcomes, executable scoring contract를 보정합니다. v3는 폐기 provenance로만 남고 v4도 독립 검토 전 `pending_independent_review`이며 seal·ranking·answer 실행은 금지됩니다.

In [1]:
# Integrated holdout master v4: CPU/offline semantic and scoring correction.
from __future__ import annotations
import copy, csv, hashlib, json, os, re, shutil, tempfile, unicodedata
from pathlib import Path
cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError('Run from repository root or notebooks/')
NOTEBOOK=ROOT/'notebooks/27_integrated_holdout_dataset.ipynb'; OUT=ROOT/'notebooks/data/27_integrated_holdout_dataset'
STATUS,VERSION,CELL_ID='pending_independent_review','integrated_holdout_master_v4','27-v4-restricted-core'
def need(ok,msg):
    if not ok: raise RuntimeError(msg)
def canon(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def digest(b): return hashlib.sha256(b).hexdigest()
def sha(p): return digest(p.read_bytes())
def put_json(p,v): p.write_text(canon(v),encoding='utf-8')
def put_jsonl(p,rows): p.write_text(''.join(canon(r)+'\n' for r in rows),encoding='utf-8')

parts_p=[OUT/f'proposal.part{i:02d}.json' for i in range(1,9)]; parts=[json.loads(p.read_text()) for p in parts_p]
need(parts[7]['counts']['overrides']==49,'PART8 final override count')
part_hashes={p.name:sha(p) for p in parts_p}; conversions=copy.deepcopy(parts[0]['conversions'])
preserved=['demo_master_claim.json','demo_projection.json','demo_public_view.json','integrity.json','lineage_contract.json','lineage_quality_audit.json','old_chunk_lineage.jsonl','raw_line_index.jsonl','run_manifest.json','source_registry.json','struct_chunk_lineage.jsonl']
preserved_before={n:sha(OUT/n) for n in preserved}
title_name='struct_searchable_title_lineage.jsonl'; title_hash_before=sha(OUT/title_name); title_bytes=(OUT/title_name).read_bytes()
v3_manifest=json.loads((OUT/'draft_dataset_manifest.json').read_text()); need(v3_manifest['schema_version'] in {'integrated_holdout_master_v3',VERSION},'v3/v4 source required')
v3_provenance=v3_manifest if v3_manifest['schema_version']=='integrated_holdout_master_v3' else v3_manifest['v3_discarded_provenance']
source_before=sha(OUT/'source_components.jsonl') if v3_manifest['schema_version']=='integrated_holdout_master_v3' else v3_manifest['v3_source_components_raw_sha256']
projection_before=sha(OUT/'candidate_projection.jsonl') if v3_manifest['schema_version']=='integrated_holdout_master_v3' else v3_manifest['v3_candidate_projection_raw_sha256']
sources=[json.loads(x) for x in (OUT/'source_components.jsonl').read_text().splitlines() if x]
claims=copy.deepcopy(json.loads((OUT/'atomic_claims.json').read_text())['claims']); by_claim={r['atomic_claim_id']:r for r in claims}
queries=copy.deepcopy(json.loads((OUT/'evaluation_queries.json').read_text())); master=[json.loads(x) for x in (OUT/'master_gold.jsonl').read_text().splitlines() if x]
qual=[json.loads(x) for x in (OUT/'qualification_by_card.jsonl').read_text().splitlines() if x]
old_proj=[json.loads(x) for x in (OUT/'candidate_projection.jsonl').read_text().splitlines() if x]
need(len(sources)==78 and len(claims)==49 and len(master)==30 and len(qual)==300,'v3 input counts')

def condition(field,op,value,unit,required=True): return {'field':field,'operator':op,'value':value,'unit':unit,'required_for_complete_answer':required}
# Exact reviewer semantic corrections.
for cid in ('C28','C29'):
    c=by_claim[cid]; c['conditions']=[x for x in c['conditions'] if x['field'] not in {'minimum_approval_amount','discountable_approval_amount_cap','maximum_discount_per_use'}]
    c['conditions'] += [condition('discountable_approval_amount_cap','eq',10000,'KRW'),condition('maximum_discount_per_use','eq',1000,'KRW')]
by_claim['C15']['required_component_ids']=sorted(set(by_claim['C15']['required_component_ids']+['S078']))
by_claim['C16']['required_component_ids']=['S017','S018','S078']
by_claim['C16']['conditions']=[condition('destination_account','eq','designated_namu_securities_demand_deposit_account',None),condition('account_status','eq','opened_and_designated',None),condition('prior_month_spend','gte',400000,'KRW')]
by_claim['C03']['conditions'][0]['operator']='all_of'
next(x for x in by_claim['C38']['conditions'] if x['field']=='excluded_period')['operator']='all_of'
next(x for x in by_claim['C41']['conditions'] if x['field']=='excluded_payment_route')['operator']='all_of'
by_claim['C42']['required_component_ids']=['S056','S057','S078']; by_claim['C42']['related_component_ids']=['S060','S077']
by_claim['C43']['required_component_ids']=['S056','S058','S078']; by_claim['C43']['related_component_ids']=['S060','S077']
by_claim['C22']['related_claim_ids']=['C49']; by_claim['C49']['related_claim_ids']=['C22']
by_claim['C35']['evaluation_status']='retired_non_evaluation'; by_claim['C35']['retirement_reason']='unused_duplicate_restaurant_claim_not_referenced_by_any_record'
for c in claims: c.setdefault('evaluation_status','active_evaluation')
active_claims=[c for c in claims if c['evaluation_status']=='active_evaluation']; need(len(active_claims)==48,'48 active claims plus retired C35')
need(not any('C35' in r['atomic_claim_ids'] for r in master),'retired claim referenced')

a05='배달앱 업종 전용 할인·적립·캐시백을 제공하면서, 동시에 영상·음악 정기구독 또는 유료 멤버십 결제 중 적어도 한 영역에도 별도 할인·적립·캐시백을 제공하는 카드를 추천해줘.'
next(r for r in queries['records'] if r['record_id']=='A05')['sealed_query_text']=a05
m_by={r['record_id']:r for r in master}; m_by['A05']['logical_form']='dedicated_delivery_app_benefit AND at_least_one_of(video_music_subscription_benefit,paid_membership_benefit)'
m_by['O02']['logical_form']='eligible_smart_cashback_destination OR eligible_smart_cashback_cashability; both branches require designated account and prior-month spend >= 400000 KRW'
for r in master:
    r['required_source_component_ids']=sorted({s for cid in r['atomic_claim_ids'] for s in by_claim[cid]['required_component_ids']})

# Per-requested-predicate audit outcomes derive from actual present claims.
predicate_claims={}
for c in active_claims: predicate_claims.setdefault(c['predicate'],set()).add(c['atomic_claim_id'])
aliases={
 'dedicated_delivery_app_discount_or_accrual_or_cashback':{'C41','C42'},
 'subscription_or_paid_membership_dedicated_discount_or_accrual_or_cashback':{'C43','C36'},
 'dedicated_theme_park_benefit':{'C17'},'dedicated_hospital_pharmacy_benefit':{'C28'},
 'dedicated_ev_charging_discount':{'C30','C34'},'airport_or_luxury_hotel_free_valet':{'C24'}
}
near_aliases={'airport_lounge_companion_included':{'C45','C46','C47'},'major_category_interest_free_installment':{'C44'}}
q_by={(q['record_id'],q['card_id']):q for q in qual}
for row in qual:
    rec=m_by[row['record_id']]; present=set(row['present_claim_ids']); outcomes=[]
    for predicate in rec['requested_predicate_ids']:
        target=set(predicate_claims.get(predicate,set()))|set(aliases.get(predicate,set())); near=set(near_aliases.get(predicate,set()))
        satisfying=sorted(present&target); mismatching=sorted(present&near)
        outcome='supported_and_satisfies' if satisfying else ('supported_but_value_or_condition_mismatch' if mismatching else 'unsupported')
        outcomes.append({'predicate_id':predicate,'outcome':outcome,'satisfying_claim_ids':satisfying,'mismatching_claim_ids':mismatching,'present_claim_ids_considered':sorted(present),'missing_claim_ids_considered':row['missing_claim_ids']})
    row['requested_predicate_outcomes']=outcomes; row.pop('query_specific_negative_predicates',None)
    row['reviewer_proposal']=None; row['status']=STATUS
allowed_outcomes={'supported_and_satisfies','supported_but_value_or_condition_mismatch','unsupported'}
need(all({x['outcome'] for x in q['requested_predicate_outcomes']}<=allowed_outcomes for q in qual),'outcome vocabulary')
need(all((x['outcome']=='supported_and_satisfies')==bool(x['satisfying_claim_ids']) and (x['outcome']=='supported_but_value_or_condition_mismatch')==bool(x['mismatching_claim_ids']) for q in qual for x in q['requested_predicate_outcomes']),'outcome evidence consistency')
def outcome(rid,card,predicate): return next(x['outcome'] for x in q_by[(rid,card)]['requested_predicate_outcomes'] if x['predicate_id']==predicate)
named_expectations={('A02','SAMSUNG','ev_charging_discount_rate'):'supported_and_satisfies',('A02','SAMSUNG','four_social_insurance_autopay_discount_rate'):'unsupported',('A04','HANA','largest_retail_category_auto_discount_rate'):'unsupported',('A04','HANA','ev_charging_discount_rate'):'supported_and_satisfies',('A09','KB','dedicated_theme_park_benefit'):'supported_and_satisfies',('A09','KB','dedicated_hospital_pharmacy_benefit'):'unsupported',('A09','SHINHAN','dedicated_theme_park_benefit'):'unsupported',('A09','SHINHAN','dedicated_hospital_pharmacy_benefit'):'supported_and_satisfies',('A10','HANA','dedicated_ev_charging_discount'):'supported_and_satisfies',('A10','HANA','airport_or_luxury_hotel_free_valet'):'unsupported',('A10','HYUNDAI','dedicated_ev_charging_discount'):'unsupported',('A10','HYUNDAI','airport_or_luxury_hotel_free_valet'):'supported_and_satisfies',('O07','IBK','family_card_annual_fee'):'unsupported',('O07','IBK','next_year_base_annual_fee_waiver'):'unsupported'}
contradictions=[{'key':k,'expected':v,'actual':outcome(*k)} for k,v in named_expectations.items() if outcome(*k)!=v]; need(not contradictions,'named qualification contradictions')

# Rebuild active-claim projections while preserving exact v3 component-level lineage maps.
component_maps={}
for p in old_proj:
    if p['projection_scope']!='atomic_claim': continue
    for sid,ids in p['full_corpus_projection']['direct_body_component_chunk_ids'].items():
        key=(p['corpus'],sid); value={'direct':ids,'title':p['full_corpus_projection']['inherited_title_component_chunk_ids'][sid],'logical':p['full_corpus_projection']['logical_component_chunk_ids'][sid],'eligible':p['searchable_eligible_projection']['component_chunk_ids'][sid]}
        if key in component_maps: need(component_maps[key]==value,f'component projection conflict {key}')
        component_maps[key]=value
projections=[]
for c in active_claims:
    for corpus in ('OLD','STRUCT'):
        maps={s:component_maps[(corpus,s)] for s in c['required_component_ids']}
        projections.append({'schema_version':VERSION,'projection_scope':'atomic_claim','atomic_claim_id':c['atomic_claim_id'],'card_id':c['card_id'],'corpus':corpus,'projection_method':'v3_exact_component_lineage_reuse_no_fuzzy','fuzzy_matching_used':False,'raw_source_reproduction_complete':True,'full_corpus_projection':{'direct_body_component_chunk_ids':{s:v['direct'] for s,v in maps.items()},'inherited_title_component_chunk_ids':{s:v['title'] for s,v in maps.items()},'logical_component_chunk_ids':{s:v['logical'] for s,v in maps.items()},'full_corpus_logical_presence_complete':all(v['logical'] for v in maps.values())},'searchable_eligible_projection':{'component_chunk_ids':{s:v['eligible'] for s,v in maps.items()},'chunk_ids':sorted({x for v in maps.values() for x in v['eligible']}),'searchable_eligible_projection_complete':all(v['eligible'] for v in maps.values()),'rule':'OLD_section_benefit_direct_body_only' if corpus=='OLD' else 'STRUCT_direct_body_or_exact_inherited_heading_title'},'payload_evidence_projection':{'payload_evidence_projection_complete':False,'status':'not_frozen_before_label_free_candidate_execution','eligible_component_chunk_ids':{s:v['eligible'] for s,v in maps.items()}},'candidate_projection_miss':not all(v['eligible'] for v in maps.values()),'dataset_invalid':False,'status':STATUS})
for rid in ('O09','O10','A09','A10'):
    for corpus in ('OLD','STRUCT'): projections.append({'schema_version':VERSION,'projection_scope':'negative_record','record_id':rid,'corpus':corpus,'raw_source_reproduction_complete':'not_applicable_negative','full_corpus_projection':'not_applicable_negative','searchable_eligible_projection':'not_applicable_negative','payload_evidence_projection':'not_applicable_negative','dataset_invalid':False,'status':STATUS})
need(len(projections)==104 and not any(p.get('candidate_projection_miss') for p in projections),'active projection count/miss')
need(sha(OUT/title_name)==title_hash_before and (OUT/title_name).read_bytes()==title_bytes,'title lineage changed')

# Canonical conversions, operators and executable metric reference.
def convert(value,conversion_id):
    spec=conversions[conversion_id]; typ=spec['type']
    if typ in {'identity','identity_boolean','identity_count'}: return value
    if typ=='divide': return value/spec['divisor']
    if typ=='korean_money_to_integer':
        if isinstance(value,(int,float)): return int(value)
        s=str(value).replace(',','').replace('원','').strip(); need(s.isdigit(),'non-canonical KRW'); return int(s)
    raise RuntimeError('unsupported conversion '+typ)
def op_eval(actual,op,expected):
    if op=='eq': return actual==expected
    if op=='gte': return isinstance(actual,(int,float)) and actual>=expected
    if op=='gt': return isinstance(actual,(int,float)) and actual>expected
    if op=='lte': return isinstance(actual,(int,float)) and actual<=expected
    if op=='neq': return actual!=expected
    if op=='in': return actual in expected
    if op=='not_in': return actual not in expected
    if op=='all_of': return isinstance(actual,(list,tuple,set)) and all(x in actual for x in expected)
    if op=='any_of': return isinstance(actual,(list,tuple,set)) and any(x in actual for x in expected)
    if op=='contains': return expected in actual
    if op=='exists': return (actual is not None)==bool(expected)
    if op=='object_has_any_key': return isinstance(actual,dict) and any(k in actual and actual[k] is not None for k in expected)
    raise RuntimeError('unsupported operator '+op)
def score_record(expected,returned,supported,required_claims,supported_claims,citations,owned_citations,insufficient=False,critical_errors=0,wrong_card_errors=0):
    expected=list(dict.fromkeys(expected)); returned=list(dict.fromkeys(returned))[:3]; supported=set(supported); required_claims=set(required_claims); supported_claims=set(supported_claims)
    if expected:
        hits=len(set(returned)&set(expected)); precision=hits/len(returned) if returned else 0.0; recall=hits/len(expected); supported_recall=len(set(expected)&supported)/len(expected); coverage=len(required_claims&supported_claims)/len(required_claims) if required_claims else 1.0; ownership=owned_citations/citations if citations else 0.0; negative_correct=None
        e2e=int(set(returned)==set(expected[:3]) and coverage==1.0 and supported_recall==1.0 and ownership==1.0 and critical_errors==0 and wrong_card_errors==0 and not insufficient)
    else:
        precision=recall=supported_recall=coverage=None; ownership=None; negative_correct=int(not returned and insufficient and wrong_card_errors==0); e2e=negative_correct
    return {'card_precision_at_3':precision,'card_recall_at_3':recall,'supported_card_recall_at_3':supported_recall,'required_claim_coverage':coverage,'citation_ownership':ownership,'negative_correctness':negative_correct,'end_to_end_exact_success':e2e}
operator_tests={'eq':op_eval(1,'eq',1),'gte':op_eval(2,'gte',1),'gt':op_eval(2,'gt',1),'lte':op_eval(1,'lte',2),'neq':op_eval(1,'neq',2),'in':op_eval('a','in',['a']),'not_in':op_eval('b','not_in',['a']),'all_of':op_eval(['a','b'],'all_of',['a','b']),'any_of':op_eval(['a'],'any_of',['a','b']),'contains':op_eval(['a'],'contains','a'),'exists':op_eval(0,'exists',True),'object_has_any_key':op_eval({'a':1},'object_has_any_key',['a'])}
conversion_tests={'CV_BOOL_EXACT':convert(False,'CV_BOOL_EXACT') is False,'CV_PERCENT_TO_RATIO':convert(5,'CV_PERCENT_TO_RATIO')==0.05,'CV_KRW_CANONICAL':convert('13,000원','CV_KRW_CANONICAL')==13000,'CV_COUNT_CANONICAL':convert(2,'CV_COUNT_CANONICAL')==2,'CV_IDENTITY':convert({'a':1},'CV_IDENTITY')=={'a':1}}
fixtures={'positive_perfect':score_record(['A'],['A'],['A'],['c1'],['c1'],1,1),'positive_partial':score_record(['A','B'],['A'],['A'],['c1','c2'],['c1'],1,1),'positive_no_output':score_record(['A'],[],[],['c1'],[],0,0),'positive_zero_citation':score_record(['A'],['A'],['A'],['c1'],['c1'],0,0),'positive_wrong_card':score_record(['A'],['B'],[],['c1'],[],1,1,wrong_card_errors=1),'negative_correct':score_record([],[],[],[],[],0,0,insufficient=True),'negative_false_recommendation':score_record([],['A'],[],[],[],0,0,insufficient=False,wrong_card_errors=1),'returned_less_than_3':score_record(['A','B'],['A'],['A'],['c1'],['c1'],1,1)}
need(all(operator_tests.values()) and all(conversion_tests.values()),'operator/conversion fixtures')
need(fixtures['positive_perfect']['end_to_end_exact_success']==1 and fixtures['positive_partial']['card_recall_at_3']==0.5 and fixtures['positive_no_output']['card_precision_at_3']==0 and fixtures['positive_zero_citation']['end_to_end_exact_success']==0 and fixtures['positive_wrong_card']['end_to_end_exact_success']==0 and fixtures['negative_correct']['negative_correctness']==1 and fixtures['negative_false_recommendation']['negative_correctness']==0 and fixtures['returned_less_than_3']['card_precision_at_3']==1.0,'metric fixture assertions')
metric_contract={'card_precision_at_3':{'record_numerator':'unique returned cards in expected set','record_denominator':'actual unique returned count capped at 3; positive no output=0','aggregate':'macro over positive 26'},'card_recall_at_3':{'record_numerator':'unique returned expected cards','record_denominator':'expected cards','aggregate':'macro over positive 26'},'supported_card_recall_at_3':{'record_numerator':'expected cards returned with satisfied payload graph','record_denominator':'expected cards','aggregate':'macro over positive 26'},'required_claim_coverage':{'record_numerator':'supported required atomic claims','record_denominator':'required atomic claims','aggregate':'macro over positive 26'},'negative_correctness':{'record_numerator':'correct closed-corpus abstention','record_denominator':'1 per negative','aggregate':'count and macro over negative 4'},'citation_ownership':{'record_numerator':'owned citations','record_denominator':'total citations','zero_citation_positive':0,'correct_negative_abstention':'N/A','aggregate':'positive records only'},'end_to_end_exact_success':{'record_value':'binary 0/1 from scorer','aggregate':'count and macro over all 30'}}
selection={'status':'predeclared_accuracy_first_v4','absolute_eligibility':{'end_to_end_exact_success':{'minimum_count':21,'denominator':30},'positive_supported_card_recall_at_3':{'minimum':0.75,'denominator_records':26},'required_claim_coverage':{'minimum':0.80,'denominator_records':26},'negative_correctness':{'required_count':4,'denominator':4},'critical_unsupported_numeric_or_condition_errors':{'maximum':0},'wrong_card_catastrophic_errors':{'maximum':0},'citation_ownership':{'required':1.0,'aggregate_denominator':'positive citations only','zero_citation_positive_is_failure':True,'correct_negative_abstention':'not_applicable'}},'paired_selection_metric':'per-record end_to_end_exact_success (0/1)','paired_rule':{'win':'candidate 1 and baseline 0','loss':'candidate 0 and baseline 1','tie':'equal','denominator':30},'critical_cohort_regression':'baseline e2e=1 to candidate=0 on a cohort-critical record OR any new critical unsupported/wrong-card/negative false recommendation','winner_rule':{'invalid':'invalid','exactly_one_candidate_absolute_eligible':'select_that_candidate','both_candidates_absolute_eligible':['absolute end_to_end gap >= 3/30','paired wins > paired losses','supported_card_recall_at_3 nonregression','required_claim_coverage nonregression','critical_cohort_regression_count == 0'],'both_candidates_hard_or_absolute_fail':'holdout_failed','otherwise':'inconclusive'},'candidate_seal_required_before':['ranking','answer_generation'],'post_result_tuning_forbidden':True}
operator_contract={'eq':'normalized actual equals expected exactly','gte':'numeric actual >= expected after declared conversion','gt':'numeric actual > expected after declared conversion','lte':'numeric actual <= expected after declared conversion','neq':'normalized actual differs from expected','in':'scalar actual belongs to expected collection','not_in':'scalar actual does not belong to expected collection','all_of':'actual collection contains every expected member','any_of':'actual collection contains at least one expected member','contains':'actual collection or string contains expected','exists':'actual non-null existence equals expected boolean','object_has_any_key':'actual object has at least one requested key with non-null supported value'}
scoring={'schema_version':VERSION,'status':STATUS,'conversion_registry':conversions,'operators':operator_contract,'metric_contract':metric_contract,'denominators':{'positive_records':26,'negative_records':4,'all_records':30},'selection_contract':selection,'reference_scorer':'score_record in revision cell','operator_self_tests':operator_tests,'conversion_self_tests':conversion_tests,'fixture_results':fixtures,'result_constants_used_in_tests':False,'execution_deferred':True}
scorer_audit={'schema_version':VERSION,'status':'PASS','operator_tests':operator_tests,'conversion_tests':conversion_tests,'fixtures':fixtures,'fixture_classes':['perfect','partial','no_output','zero_citation','wrong_card','correct_negative','false_negative_recommendation','returned_less_than_3'],'results_independent_of_candidate_outputs':True}

# Recompute four-inventory duplicate audit after the clarified A05 query text.
def norm(x): return ' '.join(re.sub(r'[^\w가-힣]+',' ',unicodedata.normalize('NFKC',str(x)).lower()).split())
specs=[r for r in parts[7]['duplicate_audit']['inventories'] if r['path'].endswith('.csv')]; inventory=[]; inventory_hashes={}
for spec in specs:
    p=ROOT/spec['path']; need(sha(p)==spec['sha256'],f'inventory hash {p}'); inventory_hashes[spec['path']]=spec['sha256']
    with p.open(encoding='utf-8',newline='') as f: inventory += [(spec['path'],r['query_id'],norm(r['query_text'])) for r in csv.DictReader(f)]
dups=[]
for r in queries['records']:
    n=norm(r['sealed_query_text']); ts=set(n.split()); exact=[]; near=[]; mx=0.0
    for path,qid,t in inventory:
        if n==t: exact.append({'inventory_path':path,'query_id':qid})
        os_=set(t.split()); j=len(ts&os_)/len(ts|os_) if ts|os_ else 1.0; mx=max(mx,j)
        if j>=0.8: near.append({'inventory_path':path,'query_id':qid,'token_jaccard':j})
    dups.append({'schema_version':VERSION,'record_id':r['record_id'],'query_normalized_sha256':digest(n.encode()),'exact_normalized_matches':exact,'token_jaccard_threshold':0.8,'token_jaccard_matches':near,'maximum_token_jaccard':mx,'manual_semantic_status':STATUS,'status':STATUS})
need(sum(len(x['exact_normalized_matches'])+len(x['token_jaccard_matches']) for x in dups)==0,'duplicate collisions')

proposal=copy.deepcopy(json.loads((OUT/'proposal_canonical.json').read_text())); proposal.pop('canonical_sha256',None); proposal['schema_version']=VERSION; proposal['status']=STATUS
for c in claims:
    compact=[c[k] for k in ('card_id','predicate','canonical_value','unit')] + [[[x[k] for k in ('field','operator','value','unit','required_for_complete_answer')] for x in c['conditions']],c['criticality'],c['requirement'],c['support_logic_type'],c['required_component_ids'],c['conversion_id']]
    proposal['claims'][c['atomic_claim_id']]=compact
proposal['claim_metadata']={c['atomic_claim_id']:{k:c[k] for k in ('evaluation_status','retirement_reason','related_component_ids','related_claim_ids') if k in c} for c in claims}
proposal['records']['A05'][0]=a05; proposal['selection_contract']=selection; proposal['conversion_registry']=conversions
proposal['v3_draft_disposition']='discarded_not_sealable'; proposal['v3_discarded_provenance']={'proposal_canonical_sha256':v3_provenance['proposal_canonical_sha256'],'generation_id':v3_provenance['generation_id'],'manifest_sha256':sha(OUT/'draft_dataset_manifest.json')}
proposal['known_limitations']=['single analyst v4 correction awaits independent review','payload evidence remains unfrozen until a future sealed label-free execution','C35 retained only as retired non-evaluation provenance']
proposal_no=copy.deepcopy(proposal); proposal_sha=digest(canon(proposal_no).encode()); proposal['canonical_sha256']=proposal_sha; generation=f'integrated_holdout_master_v4_{proposal_sha[:16]}'
for rows in (sources,claims,queries['records'],master,qual,projections,dups):
    for r in rows: r['schema_version']=VERSION; r['generation_id']=generation; r['status']=STATUS
queries['schema_version']=VERSION; queries['generation_id']=generation; queries['status']=STATUS
scoring['generation_id']=generation; scorer_audit['generation_id']=generation
source_refs={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'unit':'source_component_id','refs':sources}
claim_doc={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'active_evaluation_claim_count':48,'retired_claim_count':1,'claims':claims}
policies=json.loads((OUT/'answer_policies.json').read_text()); policies['schema_version']=VERSION; policies['generation_id']=generation; policies['status']=STATUS
q_groups={rid:[q for q in qual if q['record_id']==rid] for rid in m_by}
audits=[{'schema_version':VERSION,'generation_id':generation,'record_id':r['record_id'],'source_components_reproduced':len(r['required_source_component_ids']),'qualification_rows_reviewed':10,'predicate_outcome_contradictions':0,'full_source_reviewed':all(q['full_source_reviewed'] for q in q_groups[r['record_id']]),'reviewer_proposal':None,'status':STATUS,'limitation':'single-analyst correction pending independent review'} for r in master]
reviews=[{'schema_version':VERSION,'generation_id':generation,'record_id':r['record_id'],'sealed_query_text':next(q['sealed_query_text'] for q in queries['records'] if q['record_id']==r['record_id']),'task_family':r['task_family'],'expected_card_ids':r['expected_card_ids'],'atomic_claims':[by_claim[c] for c in r['atomic_claim_ids']],'source_components':[next(s for s in sources if s['source_component_id']==sid) for sid in r['required_source_component_ids']],'qualification_rows':q_groups[r['record_id']],'answer_policy_ref':r['answer_policy_ref'],'reviewer_proposal':None,'review_status':STATUS} for r in master]
nb=json.loads(NOTEBOOK.read_text()); cell=next(c for c in nb['cells'] if c.get('id')==CELL_ID); source=''.join(cell['source']) if isinstance(cell['source'],list) else cell['source']; cell_sha=digest(canon({'cell_id':CELL_ID,'source_text':source}).encode())

restricted=['README.md','annotation_audit.jsonl','answer_policies.json','atomic_claims.json','candidate_projection.jsonl','draft_dataset_manifest.json','draft_integrity.json','draft_review_packet.jsonl','draft_scoring_contract.json','evaluation_queries.json','master_gold.jsonl','proposal_canonical.json','qualification_by_card.jsonl','query_duplicate_audit.jsonl','source_components.jsonl','source_refs.json','scorer_reference_audit.json']
stage=Path(tempfile.mkdtemp(prefix='.v4_stage_',dir=OUT))
try:
    put_json(stage/'proposal_canonical.json',proposal); put_json(stage/'evaluation_queries.json',queries); put_jsonl(stage/'master_gold.jsonl',master); put_jsonl(stage/'source_components.jsonl',sources); put_json(stage/'source_refs.json',source_refs); put_json(stage/'atomic_claims.json',claim_doc); put_json(stage/'answer_policies.json',policies); put_jsonl(stage/'qualification_by_card.jsonl',qual); put_jsonl(stage/'annotation_audit.jsonl',audits); put_jsonl(stage/'candidate_projection.jsonl',projections); put_jsonl(stage/'query_duplicate_audit.jsonl',dups); put_json(stage/'draft_scoring_contract.json',scoring); put_json(stage/'scorer_reference_audit.json',scorer_audit); put_jsonl(stage/'draft_review_packet.jsonl',reviews)
    query_sha=sha(stage/'evaluation_queries.json'); gold_sha=sha(stage/'master_gold.jsonl'); scoring_sha=sha(stage/'draft_scoring_contract.json')
    (stage/'README.md').write_text(f'''# Integrated holdout master v4\n\n상태: `{STATUS}`. v3 reviewer 잔여 blocker/major를 보정한 accuracy-first restricted core입니다. 독립 검토와 candidate seal 전에는 ranking·answer 생성이 금지됩니다.\n\n- canonical SHA-256: `{proposal_sha}`\n- source components 78, claims 49(total)/48(active)/1(retired C35), records 30, qualification rows 300\n- C28/C29의 1만원은 최소 결제액이 아니라 할인 적용 승인금액 한도이며 건당 최대 할인은 1천원입니다.\n- O02 두 OR branch 모두 지정계좌와 전월 40만원 조건을 요구합니다.\n- metric은 positive26 record macro, negative4 correctness, all30 e2e exact로 고정했습니다.\n- title lineage와 기존 lineage/demo는 byte-exact 보존했습니다. active claim projection은 corrected component refs에 맞춰 재생성했습니다.\n- API/network/GPU/embedding/retrieval/LLM: 0\n\n단일 분석가 수정본이며 payload evidence는 아직 동결되지 않았습니다. C35는 ID provenance만 유지하는 retired non-evaluation claim입니다.\n''',encoding='utf-8')
    artifacts=sorted(n for n in restricted if n not in {'draft_dataset_manifest.json','draft_integrity.json'})
    manifest={'schema_version':VERSION,'generation_id':generation,'status':STATUS,'proposal_canonical_sha256':proposal_sha,'evaluation_queries_raw_sha256':query_sha,'master_gold_raw_sha256':gold_sha,'scoring_contract_raw_sha256':scoring_sha,'candidate_seal_status':'forbidden_pending_independent_review','ranking_answer_forbidden_before_candidate_seal':True,'v3_draft_disposition':'discarded_not_sealable','v3_discarded_provenance':v3_provenance,'proposal_part_raw_sha256':part_hashes,'revision_cell_source_sha256':cell_sha,'preserved_lineage_demo_raw_sha256':preserved_before,'preserved_title_lineage_raw_sha256':title_hash_before,'v3_source_components_raw_sha256':source_before,'v3_candidate_projection_raw_sha256':projection_before,'inventory_raw_sha256':inventory_hashes,'artifacts':{n:sha(stage/n) for n in artifacts},'manifest_self_hash_excluded':True,'integrity_self_hash_excluded':True,'atomic_replacement':'complete stage then os.replace; manifest/integrity last','execution':{'environment':'skn25','cpu_offline':True,'api_requests':0,'network':0,'gpu':0,'embedding':0,'retrieval':0,'llm':0},'known_limitations':proposal['known_limitations']}
    put_json(stage/'draft_dataset_manifest.json',manifest)
    integrity={'schema_version':VERSION,'generation_id':generation,'status':'PASS_PENDING_INDEPENDENT_REVIEW','dataset_status':STATUS,'manifest_sha256':sha(stage/'draft_dataset_manifest.json'),'proposal_canonical_sha256':proposal_sha,'counts':{'source_components':78,'atomic_claims_total':49,'atomic_claims_active':48,'atomic_claims_retired':1,'records':30,'qualification_rows':300,'qualification_false_rows':sum(not q['qualified'] for q in qual),'claim_projection_rows':96,'negative_projection_na_rows':8,'projection_misses':0,'annotation_rows':30,'review_packet_rows':30},'assertions':{'C28_C29_cap_semantics':True,'O02_both_branches_eligible':True,'full_scoring_contract':True,'operator_conversion_metric_fixtures':True,'qualification_predicate_contradictions_zero':True,'C38_C41_C03_collection_semantics':True,'C35_retired':True,'C42_C43_optional_context_separated':True,'C49_related_claim_link':True,'title_lineage_byte_exact':True,'lineage_demo_byte_exact':True,'reviewer_proposal_null':True,'v3_discarded':True,'candidate_seal_required':True,'api_network_gpu_embedding_retrieval_llm_zero':True},'artifact_hashes':manifest['artifacts']}
    put_json(stage/'draft_integrity.json',integrity)
    need(all(sha(OUT/n)==h for n,h in preserved_before.items()) and sha(OUT/title_name)==title_hash_before,'frozen lineage changed before replace')
    order=[n for n in restricted if n not in {'draft_dataset_manifest.json','draft_integrity.json'}]+['draft_dataset_manifest.json','draft_integrity.json']; need(set(order)==set(restricted),'replacement set')
    for n in order: os.replace(stage/n,OUT/n)
finally: shutil.rmtree(stage,ignore_errors=True)
need(all(sha(OUT/n)==h for n,h in preserved_before.items()) and sha(OUT/title_name)==title_hash_before,'frozen lineage changed after replace')
saved=json.loads((OUT/'draft_dataset_manifest.json').read_text()); integ=json.loads((OUT/'draft_integrity.json').read_text())
need(saved['generation_id']==generation==integ['generation_id'],'generation mismatch'); need(all(sha(OUT/n)==h for n,h in saved['artifacts'].items()),'artifact hash mismatch')
need(sha(OUT/'evaluation_queries.json')==query_sha and sha(OUT/'master_gold.jsonl')==gold_sha and sha(OUT/'draft_scoring_contract.json')==scoring_sha,'query/gold/scoring hash')
print(canon({'status':'PASS_PENDING_INDEPENDENT_REVIEW','proposal_canonical_sha256':proposal_sha,'generation_id':generation,'source_components':78,'atomic_claims_total':49,'atomic_claims_active':48,'atomic_claims_retired':1,'records':30,'qualification_rows':300,'qualification_predicate_contradictions':0,'projection_rows':104,'projection_misses':0,'title_lineage_byte_exact':True,'candidate_seal_status':'forbidden_pending_independent_review','api_network_gpu_embedding_retrieval_llm':0}))


{"api_network_gpu_embedding_retrieval_llm":0,"atomic_claims_active":48,"atomic_claims_retired":1,"atomic_claims_total":49,"candidate_seal_status":"forbidden_pending_independent_review","generation_id":"integrated_holdout_master_v4_8ab30d91da2c2ba3","projection_misses":0,"projection_rows":104,"proposal_canonical_sha256":"8ab30d91da2c2ba36b088e417769da9ff2cf8a4ec762bd8ba80051fa9619fce1","qualification_predicate_contradictions":0,"qualification_rows":300,"records":30,"source_components":78,"status":"PASS_PENDING_INDEPENDENT_REVIEW","title_lineage_byte_exact":true}


## v6 scoring contract — derived errors and batch-safe candidate failures

v4 labels/components/projection/query/gold를 byte-exact로 보호하고 scoring contract만 v6로 보정합니다. 오류 count는 response 자기보고가 아니라 scorer가 파생하고, candidate 내용 오류는 레코드 실패로 저장하여 30건 batch를 계속합니다. 독립 검토 전 seal은 계속 금지됩니다.

In [1]:
# Integrated holdout scoring v6: CPU/offline, data core byte-exact.
from __future__ import annotations
import copy, hashlib, json, math, os, shutil, tempfile, unicodedata
from pathlib import Path
cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError('Run from repository root or notebooks/')
NOTEBOOK=ROOT/'notebooks/27_integrated_holdout_dataset.ipynb'; OUT=ROOT/'notebooks/data/27_integrated_holdout_dataset'
STATUS,VERSION,CELL_ID='pending_independent_review','integrated_holdout_scoring_v6','27-v6-scoring-contract'
def need(ok,msg):
    if not ok: raise RuntimeError(msg)
def canon(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def digest(b): return hashlib.sha256(b).hexdigest()
def sha(p): return digest(p.read_bytes())
def put_json(p,v): p.write_text(canon(v),encoding='utf-8')
protected=['evaluation_queries.json','master_gold.jsonl','atomic_claims.json','source_components.jsonl','source_refs.json','qualification_by_card.jsonl','candidate_projection.jsonl','struct_searchable_title_lineage.jsonl','answer_policies.json','annotation_audit.jsonl','draft_review_packet.jsonl','proposal_canonical.json','query_duplicate_audit.jsonl']
protected_before={n:sha(OUT/n) for n in protected}
prior_manifest=json.loads((OUT/'draft_dataset_manifest.json').read_text()); need(prior_manifest['schema_version'] in {'integrated_holdout_master_v4','integrated_holdout_scoring_v5',VERSION},'v4/v5/v6 manifest required')
dataset_generation=prior_manifest['generation_id'] if prior_manifest['schema_version']=='integrated_holdout_master_v4' else prior_manifest['dataset_generation_id']
queries=json.loads((OUT/'evaluation_queries.json').read_text())['records']; records=[json.loads(x) for x in (OUT/'master_gold.jsonl').read_text().splitlines() if x]
claims=json.loads((OUT/'atomic_claims.json').read_text())['claims']; sources=[json.loads(x) for x in (OUT/'source_components.jsonl').read_text().splitlines() if x]
by_record={r['record_id']:r for r in records}; by_claim={r['atomic_claim_id']:r for r in claims}; by_source={r['source_component_id']:r for r in sources}
record_order=[*(f'D{i:02d}' for i in range(1,11)),*(f'O{i:02d}' for i in range(1,11)),*(f'A{i:02d}' for i in range(1,11))]; need([r['record_id'] for r in records]==record_order,'exact record order')
positive_ids=[r['record_id'] for r in records if r['expected_card_ids']]; negative_ids=[r['record_id'] for r in records if not r['expected_card_ids']]; need(len(positive_ids)==26 and negative_ids==['O09','O10','A09','A10'],'26/4 denominators')
axes={r['record_id']:set(r['axes']) for r in queries}; critical_ids=[rid for rid in record_order if 'numeric_condition_critical' in axes[rid] or rid in negative_ids]
need(critical_ids==['D01','D04','D05','D06','D07','D08','D10','O05','O06','O07','O09','O10','A02','A03','A04','A06','A07','A09','A10'],'critical IDs freeze')
conversions=json.loads((OUT/'proposal.part01.json').read_text())['conversions']
canonical_cards=['BC','NH','HANA','HYUNDAI','IBK','KB','LOTTE','SAMSUNG','SHINHAN','WOORI']

def normalize_scalar(value):
    if isinstance(value,str): return ' '.join(unicodedata.normalize('NFKC',value).casefold().split())
    if isinstance(value,list): return [normalize_scalar(x) for x in value]
    if isinstance(value,dict): return {k:normalize_scalar(v) for k,v in value.items()}
    return value
def convert_value(raw_value,raw_unit,conversion_id,target_unit):
    raw_value=normalize_scalar(raw_value); raw_unit=normalize_scalar(raw_unit); target_unit=normalize_scalar(target_unit)
    spec=conversions[conversion_id]; typ=spec['type']
    if typ=='divide':
        need(raw_unit==normalize_scalar(spec['source_unit']) and target_unit==normalize_scalar(spec['target_unit']),'percent conversion unit'); return raw_value/spec['divisor']
    if typ=='korean_money_to_integer':
        need(raw_unit in {'krw','원'},'KRW unit'); value=int(str(raw_value).replace(',','').replace('원','').strip()); need(target_unit.startswith('krw'),'KRW target'); return value
    if typ=='identity_boolean': need(isinstance(raw_value,bool) and raw_unit in {None,'boolean'},'boolean identity'); return raw_value
    if typ=='identity_count': need(isinstance(raw_value,(int,float)) and raw_unit==target_unit,'count identity unit'); return raw_value
    if typ=='identity': need(raw_unit==target_unit,'identity unit'); return raw_value
    raise RuntimeError('unsupported conversion')
def convert_condition(raw_value,raw_unit,target_unit):
    raw_value=normalize_scalar(raw_value); raw_unit=normalize_scalar(raw_unit); target_unit=normalize_scalar(target_unit)
    if target_unit is None: need(raw_unit is None,'condition unit must be null'); return raw_value
    if target_unit=='ratio' and raw_unit=='percent': return raw_value/100
    if target_unit.startswith('krw') and raw_unit in {'krw','원'}: return int(str(raw_value).replace(',','').replace('원','').strip())
    need(raw_unit==target_unit,'condition unit mismatch'); return raw_value
def op(actual,operator,expected):
    actual,expected=normalize_scalar(actual),normalize_scalar(expected)
    if operator=='eq': return actual==expected
    if operator=='gte': return isinstance(actual,(int,float)) and actual>=expected
    if operator=='gt': return isinstance(actual,(int,float)) and actual>expected
    if operator=='lte': return isinstance(actual,(int,float)) and actual<=expected
    if operator=='neq': return actual!=expected
    if operator=='in': return actual in expected
    if operator=='not_in': return actual not in expected
    if operator=='all_of': return isinstance(actual,(list,tuple,set)) and all(x in actual for x in expected)
    if operator=='any_of': return isinstance(actual,(list,tuple,set)) and any(x in actual for x in expected)
    if operator=='contains': return expected in actual
    if operator=='exists': return (actual is not None)==bool(expected)
    if operator=='object_has_any_key': return isinstance(actual,dict) and any(k in actual and actual[k] is not None for k in expected)
    raise RuntimeError('unknown operator')

context_hash_keys={'candidate_sha256','source_sha256','query_sha256','gold_sha256','scorer_sha256','prompt_sha256','model_sha256'}
def validate_execution_context(actual,expected):
    need(isinstance(actual,dict) and isinstance(expected,dict) and set(actual)==set(expected)==context_hash_keys,'execution context schema invalid')
    need(all(isinstance(v,str) and len(v)==64 for v in actual.values()),'execution context hash format')
    need(actual==expected,'execution context hash mismatch')
response_keys={'record_id','returned_cards','mentioned_cards','extracted_claims','insufficient_evidence'}
claim_keys={'card_id','claim_id','raw_value','raw_unit','raw_conditions','citations'}
allowed_ops={'eq','gte','gt','lte','neq','in','not_in','all_of','any_of','contains','exists','object_has_any_key'}
def validate_frozen_contract(records_map):
    need(set(records_map)==set(record_order),'contract record IDs corrupted')
    for claim in claims:
        need(claim['conversion_id'] in conversions and all(x['operator'] in allowed_ops for x in claim['conditions']),'claim conversion/operator corruption')
        need(all(s in by_source for s in claim['required_component_ids']),'claim source corruption')
    for rid in record_order:
        record=records_map[rid]; need(not record['expected_card_ids'] or record['atomic_claim_ids'],'empty required claims on positive')
        need(set(record['qualification_graphs'])==set(record['expected_card_ids']),'qualification graph/card corruption')
        for card,graph in record['qualification_graphs'].items():
            need(set(graph) in ({'all_of'},{'any_of'}),'qualification graph branch corruption')
            for cid,operator,_ in next(iter(graph.values())): need(cid in by_claim and by_claim[cid]['card_id']==card and operator in allowed_ops,'qualification graph leaf corruption')
validate_frozen_contract(by_record)
def normalize_response_content(response):
    reasons=[]
    if set(response)!=response_keys: reasons.append('response_schema_keys')
    returned=response.get('returned_cards',[]); mentioned=response.get('mentioned_cards',[]); rows=response.get('extracted_claims',[]); insufficient=response.get('insufficient_evidence',False)
    if not isinstance(returned,list) or not all(isinstance(x,str) for x in returned): reasons.append('returned_cards_schema'); returned=[]
    if not isinstance(mentioned,list) or not all(isinstance(x,str) for x in mentioned): reasons.append('mentioned_cards_schema'); mentioned=[]
    if not isinstance(rows,list): reasons.append('extracted_claims_schema'); rows=[]
    if not isinstance(insufficient,bool): reasons.append('insufficient_evidence_schema'); insufficient=False
    if len(returned)>3: reasons.append('returned_cards_exceed_k3')
    if len(returned)!=len(set(returned)): reasons.append('duplicate_returned_cards')
    if len(mentioned)!=len(set(mentioned)): reasons.append('duplicate_mentioned_cards')
    if any(x not in canonical_cards for x in returned+mentioned): reasons.append('unknown_card')
    unique=[]
    for card in returned:
        if card not in unique: unique.append(card)
    return {'returned_cards':returned,'returned_top3':unique[:3],'mentioned_cards':mentioned,'extracted_claims':rows,'insufficient_evidence':insufficient},reasons
def support_claim(row):
    reasons=[]; format_reasons=[]; citation_reasons=[]; converted=None; value_ok=conditions_ok=citation_complete=False; condition_audit=[]
    if not isinstance(row,dict): row={}; format_reasons.append('claim_not_object')
    if set(row)!=claim_keys: format_reasons.append('claim_schema_keys')
    card=row.get('card_id'); cid=row.get('claim_id'); citations=row.get('citations',[]); raw_conditions=row.get('raw_conditions',{})
    if not isinstance(card,str) or card not in canonical_cards: format_reasons.append('unknown_claim_card')
    claim=by_claim.get(cid) if isinstance(cid,str) else None
    if claim is None: format_reasons.append('unknown_claim')
    elif card!=claim['card_id']: format_reasons.append('claim_card_mismatch')
    if not isinstance(citations,list) or not all(isinstance(x,str) for x in citations): format_reasons.append('citation_schema'); citations=[]
    if len(citations)!=len(set(citations)): format_reasons.append('duplicate_citation')
    if not isinstance(raw_conditions,dict): format_reasons.append('raw_conditions_schema'); raw_conditions={}
    unknown=[s for s in citations if s not in by_source]
    if unknown: citation_reasons.append('unknown_source')
    owned=[s for s in citations if s in by_source and by_source[s]['card_id']==card]
    if len(owned)!=len(citations): citation_reasons.append('citation_ownership_mismatch')
    required=set(claim['required_component_ids']) if claim else set()
    if claim and not set(citations)<=required: citation_reasons.append('citation_excess')
    if claim and not required<=set(citations): citation_reasons.append('required_citation_missing')
    if claim and card==claim['card_id']:
        try:
            converted=convert_value(row.get('raw_value'),row.get('raw_unit'),claim['conversion_id'],claim['unit']); value_ok=op(converted,'eq',claim['canonical_value'])
        except (RuntimeError,ValueError,TypeError): reasons.append('value_or_unit_mismatch')
        conditions_ok=True
        for cond in claim['conditions']:
            if not cond['required_for_complete_answer']: continue
            raw=raw_conditions.get(cond['field']); ok=False
            if isinstance(raw,dict) and set(raw)=={'value','unit'}:
                try: ok=op(convert_condition(raw['value'],raw['unit'],cond['unit']),cond['operator'],cond['value'])
                except (RuntimeError,ValueError,TypeError): ok=False
            condition_audit.append({'field':cond['field'],'pass':ok}); conditions_ok &= ok
        citation_complete=bool(required) and required<=set(citations) and not citation_reasons
    if claim and not value_ok: reasons.append('wrong_value')
    if claim and not conditions_ok: reasons.append('required_condition_failure')
    reasons.extend(format_reasons+citation_reasons); supported=bool(claim and card==claim['card_id'] and value_ok and conditions_ok and citation_complete and not format_reasons)
    return {'card_id':card,'claim_id':cid,'converted_value':converted,'value_pass':value_ok,'required_conditions_pass':conditions_ok,'citation_complete':citation_complete,'claim_supported':supported,'condition_audit':condition_audit,'owned_citations':len(owned),'total_citations':len(citations),'format_errors':sorted(set(format_reasons)),'citation_errors':sorted(set(citation_reasons)),'outcome_reasons':sorted(set(reasons)),'critical_active':bool(claim and claim.get('evaluation_status')=='active_evaluation' and str(claim.get('criticality','')).startswith('critical'))}
def graph_support(record,support_by_claim,card):
    graph=record['qualification_graphs'][card]; branch='all_of' if 'all_of' in graph else 'any_of'; states=[]
    for cid,operator,expected in graph[branch]:
        state=support_by_claim.get((card,cid)); states.append(bool(state and state['claim_supported'] and op(state['converted_value'],operator,expected)))
    return (all(states) if branch=='all_of' else any(states)),states,branch
def score_response(response):
    record=by_record[response['record_id']]; normalized,schema_reasons=normalize_response_content(response); audits=[support_claim(x) for x in normalized['extracted_claims']]
    support_by={}
    for state in audits:
        key=(state['card_id'],state['claim_id'])
        if key not in support_by or state['claim_supported']: support_by[key]=state
    returned=normalized['returned_top3']; expected=record['expected_card_ids']; graph={card:graph_support(record,support_by,card) for card in expected}; graph_satisfied={card for card,(ok,_,_) in graph.items() if ok}; supported_returned=set(expected)&set(returned)&graph_satisfied
    card_signals=[x for x in normalized['returned_cards']+normalized['mentioned_cards'] if isinstance(x,str)]+[x['card_id'] for x in audits if isinstance(x['card_id'],str)]+[by_claim[x['claim_id']]['card_id'] for x in audits if x['claim_id'] in by_claim]
    unexpected=sorted(set(card_signals)-set(expected)); wrong_card_count=len(unexpected); negative_false_count=int(not expected and bool(card_signals))
    required_pairs={(card,cid) for card,g in record['qualification_graphs'].items() for cid,_,_ in next(iter(g.values()))}
    critical_bad={(x['card_id'],x['claim_id']) for x in audits if x['critical_active'] and not x['claim_supported']}
    for card,cid in required_pairs:
        claim=by_claim[cid]; state=support_by.get((card,cid))
        if claim.get('evaluation_status')=='active_evaluation' and str(claim.get('criticality','')).startswith('critical') and not (state and state['claim_supported']): critical_bad.add((card,cid))
    format_count=len(set(schema_reasons))+sum(len(x['format_errors']) for x in audits); citation_count=sum(len(x['citation_errors']) for x in audits); unsupported_count=sum(not x['claim_supported'] for x in audits)
    reasons=list(schema_reasons)+[f"claim:{x['claim_id']}:{reason}" for x in audits for reason in x['outcome_reasons']]+[f'wrong_card:{x}' for x in unexpected]
    if negative_false_count: reasons.append('negative_false_recommendation')
    if critical_bad: reasons.extend(f'critical_claim_failure:{card}:{cid}' for card,cid in sorted(critical_bad))
    reasons=sorted(set(reasons)); failure_count=len(reasons)
    if expected:
        card_hits=len(set(expected)&set(returned)); precision=card_hits/len(returned) if returned else 0.0; recall=card_hits/len(expected); supported_recall=len(supported_returned)/len(expected); units=[]
        for _,(_,states,branch) in graph.items(): units += states if branch=='all_of' else [any(states)]
        need(units,'contract corruption: empty required graph units'); coverage=sum(units)/len(units); total_citations=sum(x['total_citations'] for x in audits); owned=sum(x['owned_citations'] for x in audits); ownership=owned/total_citations if total_citations else 0.0; negative_correct=None
        e2e=int(set(returned)==set(expected) and supported_recall==coverage==ownership==1.0 and failure_count==0 and not normalized['insufficient_evidence'])
    else:
        precision=recall=supported_recall=coverage=ownership=None; negative_correct=int(not card_signals and normalized['insufficient_evidence'] and failure_count==0); e2e=negative_correct
    metrics={'card_precision_at_3':precision,'card_recall_at_3':recall,'supported_card_recall_at_3':supported_recall,'required_claim_coverage':coverage,'citation_ownership':ownership,'negative_correctness':negative_correct,'end_to_end_exact_success':e2e}
    need(all(v is None or isinstance(v,(int,float)) and math.isfinite(v) and 0<=v<=1 for v in metrics.values()),'scorer internal nonfinite metric')
    return {'record_id':record['record_id'],'metrics':metrics,'returned_top3':returned,'graph_satisfied_cards':sorted(graph_satisfied),'supported_returned_cards':sorted(supported_returned),'claim_audit':audits,'critical_error_count':len(critical_bad),'wrong_card_error_count':wrong_card_count,'negative_false_recommendation_count':negative_false_count,'format_error_count':format_count,'citation_error_count':citation_count,'unsupported_claim_count':unsupported_count,'candidate_failure_count':failure_count,'outcome_reasons':reasons}
def score_batch(responses,actual_context,expected_context):
    validate_execution_context(actual_context,expected_context); need(isinstance(responses,list) and len(responses)==30,'exact30 response completeness invalid')
    need(all(isinstance(x,dict) and isinstance(x.get('record_id'),str) for x in responses),'response record ID schema invalid')
    ids=[x['record_id'] for x in responses]; need(ids==record_order and len(set(ids))==30,'missing/duplicate response or order violation')
    return [score_response(x) for x in responses]
def aggregate(scored):
    need(isinstance(scored,list) and len(scored)==30,'metric denominator mismatch'); ids=[r['record_id'] for r in scored]; need(ids==record_order and len(set(ids))==30,'missing/duplicate scored record or order violation')
    by={r['record_id']:r for r in scored}; mean=lambda ids,key:sum(by[r]['metrics'][key] for r in ids)/len(ids); count_keys=('critical_error_count','wrong_card_error_count','negative_false_recommendation_count','format_error_count','citation_error_count','unsupported_claim_count','candidate_failure_count')
    result={'positive_record_count':26,'negative_record_count':4,'all_record_count':30,'positive_macro':{k:mean(positive_ids,k) for k in ('card_precision_at_3','card_recall_at_3','supported_card_recall_at_3','required_claim_coverage','citation_ownership')},'negative_correctness_count':sum(by[r]['metrics']['negative_correctness'] for r in negative_ids),'negative_correctness_macro':mean(negative_ids,'negative_correctness'),'end_to_end_exact_success_count':sum(by[r]['metrics']['end_to_end_exact_success'] for r in record_order),'end_to_end_exact_success_macro':mean(record_order,'end_to_end_exact_success'),'error_totals':{k:sum(r[k] for r in scored) for k in count_keys},'candidate_failure_record_count':sum(r['candidate_failure_count']>0 for r in scored)}
    need(all(isinstance(v,(int,float)) and math.isfinite(v) for v in [*result['positive_macro'].values(),result['negative_correctness_macro'],result['end_to_end_exact_success_macro'],*result['error_totals'].values()]),'scorer internal nonfinite aggregate')
    return result
def candidate_eligible(agg): return agg['end_to_end_exact_success_count']>=21 and agg['positive_macro']['supported_card_recall_at_3']>=0.75 and agg['positive_macro']['required_claim_coverage']>=0.80 and agg['negative_correctness_count']==4 and agg['positive_macro']['citation_ownership']==1.0 and not any(agg['error_totals'].values())
def compare_records(candidate,baseline):
    cb={r['record_id']:r for r in candidate}; bb={r['record_id']:r for r in baseline}; need(set(cb)==set(bb)==set(record_order),'paired record mismatch'); wins=sum(cb[r]['metrics']['end_to_end_exact_success']>bb[r]['metrics']['end_to_end_exact_success'] for r in record_order); losses=sum(cb[r]['metrics']['end_to_end_exact_success']<bb[r]['metrics']['end_to_end_exact_success'] for r in record_order)
    hard=('critical_error_count','wrong_card_error_count','negative_false_recommendation_count'); regressions=[r for r in critical_ids if (bb[r]['metrics']['end_to_end_exact_success']==1 and cb[r]['metrics']['end_to_end_exact_success']==0) or any(cb[r][k]>0 and bb[r][k]==0 for k in hard)]
    return {'wins':wins,'losses':losses,'ties':30-wins-losses,'critical_cohort_regression_record_ids':regressions}
def select(candidate_agg,baseline_agg,paired):
    ce,be=candidate_eligible(candidate_agg),candidate_eligible(baseline_agg)
    if ce and not be: return 'candidate'
    if be and not ce: return 'baseline'
    if not ce and not be: return 'holdout_failed'
    if candidate_agg['end_to_end_exact_success_count']-baseline_agg['end_to_end_exact_success_count']>=3 and paired['wins']>paired['losses'] and candidate_agg['positive_macro']['supported_card_recall_at_3']>=baseline_agg['positive_macro']['supported_card_recall_at_3'] and candidate_agg['positive_macro']['required_claim_coverage']>=baseline_agg['positive_macro']['required_claim_coverage'] and not paired['critical_cohort_regression_record_ids']: return 'candidate'
    return 'inconclusive'

# Result-independent fixtures run through the same executable scorer.
def satisfying_raw(cond):
    value=cond['value']; operator=cond['operator']
    if operator=='gt': value=value+1
    elif operator in {'neq','not_in'}: value='__different__'
    elif operator=='in': value=value[0]
    elif operator=='all_of': value=list(value)
    elif operator=='any_of': value=[value[0]]
    elif operator=='contains': value=[value]
    elif operator=='exists': value='present' if value else None
    elif operator=='object_has_any_key': value={value[0]:1}
    return {'value':value,'unit':cond['unit']}
def fixture_claim(cid):
    claim=by_claim[cid]; spec=conversions[claim['conversion_id']]; raw_value=copy.deepcopy(claim['canonical_value']); raw_unit=claim['unit']
    if spec['type']=='divide': raw_value=raw_value*spec['divisor']; raw_unit=spec['source_unit']
    elif spec['type']=='korean_money_to_integer': raw_unit='KRW'
    elif spec['type']=='identity_boolean': raw_unit='boolean'
    return {'card_id':claim['card_id'],'claim_id':cid,'raw_value':raw_value,'raw_unit':raw_unit,'raw_conditions':{x['field']:satisfying_raw(x) for x in claim['conditions'] if x['required_for_complete_answer']},'citations':list(claim['required_component_ids'])}
def fixture_response(rid):
    record=by_record[rid]; pairs=[]
    for card,graph in record['qualification_graphs'].items():
        for cid,_,_ in next(iter(graph.values())):
            if (card,cid) not in pairs: pairs.append((card,cid))
    return {'record_id':rid,'returned_cards':list(record['expected_card_ids']),'mentioned_cards':list(record['expected_card_ids']),'extracted_claims':[fixture_claim(cid) for _,cid in pairs],'insufficient_evidence':not bool(record['expected_card_ids'])}
base_response=fixture_response('D06'); pass_score=score_response(copy.deepcopy(base_response)); need(pass_score['metrics']['supported_card_recall_at_3']==1 and pass_score['claim_audit'][0]['converted_value']==0.3,'conversion/support pass fixture')
not_returned=copy.deepcopy(base_response); not_returned['returned_cards']=[]; not_returned=score_response(not_returned); need(not_returned['graph_satisfied_cards']==['KB'] and not_returned['metrics']['supported_card_recall_at_3']==0,'supported-not-returned fixture')
candidate_fixtures={}
def candidate_failure(name,response,check=lambda score:True):
    score=score_response(response); need(score['metrics']['end_to_end_exact_success']==0 and score['candidate_failure_count']>0 and check(score),'candidate failure fixture: '+name); candidate_fixtures[name]={'counts':{k:score[k] for k in ('critical_error_count','wrong_card_error_count','negative_false_recommendation_count','format_error_count','citation_error_count','unsupported_claim_count','candidate_failure_count')},'outcome_reasons':score['outcome_reasons']}; return score
bad=copy.deepcopy(base_response); bad['returned_cards']=['KB','KB']; candidate_failure('duplicate_returned',bad,lambda s:s['format_error_count']>0)
bad=copy.deepcopy(base_response); bad['returned_cards']=['KB','NH','HANA','BC']; candidate_failure('over_k',bad,lambda s:s['format_error_count']>0)
bad=copy.deepcopy(base_response); bad['extracted_claims'][0]['claim_id']='UNKNOWN'; candidate_failure('unknown_claim',bad,lambda s:s['format_error_count']>0)
bad=copy.deepcopy(base_response); bad['extracted_claims'][0]['citations']=['UNKNOWN']; candidate_failure('unknown_source',bad,lambda s:s['citation_error_count']>0)
bad=copy.deepcopy(base_response); bad['returned_cards']=['UNKNOWN']; candidate_failure('unknown_card',bad,lambda s:s['wrong_card_error_count']==1)
bad=copy.deepcopy(base_response); bad['extracted_claims'][0]['citations']=['S006']; candidate_failure('citation_ownership_mismatch',bad,lambda s:s['citation_error_count']>0)
bad=copy.deepcopy(base_response); bad['extracted_claims'][0]['citations']=['S039']; candidate_failure('citation_excess',bad,lambda s:s['citation_error_count']>0 and s['unsupported_claim_count']>0)
bad=copy.deepcopy(base_response); bad.pop('insufficient_evidence'); candidate_failure('schema_error',bad,lambda s:s['format_error_count']>0)
bad=copy.deepcopy(base_response); bad['gold_label']='forbidden'; candidate_failure('leakage_extra_field',bad,lambda s:s['format_error_count']>0)
bad=copy.deepcopy(base_response); bad['extracted_claims'][0]['citations']=[]; candidate_failure('zero_required_citation',bad,lambda s:s['critical_error_count']>0 and s['citation_error_count']>0)
bad=copy.deepcopy(base_response); extra=fixture_claim('C17'); extra['raw_value']={'everland':0.4,'caribbean_bay':0.3}; bad['extracted_claims'].append(extra); candidate_failure('graph_outside_wrong_critical_C17',bad,lambda s:s['critical_error_count']==1)
bad=copy.deepcopy(base_response); bad['returned_cards']=['NH']; bad['mentioned_cards']=['NH']; candidate_failure('wrong_card',bad,lambda s:s['wrong_card_error_count']==1)
bad=copy.deepcopy(base_response); bad['mentioned_cards']=['NH']; candidate_failure('mentioned_card_outside_expected',bad,lambda s:s['wrong_card_error_count']==1)
bad=copy.deepcopy(base_response); bad['extracted_claims'].append(fixture_claim('C02')); candidate_failure('claim_card_outside_expected',bad,lambda s:s['wrong_card_error_count']==1)
bad=fixture_response('O09'); bad['returned_cards']=['KB']; bad['mentioned_cards']=['KB']; candidate_failure('negative_recommendation',bad,lambda s:s['negative_false_recommendation_count']==1 and s['wrong_card_error_count']==1)
invalid_fixtures={}
def must_invalid(name,fn):
    try: fn()
    except RuntimeError as exc: invalid_fixtures[name]=str(exc)
    else: raise RuntimeError('invalid fixture did not fail: '+name)
expected_context={k:digest(k.encode()) for k in context_hash_keys}; validate_execution_context(expected_context,expected_context); perfect_responses=[fixture_response(rid) for rid in record_order]; perfect_scored=score_batch(perfect_responses,expected_context,expected_context); agg=aggregate(perfect_scored)
need(agg['positive_record_count']==26 and agg['negative_record_count']==4 and agg['all_record_count']==30 and agg['negative_correctness_count']==4 and agg['end_to_end_exact_success_count']==30 and not any(agg['error_totals'].values()),'26/4/30 executable aggregate fixture')
bad_context=copy.deepcopy(expected_context); bad_context['model_sha256']='0'*64; must_invalid('execution_context_hash_mismatch',lambda:score_batch(perfect_responses,bad_context,expected_context))
must_invalid('duplicate_response',lambda:score_batch(perfect_responses[:-1]+[perfect_responses[0]],expected_context,expected_context)); must_invalid('missing_response',lambda:score_batch(perfect_responses[:-1],expected_context,expected_context)); must_invalid('order_violation',lambda:score_batch(list(reversed(perfect_responses)),expected_context,expected_context))
nonfinite=copy.deepcopy(perfect_scored); nonfinite[0]['metrics']['card_precision_at_3']=float('nan'); must_invalid('nonfinite_metric',lambda:aggregate(nonfinite))
bad_records=copy.deepcopy(by_record); bad_records['D06']['atomic_claim_ids']=[]; must_invalid('empty_required_claims_positive',lambda:validate_frozen_contract(bad_records))
linked=copy.deepcopy(perfect_scored); linked[5]=score_response((lambda r:(r['extracted_claims'].append(dict(fixture_claim('C17'),raw_value={'everland':0.4,'caribbean_bay':0.3})) or r))(copy.deepcopy(base_response))); linked_agg=aggregate(linked); need(linked_agg['error_totals']['critical_error_count']==linked[5]['critical_error_count'] and linked_agg['error_totals']['candidate_failure_count']==linked[5]['candidate_failure_count'] and not candidate_eligible(linked_agg),'aggregate count linkage')
paired_win={'wins':3,'losses':0,'ties':27,'critical_cohort_regression_record_ids':[]}; good=copy.deepcopy(agg); baseline27=copy.deepcopy(agg); baseline27['end_to_end_exact_success_count']=27; baseline27['end_to_end_exact_success_macro']=.9
selection_fixtures={'both_eligible_candidate_wins':select(good,baseline27,paired_win)}; need(selection_fixtures['both_eligible_candidate_wins']=='candidate','both eligible candidate wins')
critical_pair=copy.deepcopy(paired_win); critical_pair['critical_cohort_regression_record_ids']=['D06']; selection_fixtures['critical_regression_blocks']=select(good,baseline27,critical_pair); need(selection_fixtures['critical_regression_blocks']=='inconclusive','critical regression block')
supported_low=copy.deepcopy(good); supported_low['positive_macro']['supported_card_recall_at_3']=.8; supported_base=copy.deepcopy(baseline27); supported_base['positive_macro']['supported_card_recall_at_3']=.9; selection_fixtures['supported_recall_regression_blocks']=select(supported_low,supported_base,paired_win); need(selection_fixtures['supported_recall_regression_blocks']=='inconclusive','supported recall regression block')
coverage_low=copy.deepcopy(good); coverage_low['positive_macro']['required_claim_coverage']=.85; coverage_base=copy.deepcopy(baseline27); coverage_base['positive_macro']['required_claim_coverage']=.9; selection_fixtures['claim_coverage_regression_blocks']=select(coverage_low,coverage_base,paired_win); need(selection_fixtures['claim_coverage_regression_blocks']=='inconclusive','claim coverage regression block')
errored=copy.deepcopy(good); errored['error_totals']['format_error_count']=1; selection_fixtures['baseline_only_eligible']=select(errored,good,paired_win); need(selection_fixtures['baseline_only_eligible']=='baseline','baseline-only eligible')
for key in good['error_totals']:
    probe=copy.deepcopy(good); probe['error_totals'][key]=1; need(not candidate_eligible(probe),'any aggregate error must absolute-fail: '+key)

invalid_conditions=['candidate seal/hash mismatch','source hash mismatch','query hash mismatch','gold hash mismatch','scorer hash mismatch','prompt hash mismatch','model hash mismatch','record IDs differ from exact frozen 30','duplicate response','missing response','metric denominator mismatch','scorer internal nonfinite metric','frozen contract corruption','gold leakage or label-free execution order violation']
candidate_failure_conditions=['unknown card/claim/source','citation ownership mismatch or excess','missing required citation','duplicate returned or mentioned card','returned cards over K3','response or claim content schema error','wrong value/unit/required condition','wrong card','negative false recommendation']
operator_contract={'eq':'normalized actual equals expected exactly','gte':'numeric actual >= expected after conversion','gt':'numeric actual > expected after conversion','lte':'numeric actual <= expected after conversion','neq':'normalized actual differs from expected','in':'scalar actual belongs to expected collection','not_in':'scalar actual not in expected collection','all_of':'actual collection contains every expected member','any_of':'actual collection contains at least one expected member','contains':'actual contains expected','exists':'actual non-null existence equals expected boolean','object_has_any_key':'object has any requested non-null key'}
metric_contract={'card_precision_at_3':{'record_numerator':'expected cards in returned_top3','record_denominator':'actual unique returned cards; positive no output=0','aggregate':'record macro over positive26'},'card_recall_at_3':{'record_numerator':'expected cards in returned_top3','record_denominator':'expected cards','aggregate':'record macro over positive26'},'supported_card_recall_at_3':{'record_numerator':'expected ∩ returned_top3 ∩ same-card graph_satisfied','record_denominator':'expected cards','aggregate':'record macro over positive26'},'required_claim_coverage':{'record_numerator':'supported all_of leaves plus satisfied any_of graph units','record_denominator':'frozen required all_of leaves plus any_of graph units','aggregate':'record macro over positive26'},'citation_ownership':{'record_numerator':'owned claim/card citations','record_denominator':'all citations','positive_zero_citation':0,'correct_negative_abstention':'N/A','aggregate':'record macro over positive26'},'negative_correctness':{'record_numerator':'correct closed-corpus abstention','record_denominator':'one','aggregate':'count and macro over negative4'},'end_to_end_exact_success':{'record_value':'binary 0/1','aggregate':'count and macro over all30'}}
selection={'absolute':{'end_to_end_exact_success_count_min':21,'supported_card_recall_at_3_macro_min':0.75,'required_claim_coverage_macro_min':0.80,'negative_correctness_required':'4/4','all_derived_error_totals_required':0,'positive_citation_ownership_macro_required':1.0},'paired_metric':'per-record end_to_end_exact_success 0/1','critical_cohort_record_ids':critical_ids,'critical_regression_definition':'baseline record e2e1 to candidate0 OR candidate-only critical/wrong-card/negative-false error','winner':'one eligible wins; both fail => holdout_failed; both eligible require candidate gap>=3/30,wins>losses,retrieval nonregression,critical regression0; otherwise inconclusive'}
nb=json.loads(NOTEBOOK.read_text()); cell=next(c for c in nb['cells'] if c.get('id')==CELL_ID); cell_source=''.join(cell['source']) if isinstance(cell['source'],list) else cell['source']; cell_sha=digest(canon({'cell_id':CELL_ID,'source_text':cell_source}).encode()); scoring_generation=f'integrated_holdout_scoring_v6_{cell_sha[:16]}'
contract={'schema_version':VERSION,'scoring_generation_id':scoring_generation,'dataset_generation_id':dataset_generation,'status':STATUS,'input_schema':{'execution_context_hash_keys':sorted(context_hash_keys),'response_required_keys':sorted(response_keys),'extracted_claim_required_keys':sorted(claim_keys),'max_returned_cards':3,'self_reported_error_counts_forbidden':True},'pipeline':['validate exact hashes/seal/order and frozen contract','normalize response content without aborting batch','normalize raw extracted values','convert raw value/unit with conversion_id','apply canonical operator to value and required conditions','evaluate every explicit claim including graph-extra claims','derive citation/critical/wrong-card/negative/format failures','evaluate same-card qualification graph','compute record metrics and outcome reasons','aggregate 26/4/30 with internal error totals','apply absolute and paired selection'],'conversion_registry':conversions,'normalization':'NFKC + casefold + whitespace collapse for strings; recursive values','operator_contract':operator_contract,'metric_contract':metric_contract,'selection_contract':selection,'invalid_conditions':invalid_conditions,'candidate_failure_conditions':candidate_failure_conditions,'reference_implementation':{'functions':['normalize_scalar','validate_execution_context','convert_value','convert_condition','op','validate_frozen_contract','normalize_response_content','support_claim','graph_support','score_response','score_batch','aggregate','compare_records','candidate_eligible','select'],'revision_cell_source_sha256':cell_sha},'self_test_summary':{'conversion_before_after':{'raw_percent':30,'converted_ratio':pass_score['claim_audit'][0]['converted_value']},'supported_not_returned_recall':not_returned['metrics']['supported_card_recall_at_3'],'candidate_failure_fixture_names':sorted(candidate_fixtures),'invalid_fixture_names':sorted(invalid_fixtures),'aggregate_denominators':[26,4,30],'aggregate_count_linkage':True,'selection_fixture_results':selection_fixtures,'any_aggregate_error_absolute_fail':True,'candidate_result_constants_used':False},'candidate_seal_status':'forbidden_pending_independent_review','execution':{'environment':'skn25','cpu_offline':True,'api_requests':0,'network':0,'gpu':0,'embedding':0,'retrieval':0,'llm':0}}
audit={'schema_version':VERSION,'scoring_generation_id':scoring_generation,'status':'PASS','protected_core_raw_sha256':protected_before,'critical_cohort_record_ids':critical_ids,'conversion_fixture':contract['self_test_summary']['conversion_before_after'],'supported_not_returned_fixture':{'graph_satisfied_cards':not_returned['graph_satisfied_cards'],'returned_cards':[],'supported_card_recall_at_3':0},'candidate_failure_fixtures':candidate_fixtures,'invalid_fixtures':invalid_fixtures,'aggregate_fixture':agg,'aggregate_count_linkage_fixture':linked_agg,'selection_fixture_results':selection_fixtures,'api_network_gpu_embedding_retrieval_llm':0}
stage=Path(tempfile.mkdtemp(prefix='.v6_stage_',dir=OUT))
try:
    put_json(stage/'draft_scoring_contract.json',contract); put_json(stage/'scorer_reference_audit.json',audit)
    scoring_sha=sha(stage/'draft_scoring_contract.json')
    (stage/'README.md').write_text(f'''# Integrated holdout master v4 + scoring v5\n\n데이터 상태는 `{STATUS}`이며 v4 labels/components/projection/query/gold는 byte-exact입니다. scoring v5만 교정했습니다. candidate seal 전 ranking·answer 생성은 금지됩니다.\n\n- scoring SHA-256: `{scoring_sha}`\n- supported-card recall 분자: expected ∩ returned Top3 ∩ same-card graph-satisfied\n- aggregate: positive 26 record macro, negative 4 correctness, all 30 end-to-end exact\n- critical cohort: numeric_condition_critical 또는 negative 4인 explicit 19 record IDs\n- raw value/unit → conversion → operator/condition → claim/citation → same-card graph → record → aggregate → selection을 단일 reference scorer로 검증했습니다.\n- duplicate/>K/unknown ID/citation mismatch·excess/zero citation/supported-not-returned/empty claims/missing record/schema/duplicate response/missing response/order 위반 fixture를 포함합니다.\n- API/network/GPU/embedding/retrieval/LLM: 0\n\n독립 검토 전 seal할 수 없으며 실제 candidate 결과는 포함하지 않습니다.\n''',encoding='utf-8')
    readme=(stage/'README.md').read_text().replace('scoring v5','scoring v6').replace('- supported-card recall','- self-reported error count를 제거하고 scorer 내부에서 critical/wrong-card/negative/format/citation error를 파생합니다. candidate 내용 오류는 record failure로 저장하고 batch를 계속합니다. / supported-card recall').replace('- API/network','- aggregate 내부 error total이 하나라도 있으면 absolute gate 실패입니다. / API/network'); (stage/'README.md').write_text(readme,encoding='utf-8')
    manifest=copy.deepcopy(prior_manifest); manifest.update({'schema_version':VERSION,'scoring_generation_id':scoring_generation,'dataset_generation_id':dataset_generation,'status':STATUS,'scoring_contract_raw_sha256':scoring_sha,'scoring_revision_cell_source_sha256':cell_sha,'protected_v4_core_raw_sha256':protected_before,'candidate_seal_status':'forbidden_pending_independent_review','execution':contract['execution']})
    manifest['artifacts']=copy.deepcopy(prior_manifest['artifacts']); manifest['artifacts'].update({'README.md':sha(stage/'README.md'),'draft_scoring_contract.json':scoring_sha,'scorer_reference_audit.json':sha(stage/'scorer_reference_audit.json')})
    put_json(stage/'draft_dataset_manifest.json',manifest)
    integrity=copy.deepcopy(json.loads((OUT/'draft_integrity.json').read_text())); integrity.update({'schema_version':VERSION,'scoring_generation_id':scoring_generation,'dataset_generation_id':dataset_generation,'status':'PASS_PENDING_INDEPENDENT_REVIEW','manifest_sha256':sha(stage/'draft_dataset_manifest.json'),'scoring_contract_sha256':scoring_sha,'protected_v4_core_raw_sha256':protected_before})
    integrity['assertions'].update({'supported_recall_requires_returned_top3':True,'single_executable_reference_scorer':True,'self_reported_error_counts_removed':True,'derived_error_totals_linked_to_aggregate':True,'candidate_failures_do_not_abort_batch':True,'conversion_before_after_fixture':True,'failure_invariant_fixtures':True,'critical_cohort_ids_explicit':True,'invalid_conditions_explicit':True,'aggregate_26_4_30_self_test':True,'selection_self_tests':True,'v4_data_core_byte_exact':True,'api_network_gpu_embedding_retrieval_llm_zero':True})
    integrity['artifact_hashes']=manifest['artifacts']; put_json(stage/'draft_integrity.json',integrity)
    need(all(sha(OUT/n)==h for n,h in protected_before.items()),'protected core changed before replace')
    for n in ('README.md','draft_scoring_contract.json','scorer_reference_audit.json','draft_dataset_manifest.json','draft_integrity.json'): os.replace(stage/n,OUT/n)
finally: shutil.rmtree(stage,ignore_errors=True)
need(all(sha(OUT/n)==h for n,h in protected_before.items()),'protected core changed after replace')
saved=json.loads((OUT/'draft_dataset_manifest.json').read_text()); integ=json.loads((OUT/'draft_integrity.json').read_text())
need(saved['scoring_contract_raw_sha256']==sha(OUT/'draft_scoring_contract.json') and integ['manifest_sha256']==sha(OUT/'draft_dataset_manifest.json'),'scoring/manifest hash')
need(all(sha(OUT/n)==h for n,h in saved['artifacts'].items()),'artifact hashes')
print(canon({'status':'PASS_PENDING_INDEPENDENT_REVIEW','scoring_generation_id':scoring_generation,'dataset_generation_id':dataset_generation,'scoring_contract_sha256':scoring_sha,'protected_core_files':len(protected_before),'protected_core_byte_exact':True,'positive_records':26,'negative_records':4,'all_records':30,'critical_cohort_records':len(critical_ids),'candidate_failure_fixtures':len(candidate_fixtures),'invalid_fixtures':len(invalid_fixtures),'candidate_seal_status':'forbidden_pending_independent_review','api_network_gpu_embedding_retrieval_llm':0}))


{"all_records":30,"api_network_gpu_embedding_retrieval_llm":0,"candidate_failure_fixtures":15,"candidate_seal_status":"forbidden_pending_independent_review","critical_cohort_records":19,"dataset_generation_id":"integrated_holdout_master_v4_8ab30d91da2c2ba3","invalid_fixtures":6,"negative_records":4,"positive_records":26,"protected_core_byte_exact":true,"protected_core_files":13,"scoring_contract_sha256":"795617c731b5ac086e3acfe1e77f848a403636147a08456a738dd7b01e87072e","scoring_generation_id":"integrated_holdout_scoring_v6_af388e5a32fdf986","status":"PASS_PENDING_INDEPENDENT_REVIEW"}


## Final dataset seal — sealed, not evaluated

Independent review의 `SEALABLE` 판정을 기록하고 dataset lifecycle을 `sealed_not_evaluated`로 전환합니다. 이 seal은 candidate ranking·LLM·embedding 실행 승인이 아니며, 실행은 계속 금지됩니다.

In [1]:
# Final seal transition: CPU/offline; core, scoring, lineage, and demo are immutable.
from __future__ import annotations
import ast, copy, hashlib, json, os, shutil, tempfile
from pathlib import Path
cwd=Path.cwd().resolve()
if (cwd/'notebooks').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise RuntimeError('Run from repository root or notebooks/')
NOTEBOOK=ROOT/'notebooks/27_integrated_holdout_dataset.ipynb'; OUT=ROOT/'notebooks/data/27_integrated_holdout_dataset'; STATUS='sealed_not_evaluated'
def need(ok,msg):
    if not ok: raise RuntimeError(msg)
def canon(v): return json.dumps(v,ensure_ascii=False,sort_keys=True,separators=(',',':'))
def digest(b): return hashlib.sha256(b).hexdigest()
def sha(p): return digest(p.read_bytes())
def put_json(p,v): p.write_text(canon(v),encoding='utf-8')
manifest_before=json.loads((OUT/'draft_dataset_manifest.json').read_text()); integrity_before=json.loads((OUT/'draft_integrity.json').read_text()); scoring=json.loads((OUT/'draft_scoring_contract.json').read_text())
need(manifest_before['schema_version']=='integrated_holdout_scoring_v6' and scoring['schema_version']=='integrated_holdout_scoring_v6','reviewed scoring v6 required')
core_before=dict(manifest_before['protected_v4_core_raw_sha256']); scoring_before={n:sha(OUT/n) for n in ('draft_scoring_contract.json','scorer_reference_audit.json')}; lineage_before=dict(manifest_before['preserved_lineage_demo_raw_sha256']); parts_before=dict(manifest_before['proposal_part_raw_sha256'])
need(all(sha(OUT/n)==h for n,h in core_before.items()),'core changed before seal'); need(all(sha(OUT/n)==h for n,h in scoring_before.items()),'scoring changed before seal'); need(all(sha(OUT/p)==h for p,h in parts_before.items()),'proposal part changed before seal'); need(all(sha(OUT/n)==h for n,h in lineage_before.items()),'lineage/demo changed before seal')
nb=json.loads(NOTEBOOK.read_text()); scorer_cell=next(c for c in nb['cells'] if c.get('id')=='27-v6-scoring-contract'); scorer_source=''.join(scorer_cell['source']) if isinstance(scorer_cell['source'],list) else scorer_cell['source']; scorer_cell_sha=digest(canon({'cell_id':'27-v6-scoring-contract','source_text':scorer_source}).encode()); need(scorer_cell_sha==scoring['reference_implementation']['revision_cell_source_sha256'],'scorer source hash mismatch')
tree=ast.parse(scorer_source); selected=[n for n in tree.body if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)) and n.name in {'candidate_eligible','select'}]; need({n.name for n in selected}=={'candidate_eligible','select'},'selection functions missing'); namespace={}; exec(compile(ast.Module(body=selected,type_ignores=[]),'<sealed-v6-selection>','exec'),namespace)
good={'end_to_end_exact_success_count':30,'positive_macro':{'supported_card_recall_at_3':1.0,'required_claim_coverage':1.0,'citation_ownership':1.0},'negative_correctness_count':4,'error_totals':{'critical_error_count':0,'wrong_card_error_count':0,'negative_false_recommendation_count':0,'format_error_count':0,'citation_error_count':0,'unsupported_claim_count':0,'candidate_failure_count':0}}; bad=copy.deepcopy(good); bad['end_to_end_exact_success_count']=20; paired={'wins':3,'losses':0,'ties':27,'critical_cohort_regression_record_ids':[]}
selection_supplement={'candidate_only_eligible':namespace['select'](good,bad,paired),'both_fail':namespace['select'](bad,bad,paired)}; need(selection_supplement=={'candidate_only_eligible':'candidate','both_fail':'holdout_failed'},'selection supplement fixture')
records=[json.loads(x) for x in (OUT/'master_gold.jsonl').read_text().splitlines() if x]; record_ids=[*(f'D{i:02d}' for i in range(1,11)),*(f'O{i:02d}' for i in range(1,11)),*(f'A{i:02d}' for i in range(1,11))]; need([r['record_id'] for r in records]==record_ids,'record IDs not exact30')
proposal=json.loads((OUT/'proposal_canonical.json').read_text()); generation=manifest_before['dataset_generation_id']; reviewer={'review_role':'independent_reviewer','blocker_count':0,'major_count':0,'minor_count':1,'verdict':'SEALABLE','review_date':'2026-08-28'}
seal_core={'schema_version':'integrated_holdout_dataset_seal_v1','status':STATUS,'generation_id':generation,'master_dataset_core_canonical_sha256':proposal['canonical_sha256'],'master_dataset_core_artifact_sha256':sha(OUT/'proposal_canonical.json'),'record_ids':record_ids,'record_count':30,'hashes':{'query':sha(OUT/'evaluation_queries.json'),'gold':sha(OUT/'master_gold.jsonl'),'scoring':sha(OUT/'draft_scoring_contract.json'),'projection':sha(OUT/'candidate_projection.jsonl'),'struct_searchable_title_lineage':sha(OUT/'struct_searchable_title_lineage.jsonl'),'source_components':sha(OUT/'source_components.jsonl'),'source_refs':sha(OUT/'source_refs.json'),'claims':sha(OUT/'atomic_claims.json')},'reviewer_provenance':reviewer,'selection_fixture_supplement':selection_supplement,'scorer_source_sha256':scorer_cell_sha,'demo_exclusion':{'excluded_from_evaluation':True,'preserved_raw_sha256':lineage_before},'candidate_execution':{'authorized':False,'ranking':0,'llm':0,'embedding':0,'retrieval':0},'execution':{'environment':'skn25','cpu_offline':True,'api_requests':0,'network':0,'gpu':0,'embedding':0,'retrieval':0,'llm':0}}
seal_sha=digest(canon(seal_core).encode()); seal={'seal_core':seal_core,'dataset_seal_sha256':seal_sha,'self_hash_excludes':'dataset_seal_sha256 wrapper field'}
stage=Path(tempfile.mkdtemp(prefix='.seal_stage_',dir=OUT))
try:
    put_json(stage/'dataset_seal.json',seal)
    (stage/'README.md').write_text(f'''# Integrated holdout dataset — sealed, not evaluated

상태: `{STATUS}`  
Dataset seal SHA-256: `{seal_sha}`

Independent reviewer가 2026-08-28에 Blocker 0, Major 0, Minor 1로 `SEALABLE`을 판정했습니다. scoring v6의 candidate-only eligible/both-fail 분기를 seal 단계에서 추가 검증했고 scorer 소스·계약·해시는 변경하지 않았습니다.

30개 record와 query/gold/scoring/projection/source/claims, lineage/demo 제외 계약을 해시로 고정했습니다. Demo는 모든 평가 분모에서 제외됩니다.

이 seal은 평가 완료나 실행 승인을 의미하지 않습니다. candidate ranking·retrieval·embedding·LLM 실행은 계속 금지됩니다.
''',encoding='utf-8')
    manifest=copy.deepcopy(manifest_before); manifest.update({'status':STATUS,'dataset_status':STATUS,'candidate_seal_status':STATUS,'dataset_seal_sha256':seal_sha,'dataset_seal_raw_sha256':sha(stage/'dataset_seal.json'),'reviewer_provenance':reviewer,'candidate_execution_authorized':False,'seal_revision_cell_source_sha256':digest(canon({'cell_id':'27-final-dataset-seal','source_text':''.join(next(c for c in nb['cells'] if c.get('id')=='27-final-dataset-seal')['source'])}).encode())})
    manifest['known_limitations']=['sealed dataset has not been evaluated','candidate execution remains unauthorized','C35 retained only as retired non-evaluation provenance']; manifest['artifacts']=copy.deepcopy(manifest_before['artifacts']); manifest['artifacts'].update({'README.md':sha(stage/'README.md'),'dataset_seal.json':sha(stage/'dataset_seal.json')}); put_json(stage/'draft_dataset_manifest.json',manifest)
    integrity=copy.deepcopy(integrity_before); integrity.update({'status':'PASS_SEALED_NOT_EVALUATED','dataset_status':STATUS,'candidate_seal_status':STATUS,'dataset_seal_sha256':seal_sha,'dataset_seal_raw_sha256':sha(stage/'dataset_seal.json'),'manifest_sha256':sha(stage/'draft_dataset_manifest.json'),'reviewer_provenance':reviewer,'protected_v4_core_raw_sha256':core_before,'scoring_contract_sha256':scoring_before['draft_scoring_contract.json']}); integrity['assertions'].update({'independent_review_sealable':True,'selection_candidate_only_fixture':True,'selection_both_fail_fixture':True,'scorer_semantics_byte_exact':True,'record_ids_exact30':True,'demo_excluded':True,'candidate_execution_forbidden':True,'sealed_not_evaluated':True}); integrity['artifact_hashes']=manifest['artifacts']; put_json(stage/'draft_integrity.json',integrity)
    need(all(sha(OUT/n)==h for n,h in core_before.items()) and all(sha(OUT/n)==h for n,h in scoring_before.items()) and all(sha(OUT/n)==h for n,h in lineage_before.items()),'protected files changed before replace')
    for name in ('README.md','dataset_seal.json','draft_dataset_manifest.json','draft_integrity.json'): os.replace(stage/name,OUT/name)
finally: shutil.rmtree(stage,ignore_errors=True)
need(all(sha(OUT/n)==h for n,h in core_before.items()) and all(sha(OUT/n)==h for n,h in scoring_before.items()) and all(sha(OUT/n)==h for n,h in lineage_before.items()),'protected files changed after seal')
saved=json.loads((OUT/'draft_dataset_manifest.json').read_text()); integ=json.loads((OUT/'draft_integrity.json').read_text()); sealed=json.loads((OUT/'dataset_seal.json').read_text()); need(saved['status']==integ['dataset_status']==sealed['seal_core']['status']==STATUS,'sealed status mismatch'); need(digest(canon(sealed['seal_core']).encode())==sealed['dataset_seal_sha256']==saved['dataset_seal_sha256']==integ['dataset_seal_sha256'],'seal hash mismatch'); need(saved['dataset_seal_raw_sha256']==sha(OUT/'dataset_seal.json') and integ['manifest_sha256']==sha(OUT/'draft_dataset_manifest.json'),'raw hash mismatch'); need(all(sha(OUT/n)==h for n,h in saved['artifacts'].items()),'artifact hash mismatch')
print(canon({'status':STATUS,'dataset_seal_sha256':seal_sha,'dataset_seal_raw_sha256':sha(OUT/'dataset_seal.json'),'generation_id':generation,'records':30,'review_verdict':'SEALABLE','blockers':0,'majors':0,'minors':1,'selection_fixture_supplement':selection_supplement,'protected_core_files':len(core_before),'protected_scoring_files':len(scoring_before),'protected_lineage_demo_files':len(lineage_before),'candidate_execution_authorized':False,'api_network_gpu_embedding_retrieval_llm':0}))


{"api_network_gpu_embedding_retrieval_llm":0,"blockers":0,"candidate_execution_authorized":false,"dataset_seal_raw_sha256":"13ac776159b1283455430547a607691d358ffd27fc4ca1aea4d79a8796e8dd69","dataset_seal_sha256":"0580b54c5ca37c3e955e69f5e49c04a2f6c010c0472c7471ced54ee2b3407c6e","generation_id":"integrated_holdout_master_v4_8ab30d91da2c2ba3","majors":0,"minors":1,"protected_core_files":13,"protected_lineage_demo_files":11,"protected_scoring_files":2,"records":30,"review_verdict":"SEALABLE","selection_fixture_supplement":{"both_fail":"holdout_failed","candidate_only_eligible":"candidate"},"status":"sealed_not_evaluated"}
